[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DigitalPasts/alpcourse/blob/main/notebooks/02_Python_Brush_up.ipynb)

# Brushing up on your Python Skills

The basics of this class are taught in Python. And the neglected basics of ALP is preprocessing our texts.

Preprocessing for ALP is much broader than what computer and data scientists usually mean. Philological conventions in printed and digital publications hold much more information that needs to be correctly parsed before any computational manipulation (analysis).

In this notebook, we are going to provide four examples of messy texts: two in Egyptian and two in Akkadian. We are going to work through the process of how we should parse the texts, what information we are losing when parsing them, and brushing up on basic Python syntax and functions while we're at it.

## Akkadian Example 1:

https://cdli.mpiwg-berlin.mpg.de/artifacts/225104

&P225104 = TIM 10, 134
#atf: use lexical
#Nippur 2N-T496; proverb; Alster proverbs
@tablet
@obverse
@column 1
1. dub-sar hu-ru
2. a-ga-asz-gi4-gi4-me!(|ME+ASZ|)-e-ne
3. dub-sar hu-ru
4. a-ga-asz-gi4-gi4-me!(|ME+ASZ|)-e#-ne
@reverse
@column 1
1. igi-bi 3(disz) 3(asz) 6(disz)



### Task 1:

How do we turn this raw text into a list of words?

In [ ]:
akk1 = """&P225104 = TIM 10, 134
#atf: use lexical
#Nippur 2N-T496; proverb; Alster proverbs
@tablet
@obverse
@column 1
1. dub-sar hu-ru
2. a-ga-asz-gi4-gi4-me!(|ME+ASZ|)-e-ne
3. dub-sar hu-ru
4. a-ga-asz-gi4-gi4-me!(|ME+ASZ|)-e#-ne
@reverse
@column 1
1. igi-bi 3(disz) 3(asz) 6(disz)
"""

In [ ]:
akk1

In [ ]:
# split string to lines of texts
lines = akk1.split("\n")
lines

In [ ]:
# remove blanks

lines_full = []
for line in lines:
  if line != "":
    lines_full.append(line)

lines_full

In [ ]:
# keep only lines that begin with a number
# use regular expressions

import re

text_lines = []
for line in lines_full:
  if re.match(r"^\d", line) != None:
    text_lines.append(line)

text_lines


In [ ]:
# separate lines into words

words_appended = []
words_extended = []
for line in text_lines:
  temp_words = line.split()
  words_appended.append(temp_words[1:]) # creates list of lists
  words_extended.extend(temp_words[1:]) # creates list

print(words_appended)
print("-------------------------------")
print(words_extended)

Rewrite the code above as a function in the code cell below. Also think about what information did we lose when preprocessing the texts in this way?

In [ ]:
# rewrite the code above as a function



### Task 2:

Create a dictionary from the raw texts, of the following format:

```
{"pnum": ...
 "textID": ...
 "surface": [{
  "surfaceType": ...
  "columns": [{
    "columnNum": ...
    "text": [{
      "lineNum": ...
      "words": [..., ..., ...]
    }]
  }]
 }]}
```

In [ ]:
# separate text into lines

lines = akk1.split("\n")

lines_full = []
for line in lines:
  if line != "":
    lines_full.append(line)

lines_full

In [ ]:
# store the pnum and textID in variables

text_ids = lines_full[0]
pnum, textID = text_ids.split("=")

pnum = pnum.strip()[1:]
textID = textID.strip()
print(pnum)
print(textID)

In [ ]:
# create a dictionary for each surface (simple no regex method)
# what do you do when you have different types of inscribed object? (e.g. cylinder, prism, bowl, slab, etc.)

valid_surface_values = ["@obverse", "@reverse"]

surface_idx = []

for index, line in enumerate(lines_full):
  if line in valid_surface_values: # what is dangerous in this line? if the line of text is not exactly(!) part of surface, no lines will be found
    surface_idx.append(index)
print(surface_idx)

In [ ]:
# create a dictionary for each surface (complicated with regex method)
# what do you do when you have different type of inscribed object? (e.g. cylinder, prism, bowl, slab, etc.)

valid_surface_values = ["@obverse", "@reverse"]

pattern = r"^(?:" + "|".join([re.escape(value) for value in valid_surface_values]) + ")" # This is called a list comprehension

surface_idx = []

for index, line in enumerate(lines_full): # returns the index for the line and the content of the line
  if re.match(pattern, line) != None:
    surface_idx.append(index)
print(surface_idx)

In [ ]:
# same code like in cell above but without list comprehension
# create a dictionary for each surface (complicated with regex method)
# what do you do when you have different type of inscribed object? (e.g. cylinder, prism, bowl, slab, etc.)

valid_surface_values = ["@obverse", "@reverse"]

#pattern = r"^(?:" + "|".join([re.escape(value) for value in valid_surface_values]) + ")" # This is called a list comprehension

escaped_values = []
for value in valid_surface_values:
    escaped_values.append(re.escape(value))
print(escaped_values)

pattern = r"^(?:" + "|".join(escaped_values) + ")"
    
surface_idx = []

for index, line in enumerate(lines_full): # returns the index for the line and the content of the line
  if re.match(pattern, line) != None:
    surface_idx.append(index)
print(surface_idx)

In [ ]:
# use surface indices to create surface dictionaries
# surfaceType; columnNum; lineNum; words
# surfaceType extracted using id values of lines
# columnNum needs first to check whether a column actually exists, then extracted using regex(?)/tokenize on space for any number after the word column
# lineNum is regex for any line that begins with a number plus any tags attached: how would be best to define line numbers, as integers or as string variables?
# words extracted from each text line after lineNum using regex and tokenized on spaces

for index, id in enumerate(surface_idx):
    surfaceType = lines_full[id].replace('@', '')
    print(index, id)
    if index < len(surface_idx) - 1:
        end_of_surface = surface_idx[index+1]
    else:
        end_of_surface = len(lines_full)

    # Extract the text content for the current surface designation
    surface_content = lines_full[id+1:end_of_surface]

    # Print the surface type and its content
    print(f"Surface Type: {surfaceType}")
    # print("Content:")
    # print('\n'.join(surface_content))
    print('---')

    # Extract column number, line numbers, and words for each surface content
    for line in surface_content:
        columnNum = None
        lineNum = None
        words = []

        # Check if the line contains a column number
        if '@column' in line:
            parts = line.split()
            if len(parts) >= 2:
                try:
                    columnNum = int(parts[1])
                except ValueError:
                    pass
            print(f"Column Number: {columnNum}")
            print('---')
            continue  # Skip processing the line with @column

        # Check if the line contains a line number
        if '.' in line:
            parts = line.split('.')
            if len(parts) >= 2:
                lineNum = parts[0].strip()

        # Tokenize the words in the line
        if lineNum:
            words = parts[1].strip().split()
        else:
            words = line.strip().split()

        # Print the extracted information for each line
        print(f"Line Number: {lineNum}")
        print(f"Words: {words}")
        print('---')

In [ ]:
# Combine the surfaces and metadata into one dictionary

output = {
    "pnum": pnum,
    "textID": textID,
    "surface": []
}

for index, id in enumerate(surface_idx):
    surfaceType = lines_full[id].replace('@', '')
    surface = {
        "surfaceType": surfaceType,
        "columns": []
    }

    if index < len(surface_idx) - 1:
        end_of_surface = surface_idx[index+1]
    else:
        end_of_surface = len(lines_full)

    # Extract the text content for the current surface designation
    surface_content = lines_full[id+1:end_of_surface]

    # Extract column number, line numbers, and words for each surface content
    columnNum = None
    column = {
        "columnNum": None,
        "text": []
    }
    for line in surface_content:
        lineNum = None
        words = []

        # Check if the line contains a column number
        if '@column' in line:
            parts = line.split()
            if len(parts) >= 2:
                try:
                    columnNum = int(parts[1])
                    column["columnNum"] = columnNum
                except ValueError:
                    pass
            continue  # Skip processing the line with @column

        # Check if the line contains a line number
        if '.' in line:
            parts = line.split('.')
            if len(parts) >= 2:
                lineNum = parts[0].strip()

        # Tokenize the words in the line
        if lineNum:
            words = parts[1].strip().split()
        else:
            words = line.strip().split()

        # Add the line information to the column
        line_info = {
            "lineNum": lineNum,
            "words": words
        }
        column["text"].append(line_info)

    # Add the column to the surface
    surface["columns"].append(column)

    # Add the surface to the output
    output["surface"].append(surface)

# Print the output in the specified dictionary format
print(output)

In [ ]:
# Save the output dictionary as a JSON file

import json, os
os.makedirs('data/akkadian', exist_ok=True)
with open(f"data/akkadian/{pnum}.json", "w") as json_file:
    json.dump(output, json_file, indent=4)

Rewrite the code above into a function in the box below, use AI

In [ ]:
# rewrite the code above into a function

print(json_file)

## Egyptian Example 1:

A sentence from the sarcophagus of the Napatan king Aspelta (c. 600-580 BCE), found in his pyramid in Nuri, Sudan (Nu. 8), https://collections.mfa.org/objects/145117

Get the context of the sentence from the Thesaurus Linguae Aegyptiae: https://thesaurus-linguae-aegyptiae.de/text/27KHHMEP4VHSDH737F2OFLKNSE/sentences

In [ ]:
# This Dictionary was created from the original json file

eg1 = {'publication_statement': {'credit_citation': 'Doris Topmann, Sentence ID 2CBOF5UQ7JGETCXG2CQKPCWDZM <https://github.com/thesaurus-linguae-aegyptiae/tla-raw-data/blob/v17/sentences/2CBOF5UQ7JGETCXG2CQKPCWDZM.json>, in: Thesaurus Linguae Aegyptiae: Raw Data <https://github.com/thesaurus-linguae-aegyptiae/tla-raw-data>, Corpus issue 17 (31 October 2022), ed. by Tonio Sebastian Richter & Daniel A. Werning on behalf of the Berlin-Brandenburgische Akademie der Wissenschaften and Hans-Werner Fischer-Elfert & Peter Dils on behalf of the Sächsische Akademie der Wissenschaften zu Leipzig (first published: 22 September 2023)', 'collection_editors': 'Tonio Sebastian Richter & Daniel A. Werning on behalf of the Berlin-Brandenburgische Akademie der Wissenschaften and Hans-Werner Fischer-Elfert & Peter Dils on behalf of the Sächsische Akademie der Wissenschaften zu Leipzig', 'data_engineers': {'input_software_BTS': ['Christoph Plutte', 'Jakob Höper'], 'database_curation': ['Simon D. Schweitzer'], 'data_transformation': ['Jakob Höper', 'R. Dominik Blöse', 'Daniel A. Werning']}, 'date_published_in_TLA': '2022-10-31', 'rawdata_first_published': '2023-09-22', 'corresponding_TLA_URL': 'https://thesaurus-linguae-aegyptiae.de/sentence/2CBOF5UQ7JGETCXG2CQKPCWDZM', 'license': 'Creative Commons Attribution-ShareAlike 4.0 International (CC BY-SA 4.0) <https://creativecommons.org/licenses/by-sa/4.0/>'}, 'context': {'line': 'III', 'paragraph': None, 'pos': 7, 'textId': '27KHHMEP4VHSDH737F2OFLKNSE', 'textType': 'Text', 'variants': 1}, 'eclass': 'BTSSentence', 'glyphs': {'mdc_compact': None, 'unicode': None}, 'id': '2CBOF5UQ7JGETCXG2CQKPCWDZM', 'relations': {'contains': [{'eclass': 'BTSAnnotation', 'id': 'DYJEAXFKBJAXJPVLJGWREJZJ5M', 'ranges': [{'end': 'OKLGJLCEQFHU7HDRYUTYR352YA', 'start': '22TFIMS2CBBCFFCDSCAIT3HR3Y'}], 'type': 'ägyptologische Textsegmentierung'}], 'partOf': [{'eclass': 'BTSText', 'id': '27KHHMEP4VHSDH737F2OFLKNSE', 'name': 'Isis (HT 15, HT 14, HT 17)', 'type': 'Text'}]}, 'tokens': [{'annoTypes': ['ägyptologische Textsegmentierung'], 'flexion': {'btsGloss': '(unspecified)', 'lingGloss': 'PTCL', 'numeric': 3}, 'glyphs': {'mdc_artificially_aligned': True, 'mdc_compact': 'D35:N35', 'mdc_original': 'D35-N35', 'mdc_original_safe': None, 'mdc_tla': 'D35-N35', 'order': [1, 2], 'unicode': '𓂜𓈖'}, 'id': '22TFIMS2CBBCFFCDSCAIT3HR3Y', 'label': 'nn', 'lemma': {'POS': {'type': 'particle'}, 'id': '851961'}, 'transcription': {'mdc': 'nn', 'unicode': 'nn'}, 'translations': {'de': ['[Negationspartikel]']}, 'type': 'word'}, {'annoTypes': ['ägyptologische Textsegmentierung'], 'flexion': {'btsGloss': 'SC.act.ngem.nom.subj_Neg.nn', 'lingGloss': 'V\\tam.act', 'numeric': 210020}, 'glyphs': {'mdc_artificially_aligned': False, 'mdc_compact': 'W11-V28-A7', 'mdc_original': 'W11-V28-A7', 'mdc_original_safe': None, 'mdc_tla': 'W11-V28-A7', 'order': [2, 3, 4], 'unicode': '𓎼𓎛𓀉'}, 'id': 'IOLUGQXLCRGNLMTAPJ65LI7MHU', 'label': 'gḥ', 'lemma': {'POS': {'subtype': 'verb_3-lit', 'type': 'verb'}, 'id': '166480'}, 'transcription': {'mdc': 'gH', 'unicode': 'gḥ'}, 'translations': {'de': ['matt sein']}, 'type': 'word'}, {'annoTypes': ['ägyptologische Textsegmentierung'], 'flexion': {'btsGloss': 'Noun.pl.stpr.3sgm', 'lingGloss': 'N.f:pl:stpr', 'numeric': 70154}, 'glyphs': {'mdc_artificially_aligned': True, 'mdc_compact': 'D36:X1*F51B-Z2', 'mdc_original': 'D36-X1-F51B-Z2', 'mdc_original_safe': None, 'mdc_tla': 'D36-X1-F51B-Z2', 'order': [5, 6, 7, 8], 'unicode': '𓂝𓏏𓄹︀\U00013440𓏥'}, 'id': 'GUVBJUGCSVF5VN55PN6RYS4YLI', 'label': 'ꜥ,t.pl', 'lemma': {'POS': {'subtype': 'substantive_fem', 'type': 'substantive'}, 'id': '34550'}, 'transcription': {'mdc': 'a.t.PL', 'unicode': 'ꜥ.t.PL'}, 'translations': {'de': ['Glied; Körperteil']}, 'type': 'word'}, {'annoTypes': ['ägyptologische Textsegmentierung'], 'flexion': {'btsGloss': '(unspecified)', 'lingGloss': '-3sg.m', 'numeric': 3}, 'glyphs': {'mdc_artificially_aligned': False, 'mdc_compact': 'I9', 'mdc_original': 'I9', 'mdc_original_safe': None, 'mdc_tla': 'I9', 'order': [9], 'unicode': '𓆑'}, 'id': 'GIHCJ27JXVAM7GDUYWGEPKBRB4', 'label': '=f', 'lemma': {'POS': {'subtype': 'personal_pronoun', 'type': 'pronoun'}, 'id': '10050'}, 'transcription': {'mdc': '=f', 'unicode': '=f'}, 'translations': {'de': ['[Suffix Pron. sg.3.m.]']}, 'type': 'word'}, {'annoTypes': ['ägyptologische Textsegmentierung'], 'flexion': {'btsGloss': '(unspecified)', 'lingGloss': 'dem.f.pl', 'numeric': 3}, 'glyphs': {'mdc_artificially_aligned': True, 'mdc_compact': 'M17-Q3:N35', 'mdc_original': 'M17-Q3-N35', 'mdc_original_safe': None, 'mdc_tla': 'M17-Q3-N35', 'order': [10, 11, 12], 'unicode': '𓇋𓊪𓈖'}, 'id': 'Z6HTGGPBPRDT3OZTZNXRF2GRDA', 'label': 'jp〈t〉n', 'lemma': {'POS': {'subtype': 'demonstrative_pronoun', 'type': 'pronoun'}, 'id': '850009'}, 'transcription': {'mdc': 'jp〈t〉n', 'unicode': 'jp〈t〉n'}, 'translations': {'de': ['diese [Dem.Pron. pl.f.]']}, 'type': 'word'}, {'annoTypes': ['ägyptologische Textsegmentierung'], 'flexion': {'btsGloss': '(unspecified)', 'lingGloss': 'TITL', 'numeric': 3}, 'glyphs': {'mdc_artificially_aligned': False, 'mdc_compact': 'D4-Q1-A40', 'mdc_original': 'D4-Q1-A40', 'mdc_original_safe': None, 'mdc_tla': 'D4-Q1-A40', 'order': [13, 14, 15], 'unicode': '𓁹𓊨𓀭'}, 'id': 'UCFJWBLRKJG4NJWTWT22WDR2MU', 'label': 'Wsr,w', 'lemma': {'POS': {'subtype': 'title', 'type': 'epitheton_title'}, 'id': '49461'}, 'transcription': {'mdc': 'wsr.w', 'unicode': 'Wsr.w'}, 'translations': {'de': ['Osiris (Totentitel des Verstorbenen)']}, 'type': 'word'}, {'annoTypes': ['ägyptologische Textsegmentierung'], 'flexion': {'btsGloss': '(unspecified)', 'lingGloss': 'N', 'numeric': 3}, 'glyphs': {'mdc_artificially_aligned': True, 'mdc_compact': 'M23-X1:N35', 'mdc_original': 'M23-X1-N35', 'mdc_original_safe': None, 'mdc_tla': 'M23-X1-N35', 'order': [16, 17, 18], 'unicode': '𓇓𓏏𓈖'}, 'id': 'LI5FJI4ZUJEMPIKS5RQ5HHNBUE', 'label': 'nzw', 'lemma': {'POS': {'type': 'substantive'}, 'id': '88040'}, 'transcription': {'mdc': 'nzw', 'unicode': 'nzw'}, 'translations': {'de': ['König']}, 'type': 'word'}, {'annoTypes': ['ägyptologische Textsegmentierung'], 'flexion': {'btsGloss': '(unspecified)', 'lingGloss': 'ROYLN', 'numeric': 3}, 'glyphs': {'mdc_artificially_aligned': True, 'mdc_compact': 'V30:N17-N17', 'mdc_original': 'V30-N17-N17', 'mdc_original_safe': None, 'mdc_tla': 'V30-N17-N17', 'order': [19, 20, 21], 'unicode': '𓎟𓇿𓇿'}, 'id': 'ICADWHGbHkfdokpooG4eCy3Zfe8', 'label': 'nb-Tꜣ,du', 'lemma': {'POS': {'subtype': 'epith_king', 'type': 'epitheton_title'}, 'id': '400038'}, 'transcription': {'mdc': 'nb-tA.DU', 'unicode': 'nb-Tꜣ.DU'}, 'translations': {'de': ['Herr der Beiden Länder (Könige)']}, 'type': 'word'}, {'annoTypes': ['ägyptologische Textsegmentierung'], 'flexion': {'btsGloss': '(unspecified)', 'lingGloss': 'TITL', 'numeric': 3}, 'glyphs': {'mdc_artificially_aligned': True, 'mdc_compact': 'V30:D4-Aa1*X1:Y1', 'mdc_original': 'V30-D4-Aa1-X1-Y1', 'mdc_original_safe': None, 'mdc_tla': 'V30-D4-Aa1-X1-Y1', 'order': [22, 23, 24, 25, 26], 'unicode': '𓎟𓁹𓐍𓏏𓏛'}, 'id': 'ICADWHT2O1dc30SXuRZUlquIDpM', 'label': 'nb-jr(,t)-(j)ḫ,t', 'lemma': {'POS': {'subtype': 'title', 'type': 'epitheton_title'}, 'id': '400354'}, 'transcription': {'mdc': 'nb-jr(.t)-(j)x.t', 'unicode': 'nb-jr(.t)-(j)ḫ.t'}, 'translations': {'de': ['Herr des Rituals']}, 'type': 'word'}, {'annoTypes': ['ägyptologische Textsegmentierung'], 'flexion': {'btsGloss': '(unspecified)', 'lingGloss': 'ROYLN', 'numeric': 3}, 'glyphs': {'mdc_artificially_aligned': True, 'mdc_compact': '<-M17-O34:Q3-E23-N17->', 'mdc_original': '<-M17-O34-Q3-E23-N17->', 'mdc_original_safe': None, 'mdc_tla': '<-M17-O34-Q3-E23-N17->', 'order': [18, 19, 20, 21, 22, 23], 'unicode': '𓍹\U0001343c𓇋𓊃𓊪𓃭𓇿\U0001343d𓍺'}, 'id': 'J3MLYALWVNAMDDG33VZ3RIEEUA', 'label': 'Jsplt', 'lemma': {'POS': {'subtype': 'kings_name', 'type': 'entity_name'}, 'id': '850103'}, 'transcription': {'mdc': 'jsplt', 'unicode': 'Jsplt'}, 'translations': {'de': ['Aspelta']}, 'type': 'word'}, {'annoTypes': ['ägyptologische Textsegmentierung'], 'flexion': {'btsGloss': '(unspecified)', 'lingGloss': 'N.m:sg', 'numeric': 3}, 'glyphs': {'mdc_artificially_aligned': True, 'mdc_compact': 'U5:D36-P8h', 'mdc_original': 'U5-D36-P8h', 'mdc_original_safe': None, 'mdc_tla': 'U5-D36-P8h', 'order': [25, 26, 27], 'unicode': '𓌷𓂝𓊤︂'}, 'id': 'OKLGJLCEQFHU7HDRYUTYR352YA', 'label': 'mꜣꜥ-ḫrw', 'lemma': {'POS': {'subtype': 'substantive_masc', 'type': 'substantive'}, 'id': '66750'}, 'transcription': {'mdc': 'mAa-xrw', 'unicode': 'mꜣꜥ-ḫrw'}, 'translations': {'de': ['Gerechtfertigter (der selige Tote)']}, 'type': 'word'}], 'transcription': {'mdc': 'nn gH a.t.PL=f jp〈t〉n wsr.w nzw nb-tA.DU nb-jr(.t)-(j)x.t jsplt mAa-xrw', 'unicode': 'nn gḥ ꜥ.t.PL=f jp〈t〉n Wsr.w nzw nb-Tꜣ.DU nb-jr(.t)-(j)ḫ.t Jsplt mꜣꜥ-ḫrw'}, 'translations': {'de': ['Diese seine Glieder werden nicht matt sein, (die des) Osiris Königs, des Herrn der Beiden Länder, des Herrn des Rituals, Aspelta, des Gerechtfertigten.']}, 'type': None, 'wordCount': 11, 'editors': {'author': 'Doris Topmann', 'contributors': None, 'created': '2020-12-23 12:24:26', 'type': None, 'updated': '2022-08-29 10:22:01'}}
print(eg1)

In [ ]:
# parse the dictionary (json)

unicodeHiero = []
transcription = []
translLemma = []
posLemma = []
tokenID = []

for text_word in eg1["tokens"] :
    print(text_word["glyphs"]["unicode"], text_word["transcription"]["unicode"], text_word["translations"]["de"][0], text_word["lemma"]["POS"]["type"], text_word["id"] )
    tokenID.append(text_word["id"])
    unicodeHiero.append(text_word["glyphs"]["unicode"])
    translLemma.append(text_word["translations"]["de"][0])
    posLemma.append(text_word["lemma"]["POS"]["type"])
    
    if text_word["transcription"]["unicode"][0] == "=" : # replace equal sign as it will cause trouble in spreadsheet software like MS Excel
        transcription.append(text_word["transcription"]["unicode"].replace("=", '⸗')) # U+2E17
    else :
        transcription.append(text_word["transcription"]["unicode"])
        
    

In [ ]:
# get the ID of this sentence

sentenceID = eg1["id"]

In [ ]:
# create a dataframe and fill it

import pandas as pd

df_eg = pd.DataFrame({
    'unicode_hieroglyphs': unicodeHiero,
    'unicode_transcription': transcription,
    'lemma_translation': translLemma,
    'part-of-speech': posLemma,
    'tokenID' : tokenID
})

df_eg

In [ ]:
# save as *.csv

import os
os.makedirs('data/egyptian', exist_ok=True)
fileName = "aspelta_TLA_Sentence_" + sentenceID + ".csv"
df_eg.to_csv("data/egyptian/" + fileName)

## Akkadian Example 2: Achaemenid Babylonian Texts (CoNLL-U format)

> **Archived (original example):** The original version of this cell loaded raw HTML from
> the Achemenet website (`http://www.achemenet.com`) and stripped `<sub>`, `<sup>`, and `<i>`
> markup to produce a clean transliteration, then parsed it into word- and sign-level
> dictionaries. That site has since migrated its data to Zenodo as structured annotated
> files. The HTML-cleaning techniques demonstrated there (`.replace()`, `re.sub()`,
> hyphen-splitting) remain relevant and are also illustrated in Akkadian Example 1.

---

The [Achemenet project](http://www.achemenet.com) published transliterations of Babylonian
cuneiform texts from the Achaemenid Persian Empire (550–330 BCE). The corpus has been
automatically lemmatized by the University of Helsinki and published on Zenodo:

> Alstola, T., Sahala, A., Valk, J., and Ong, M. (2024). *Linguistically Annotated
> Achemenet Babylonian Texts*. Zenodo.
> [https://doi.org/10.5281/zenodo.14223709](https://doi.org/10.5281/zenodo.14223709)

The files use a **CoNLL-U** format — tab-separated, one token per line, with comment lines
beginning with `#`. In Ancient Language Processing there is no single dominant annotation
standard. Each corpus project makes its own structural choices: ATF, JSON, XML, CSV,
CoNLL-U... The skill is not memorising any one format, but learning to read whatever
structure a corpus provides and reshaping it around your specific research question.

The Achemenet CoNLL-U uses 17 columns:

| # | Column | Content |
|---|--------|---------|
| 1 | `id` | Token index within the text |
| 2 | `form` | Transliteration (ORACC ATF conventions) |
| 3 | `lemma` | Dictionary lemma |
| 4 | `upos` | Universal POS tag |
| 5 | `xpos` | Project-specific POS tag |
| 6 | `feats` | Morphological features |
| 7 | `head` | Index of syntactic head |
| 8 | `deprel` | Dependency relation |
| 9 | `deps` | Enhanced dependencies |
| 10 | `misc` | Miscellaneous features (e.g. `lacuna_large`) |
| 11 | `eng` | English gloss |
| 12–17 | `norm`, `lang`, `formctx`, `xposctx`, `score`, `lock` | Additional metadata |

Text boundaries are marked by comment lines: `# X001171 = Strassmaier, Darius 1`

**Task:** Parse the CoNLL-U data, select the columns relevant to your analysis,
and export to our custom CSV format.


In [ ]:
# The Achemenet corpus is available from Zenodo:
# https://doi.org/10.5281/zenodo.14223709
#
# To download and work with the full archive in Colab:
#
#   import requests, zipfile, io
#   r = requests.get('https://zenodo.org/records/19067652/files/Achemenet.zip?download=1')
#   z = zipfile.ZipFile(io.BytesIO(r.content))
#   raw = z.read('Strassmaier.conllu').decode('utf-8')
#
# Below we embed the first text (Strassmaier, Darius 1) directly
# so the notebook runs without a network connection.

akk2_conllu = """# global.columns = id form lemma upos xpos feats head deprel deps misc eng norm lang formctx xposctx score lock
# X001171 = Strassmaier, Darius 1
1	10	_	_	n	empty	root	0	_	num_cardinal	_	_	_	_	_	_	_
2	UDU-NITA₂	immeru	_	N	empty	child	1	_	_	sheep	_	_	_	_	4.0	_
3	1+en	ištēn	_	NU	empty	child	1	_	_	one	_	_	_	_	4.0	_
4	GU₄	alpu	_	N	empty	child	1	_	_	bull	_	_	_	_	5.0	_
5	NINDA	nindanu	_	N	empty	child	1	_	_	rod	_	_	_	_	4.0	_
6	...	_	_	u	empty	child	1	_	lacuna_large	_	_	_	_	_	_	_
7	ša₂	ša	_	DET	empty	child	1	_	_	of	_	_	_	_	5.0	_
8	{lu₂}KIN-GI₄-A	mār-šipri	_	N	empty	child	1	_	_	X-X	_	_	_	_	5.0	_
9	ša₂	ša	_	DET	empty	child	1	_	_	of	_	_	_	_	5.0	_
10	muh-hi	muhhu	_	N	empty	child	1	_	_	skull	_	_	_	_	5.0	_
11	...	_	_	u	empty	child	1	_	lacuna_large	_	_	_	_	_	_	_
12	id-din-nu	nadānu	_	V	empty	child	1	_	_	give	_	_	_	_	5.0	_
13	ina	ina	_	PRP	empty	child	1	_	_	in	_	_	_	_	5.0	_
14	lib₃-bi	libbu	_	N	empty	child	1	_	_	interior	_	_	_	_	5.0	_
15	...	_	_	u	empty	child	1	_	lacuna_large	_	_	_	_	_	_	_
16	1	_	_	n	empty	child	1	_	num_cardinal	_	_	_	_	_	_	_
17	GU₄	alpu	_	N	empty	child	1	_	_	bull	_	_	_	_	5.0	_
18	šuk-lu-lu	šuklulu	_	AJ	empty	child	1	_	_	complete	_	_	_	_	5.0	_
19	U₄	ūmu	_	N	empty	child	1	_	_	day	_	_	_	_	5.0	_
20	20-KAM	_	_	n	empty	child	1	_	num_ordinal	_	_	_	_	_	_	_
21	ša₂	ša	_	DET	empty	child	1	_	_	of	_	_	_	_	5.0	_
22	{iti}AB	Ṭebetu	_	MN	empty	child	1	_	_	Ṭebetu	_	_	_	_	5.0	_
23	it-tal-ka	alāku	_	V	empty	child	1	_	_	go	_	_	_	_	4.0	_
24	2	_	_	n	empty	child	1	_	num_cardinal	_	_	_	_	_	_	_
25	UDU-NITA₂	immeru	_	N	empty	child	1	_	_	sheep	_	_	_	_	4.0	_
26	ša₂	ša	_	REL	empty	child	1	_	_	that	_	_	_	_	5.0	_
27	ina	ina	_	PRP	empty	child	1	_	_	in	_	_	_	_	5.0	_
28	IGI	pānu	_	N	empty	child	1	_	_	front	_	_	_	_	5.0	_
29	{m}...	_	_	u	empty	child	1	_	lacuna_large	_	_	_	_	_	_	_
30	6	_	_	n	empty	child	1	_	num_cardinal	_	_	_	_	_	_	_
31	UDU-NITA₂	immeru	_	N	empty	child	1	_	_	sheep	_	_	_	_	4.0	_
32	...	_	_	u	empty	child	1	_	lacuna_large	_	_	_	_	_	_	_
33	...	_	_	u	empty	child	1	_	lacuna_large	_	_	_	_	_	_	_
34	ina	ina	_	PRP	empty	child	1	_	_	in	_	_	_	_	5.0	_
35	IGI	mahru	_	N	empty	child	1	_	_	front	_	_	_	_	5.0	_
36	{m}IR₃-{d}SAGGAR₂	Arad-Bunene	_	PN	empty	child	1	_	_	Arad-Bunene	_	_	_	_	4.0	_
37	1/3	_	_	n	empty	child	1	_	_	_	_	_	_	_	5.0	_
38	2	_	_	n	empty	child	1	_	num_cardinal	_	_	_	_	_	_	_
39	GIN₂	šiqlu	_	N	empty	child	1	_	_	unit	_	_	_	_	5.0	_
40	KU₃-BABBAR	kaspu	_	N	empty	child	1	_	_	silver	_	_	_	_	4.0	_
41	TA	ištu	_	PRP	empty	child	1	_	_	from	_	_	_	_	5.0	_
42	re-hi	rēhu	_	AJ	empty	child	1	_	_	remaining	_	_	_	_	5.0	_
43	a-na	ana	_	PRP	empty	child	1	_	_	to	_	_	_	_	5.0	_
44	6	_	_	n	empty	child	1	_	num_cardinal	_	_	_	_	_	_	_
45	UDU-NITA₂	immeru	_	N	empty	child	1	_	_	sheep	_	_	_	_	4.0	_
46	a-na	ana	_	PRP	empty	child	1	_	_	to	_	_	_	_	5.0	_
47	{m}NI-TIN-TU₄	Nidintu	_	PN	empty	child	1	_	_	Nidintu	_	_	_	_	4.0	_
48	SI₃-NU	nadānu	_	V	empty	child	1	_	_	give	_	_	_	_	0.0	_
49	UDU-NITA₂	immeru	_	N	empty	child	1	_	_	sheep	_	_	_	_	4.0	_
50	a-na	ana	_	PRP	empty	child	1	_	_	to	_	_	_	_	5.0	_
51	sat-tuk	sattukku	_	N	empty	child	1	_	_	regular offering	_	_	_	_	4.0	_
52	ina	ina	_	PRP	empty	child	1	_	_	in	_	_	_	_	5.0	_
53	IGI	mahru	_	N	empty	child	1	_	_	front	_	_	_	_	5.0	_
54	{m}šu-zu-bu	Šuzubu	_	PN	empty	child	1	_	_	Šuzubu	_	_	_	_	4.0	_
55	{iti}ZIZ₂	Šabaṭu	_	MN	empty	child	1	_	_	Šabaṭu	_	_	_	_	5.0	_
56	U₄	ūmu	_	N	empty	child	1	_	_	day	_	_	_	_	5.0	_
57	20-KAM	_	_	n	empty	child	1	_	num_ordinal	_	_	_	_	_	_	_
58	MU	šattu	_	N	empty	child	1	_	_	year	_	_	_	_	5.0	_
59	SAG	rēšu	_	N	empty	child	1	_	_	head	_	_	_	_	5.0	_
60	...	_	_	u	empty	child	1	_	lacuna_large	_	_	_	_	_	_	_
61	{m}da-ri-uš-šu₂	Darius	_	RN	empty	child	1	_	_	Darius	_	_	_	_	4.0	_
62	LUGAL	šarru	_	N	empty	child	1	_	_	king	_	_	_	_	5.0	_
63	TIN-TIR{ki}	Babili	_	SN	empty	child	1	_	_	Babili	_	_	_	_	4.0	_
64	LUGAL	šarru	_	N	empty	child	1	_	_	king	_	_	_	_	5.0	_
65	KUR-KUR	mātu	_	N	empty	child	1	_	_	lands	_	_	_	_	4.0	TRUE"""

# Show the first few lines to understand the raw format
for line in akk2_conllu.strip().split('\n')[:8]:
    print(line)


In [ ]:
# Parse the CoNLL-U content.
# Rules:
#   '# global.columns = ...' defines the 17 column names
#   '# X001171 = Strassmaier, Darius 1' opens a new text
#   all other non-empty lines are tab-separated token rows

columns = None
tokens = []
current_text_id = None

for line in akk2_conllu.strip().split('\n'):
    line = line.strip()
    if not line:
        continue
    if line.startswith('# global.columns'):
        columns = line.split(' = ')[1].split()
    elif line.startswith('# X') and ' = ' in line:
        current_text_id = line[2:].split(' = ', 1)[0].strip()
    elif not line.startswith('#') and '\t' in line:
        token = dict(zip(columns, line.split('\t')))
        token['text_id'] = current_text_id
        tokens.append(token)

print(f'Columns ({len(columns)}): {columns}')
print(f'Text: {current_text_id}  |  Tokens: {len(tokens)}')
print()
print('First 5 tokens:')
for tok in tokens[:5]:
    print(f"  id={tok['id']:>3}  form={tok['form']:<22}  lemma={tok['lemma']:<15}  pos={tok['xpos']:<5}  eng={tok['eng']}")


In [ ]:
# Map CoNLL-U fields onto the course standard CSV format.
# Reference: data/rinap01/Q003414.csv in the ALP-course repository.
# Columns that cannot be derived from CoNLL-U are left empty.
#
#  ref          text_id.word_index (e.g. X001171.1)
#  inst         cf[sense]pos$norm  — ORACC lemma-instance notation
#  frag         ATF transliteration form
#  norm         normalised form
#  cf           citation form (lemma headword)
#  sense        English gloss
#  pos          part-of-speech tag
#  unicode      list of Unicode cuneiform signs per sign  (NOT in CoNLL-U)
#  unicode_word cuneiform signs for the whole word        (NOT in CoNLL-U)
#  reading      sign reading (= frag in this source)
#  break        damage status: 'missing' if lacuna, else 'complete'
#  break_perc   fraction of missing signs
#  mask         (not available)
#  lang         language code
#  text         text identifier
#  line         line number  (NOT encoded per-token in this CoNLL-U)
#  word         word index within text

import pandas as pd

rows_out = []
for tok in tokens:
    is_lacuna = 'lacuna' in tok.get('misc', '')
    cf    = tok.get('lemma', '')
    norm  = tok.get('norm', '')
    sense = tok.get('eng', '')
    pos   = tok.get('xpos', '')
    frag  = tok.get('form', '')
    tid   = tok.get('text_id', '')
    wid   = tok.get('id', '')

    rows_out.append({
        'ref':          f"{tid}.{wid}",
        'inst':         f"{cf}[{sense}]{pos}${norm}" if cf and cf != '_' else '',
        'frag':         frag,
        'norm':         norm if norm != '_' else '',
        'cf':           cf   if cf   != '_' else '',
        'sense':        sense if sense != '_' else '',
        'pos':          pos   if pos   != '_' else '',
        'unicode':      '',
        'unicode_word': '',
        'reading':      frag,
        'break':        "['missing']" if is_lacuna else "['complete']",
        'break_perc':   '1.0' if is_lacuna else '0.0',
        'mask':         '',
        'lang':         tok.get('lang', ''),
        'text':         tid,
        'line':         '',
        'word':         wid,
    })

df_akk2 = pd.DataFrame(rows_out)
df_akk2.head(10)


In [ ]:
import os
os.makedirs('data/akkadian', exist_ok=True)
df_akk2.to_csv('data/akkadian/Achemenet_Strassmaier_Darius1.csv', index=False)
print('Saved to data/akkadian/Achemenet_Strassmaier_Darius1.csv')
print(df_akk2[['ref','frag','cf','sense','pos','break']].to_string(index=False))

# Discussion questions:
# - Which fields are empty? What does that tell us about what CoNLL-U prioritises
#   vs. what the ORACC pipeline provides?
# - The 'line' field is empty here. How would you recover it from the CoNLL-U text?
# - Why does some cells show '_' for lemma or sense? What does that represent?

## Egyptian Example 2: Texts from the webiste of "Projet Karnak"

**Example: Hittite-Egyptian peace treaty from c. 1258 BC (Hattusili III. and Ramesses II.) = [KIU32](http://sith.huma-num.fr/karnak/32)**

Same text in the *Thesaurus Linguae Aegyptiae*: </br>
Silke Grallert, with contributions by AV Wortschatz der ägyptischen Sprache, Simon D. Schweitzer, Anja Weber, Jonas Treptow, "Textfeld" [Text-ID Q66COJF7HZHQ7ATL25T5KRNUVM](https://thesaurus-linguae-aegyptiae.de/text/Q66COJF7HZHQ7ATL25T5KRNUVM), in: *Thesaurus Linguae Aegyptiae*, Corpus issue 20, Web app version 2.4.1, 3/5/2026, ed. by Tonio Sebastian Richter & Daniel A. Werning on behalf of Berlin-Brandenburgischen Akademie der Wissenschaften and Hans-Werner Fischer-Elfert & Peter Dils on behalf of Sächsischen Akademie der Wissenschaften zu Leipzig (accessed: 4/13/2026) 

**This is what the text looks like in HTML** (double click in the cell to the raw HTML, not a preview of the rendering)


En plus de cette copie, une deuxième transcription égyptienne du traité est connue au Ramesseum (PM II<sup>2</sup>, p. 433-434).</div><br /><a name="inscription"></a><h3>Inscription</h3> <br /><b>Amon-Rê</b><br />
<div id="l1" style="padding-bottom:10px;"><img src="http://sith.huma-num.fr/KIUH/KIU32-1.png" /></div> <b>1</b> <em><a href="http://sith.huma-num.fr/vocable/30" class="i v30" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V30-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/30&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Donner, offrir, accorder &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme verbe &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">d~n</a><span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V1-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 1<sup>re</sup> personne singulier">&#11799;j</span> <a href="http://sith.huma-num.fr/vocable/188" class="i v188" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V188-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/188&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; À, pour &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">n</a><span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V2-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 2<sup>e</sup> personne masculin singulier">&#11799;k</span> <a href="http://sith.huma-num.fr/vocable/297" class="i v297" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V297-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/297&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Force, puissance &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pḥty</a> <a href="http://sith.huma-num.fr/vocable/56" class="i v56" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V56-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/56&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Seigneur, maître, responsable &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nbwy</a> <a href="http://sith.huma-num.fr/vocable/20" class="i v20" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V20-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/20&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Faire, agir &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme verbe &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">jr</a><span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V2-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 2<sup>e</sup> personne masculin singulier">&#11799;k</span> <a href="http://sith.huma-num.fr/vocable/256" class="i v256" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V256-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/256&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Limite, frontière &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">tȝšw</a><span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V2-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 2<sup>e</sup> personne masculin singulier">&#11799;k</span> <a href="http://sith.huma-num.fr/vocable/184" class="i v184" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V184-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/184&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Pour, vers &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">r</a> <a href="http://sith.huma-num.fr/vocable/37" class="i v37" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V37-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/37&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Aimer, désirer &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme verbe &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">mr~n</a><span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V2-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 2<sup>e</sup> personne masculin singulier">&#11799;k</span></em><br /><br />

<br /><b>Mout</b><br />
<div id="l2" style="padding-bottom:10px;"><img src="http://sith.huma-num.fr/KIUH/KIU32-2.png" /></div> <b>2</b> <em><a href="http://sith.huma-num.fr/vocable/30" class="i v30" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V30-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/30&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Donner, offrir, accorder &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme verbe &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">d~n</a><span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V1-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 1<sup>re</sup> personne singulier">&#11799;j</span> <a href="http://sith.huma-num.fr/vocable/188" class="i v188" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V188-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/188&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; À, pour &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">n</a><span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V2-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 2<sup>e</sup> personne masculin singulier">&#11799;k</span> <a href="http://sith.huma-num.fr/vocable/203" class="i v203" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V203-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/203&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Pays, terre &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">tȝw</a> <a href="http://sith.huma-num.fr/vocable/150" class="i v150" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V150-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/150&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Tout, chaque &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé pour qualifier un substantif (adjectif épithète) &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nbw</a> <a href="http://sith.huma-num.fr/vocable/241" class="i v241" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V241-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/241&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Contrée étrangère &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ḫȝswt</a> <a href="http://sith.huma-num.fr/vocable/150" class="i v150" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V150-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/150&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Tout, chaque &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé pour qualifier un substantif (adjectif épithète) &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nbwt</a></em><br /><br />

<br /><b>Corps du texte</b><br />
<div id="l3" style="padding-bottom:10px;"><img src="http://sith.huma-num.fr/KIUH/KIU32-3.png" /></div> <b>3</b> <em><a href="http://sith.huma-num.fr/vocable/330" class="i v330" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V330-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/330&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Année (du règne) &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ḥȝt-sp</a> 21 <a href="http://sith.huma-num.fr/vocable/1612" class="i v1612" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V1612-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/1612&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Premier &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">tpy</a> <a href="http://sith.huma-num.fr/vocable/383" class="i v383" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V383-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/383&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Saison <i>peret</i> &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">prt</a> <a href="http://sith.huma-num.fr/vocable/418" class="i v418" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V418-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/418&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Jour &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">sw</a> 21 <a href="http://sith.huma-num.fr/vocable/175" class="i v175" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V175-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/175&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Auprès de &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ḫr</a> <a href="http://sith.huma-num.fr/vocable/33" class="i v33" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V33-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/33&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Majesté &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ḥm</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/vocable/10" class="i v10" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V10-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/10&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Roi &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nswt</a> <a href="http://sith.huma-num.fr/vocable/11" class="i v11" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V11-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/11&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Roi, roi de Basse-Égypte &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">bjty</a> <a href="http://sith.huma-num.fr/titulature/20" class="i t20" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/titulatures/T20-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/titulature/20&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Ramsès II &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Nom de couronnement  &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Wsr-Mȝʿt-Rʿ-stp~n-Rʿ</a> <a href="http://sith.huma-num.fr/vocable/12" class="i v12" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V12-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/12&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Fils &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">sȝ</a> <a href="http://sith.huma-num.fr/theonyme/9" class="i d9" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/theonymes/D9-G1.png&quot; height=&quot;35px;&quot; /&gt;  &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/theonyme/9&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Rê &lt;/span&gt;&lt;/a&gt; &lt;br /&gt; Th&eacute;onyme ou d&eacute;signation divine &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt; ">Rʿ</a> <a href="http://sith.huma-num.fr/titulature/21" class="i t21" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/titulatures/T21-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/titulature/21&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Ramsès II &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Nom de fils de Rê  &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Rʿ-ms-sw-mry-Jmn</a> <a href="http://sith.huma-num.fr/vocable/30" class="i v30" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V30-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/30&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Donner, offrir, accorder &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme verbe &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">dw</a> <a href="http://sith.huma-num.fr/vocable/44" class="i v44" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V44-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/44&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Vie &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ʿnḫ</a> <a href="http://sith.huma-num.fr/vocable/171" class="i v171" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V171-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/171&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Éternellement &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme adverbe &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ḏt</a> <a href="http://sith.huma-num.fr/vocable/192" class="i v192" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V192-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/192&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Éternellement &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme adverbe &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nḥḥ</a> <a href="http://sith.huma-num.fr/vocable/79" class="i v79" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V79-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/79&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Aimé de &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">mry</a> <a href="http://sith.huma-num.fr/theonyme/10" class="i d10" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/theonymes/D10-G1.png&quot; height=&quot;35px;&quot; /&gt;  &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/theonyme/10&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Amon-Rê &lt;/span&gt;&lt;/a&gt; &lt;br /&gt; Th&eacute;onyme ou d&eacute;signation divine &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt; ">Jmn-Rʿ</a> Ḥr-Ȝḫty <a href="http://sith.huma-num.fr/theonyme/7" class="i d7" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/theonymes/D7-G1.png&quot; height=&quot;35px;&quot; /&gt;  &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/theonyme/7&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Ptah &lt;/span&gt;&lt;/a&gt; &lt;br /&gt; Th&eacute;onyme ou d&eacute;signation divine &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt; ">Ptḥ</a> <a href="http://sith.huma-num.fr/vocable/984" class="i v984" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V984-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/984&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Qui est au sud, méridional &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme verbe &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">rsy</a> <a href="http://sith.huma-num.fr/vocable/360" class="i v360" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V360-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/360&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Mur, enceinte, édifice, bâtiment &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">jnb</a><span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V4-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne masculin singulier">&#11799;f</span> <a href="http://sith.huma-num.fr/vocable/56" class="i v56" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V56-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/56&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Seigneur, maître, responsable &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nb</a> <a href="http://sith.huma-num.fr/toponyme/38" class="i l38" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L38-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/38&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Ânkh-Taouy &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Localit&eacute; ou territoire &eacute;gyptien &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ʿnḫ-Tȝwy</a> <a href="http://sith.huma-num.fr/theonyme/17" class="i d17" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/theonymes/D17-G1.png&quot; height=&quot;35px;&quot; /&gt;  &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/theonyme/17&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Mout &lt;/span&gt;&lt;/a&gt; &lt;br /&gt; Th&eacute;onyme ou d&eacute;signation divine &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt; ">Mwt</a> <a href="http://sith.huma-num.fr/vocable/162" class="i v162" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V162-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/162&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Maîtresse, souveraine &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nbt</a> <a href="http://sith.huma-num.fr/toponyme/17" class="i l17" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L17-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/17&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Ichérou &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Nom de monument &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Jšrw</a> <a href="http://sith.huma-num.fr/theonyme/126" class="i d126" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/theonymes/D126-G1.png&quot; height=&quot;35px;&quot; /&gt;  &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/theonyme/126&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Khonsou-Nefer-Hotep &lt;/span&gt;&lt;/a&gt; &lt;br /&gt; Th&eacute;onyme ou d&eacute;signation divine &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt; ">Ḫnsw-Nfr-ḥtp</a> <a href="http://sith.huma-num.fr/vocable/25" class="i v25" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V25-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/25&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Paraître, apparaître, se lever &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme verbe &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ḫʿw</a> <a href="http://sith.huma-num.fr/vocable/166" class="i v166" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V166-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/166&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Sur, concernant &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ḥr</a> <a href="http://sith.huma-num.fr/vocable/24" class="i v24" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V24-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/24&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Siège, trône, lieu &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">st</a> <a href="http://sith.huma-num.fr/theonyme/8" class="i d8" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/theonymes/D8-G1.png&quot; height=&quot;35px;&quot; /&gt;  &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/theonyme/8&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Horus &lt;/span&gt;&lt;/a&gt; &lt;br /&gt; Th&eacute;onyme ou d&eacute;signation divine &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt; ">Ḥr</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nyt</a> <a href="http://sith.huma-num.fr/vocable/41" class="i v41" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V41-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/41&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Vivants &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ʿnḫw</a> <a href="http://sith.huma-num.fr/vocable/170" class="i v170" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V170-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/170&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Comme &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">mj</a> <a href="http://sith.huma-num.fr/vocable/22" class="i v22" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V22-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/22&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Père &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">jt</a><span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V4-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne masculin singulier">&#11799;f</span> <a href="http://sith.huma-num.fr/theonyme/11" class="i d11" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/theonymes/D11-G1.png&quot; height=&quot;35px;&quot; /&gt;  &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/theonyme/11&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Rê-Horakhty &lt;/span&gt;&lt;/a&gt; &lt;br /&gt; Th&eacute;onyme ou d&eacute;signation divine &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt; ">Rʿ-Ḥr-Ȝḫty</a> <a href="http://sith.huma-num.fr/vocable/171" class="i v171" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V171-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/171&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Éternellement &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme adverbe &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ḏt</a> <a href="http://sith.huma-num.fr/vocable/32" class="i v32" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V32-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/32&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Fois, occasion &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">sp</a> 2 <a href="http://sith.huma-num.fr/vocable/192" class="i v192" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V192-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/192&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Éternellement &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme adverbe &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nḥḥ</a></em><br /><br />
<div id="l4" style="padding-bottom:10px;"><img src="http://sith.huma-num.fr/KIUH/KIU32-4.png" /></div> <b>4</b> <em><a href="http://sith.huma-num.fr/vocable/322" class="i v322" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V322-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/322&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Jour &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">hrw</a> <a href="http://sith.huma-num.fr/vocable/155" class="i v155" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V155-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/155&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme pronom &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pn</a> <a href="http://sith.huma-num.fr/vocable/519" class="i v519" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V519-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/519&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Particule&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme particule &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">jsṯ</a> <a href="http://sith.huma-num.fr/vocable/33" class="i v33" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V33-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/33&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Majesté &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ḥm</a><span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V4-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne masculin singulier">&#11799;f</span> <a href="http://sith.huma-num.fr/vocable/184" class="i v184" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V184-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/184&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Pour, vers &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">r</a> <a href="http://sith.huma-num.fr/vocable/321" class="i v321" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V321-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/321&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Ville, localité, port &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">dmj</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/vocable/62" class="i v62" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V62-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/62&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Maison, domaine, temple &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pr</a> <a href="http://sith.huma-num.fr/titulature/21" class="i t21" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/titulatures/T21-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/titulature/21&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Ramsès II &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Nom de fils de Rê  &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Rʿ-ms-sw-mry-Jmn</a> <a href="http://sith.huma-num.fr/vocable/166" class="i v166" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V166-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/166&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Sur, concernant &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ḥr</a> <a href="http://sith.huma-num.fr/vocable/20" class="i v20" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V20-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/20&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Faire, agir &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme verbe &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">jrt</a> ḥsst <a href="http://sith.huma-num.fr/vocable/22" class="i v22" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V22-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/22&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Père &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">jt</a><span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V4-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne masculin singulier">&#11799;f</span> <a href="http://sith.huma-num.fr/theonyme/10" class="i d10" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/theonymes/D10-G1.png&quot; height=&quot;35px;&quot; /&gt;  &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/theonyme/10&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Amon-Rê &lt;/span&gt;&lt;/a&gt; &lt;br /&gt; Th&eacute;onyme ou d&eacute;signation divine &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt; ">Jmn-Rʿ</a> Ḥr-ȝḫty <a href="http://sith.huma-num.fr/theonyme/2" class="i d2" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/theonymes/D2-G1.png&quot; height=&quot;35px;&quot; /&gt;  &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/theonyme/2&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Atoum &lt;/span&gt;&lt;/a&gt; &lt;br /&gt; Th&eacute;onyme ou d&eacute;signation divine &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt; ">Jtm</a> <a href="http://sith.huma-num.fr/vocable/56" class="i v56" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V56-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/56&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Seigneur, maître, responsable &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nb</a> <a href="http://sith.huma-num.fr/toponyme/1" class="i l1" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L1-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/1&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Double Pays, Égypte &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Aire g&eacute;ographique &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Tȝwy</a> <a href="http://sith.huma-num.fr/theonyme/54" class="i d54" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/theonymes/D54-G1.png&quot; height=&quot;35px;&quot; /&gt;  &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/theonyme/54&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; L’Héliopolitain, Celui d’Héliopolis &lt;/span&gt;&lt;/a&gt; &lt;br /&gt; Th&eacute;onyme ou d&eacute;signation divine &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt; ">Jwnwy</a> <a href="http://sith.huma-num.fr/theonyme/1" class="i d1" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/theonymes/D1-G1.png&quot; height=&quot;35px;&quot; /&gt;  &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/theonyme/1&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Amon &lt;/span&gt;&lt;/a&gt; &lt;br /&gt; Th&eacute;onyme ou d&eacute;signation divine &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt; ">Jmn</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/titulature/21" class="i t21" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/titulatures/T21-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/titulature/21&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Ramsès II &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Nom de fils de Rê  &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Rʿ-ms-sw-mry-Jmn</a> <a href="http://sith.huma-num.fr/theonyme/7" class="i d7" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/theonymes/D7-G1.png&quot; height=&quot;35px;&quot; /&gt;  &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/theonyme/7&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Ptah &lt;/span&gt;&lt;/a&gt; &lt;br /&gt; Th&eacute;onyme ou d&eacute;signation divine &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt; ">Ptḥ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/titulature/21" class="i t21" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/titulatures/T21-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/titulature/21&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Ramsès II &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Nom de fils de Rê  &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Rʿ-ms-sw-mry-Jmn</a> <a href="http://sith.huma-num.fr/theonyme/25" class="i d25" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/theonymes/D25-G1.png&quot; height=&quot;35px;&quot; /&gt;  &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/theonyme/25&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Seth &lt;/span&gt;&lt;/a&gt; &lt;br /&gt; Th&eacute;onyme ou d&eacute;signation divine &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt; ">Stẖ</a> <a href="http://sith.huma-num.fr/vocable/156" class="i v156" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V156-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/156&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé pour qualifier un substantif (adjectif épithète) &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ʿȝ</a> <a href="http://sith.huma-num.fr/vocable/297" class="i v297" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V297-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/297&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Force, puissance &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pḥty</a> <a href="http://sith.huma-num.fr/vocable/12" class="i v12" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V12-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/12&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Fils &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">sȝ</a> <a href="http://sith.huma-num.fr/theonyme/22" class="i d22" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/theonymes/D22-G1.png&quot; height=&quot;35px;&quot; /&gt;  &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/theonyme/22&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Nout &lt;/span&gt;&lt;/a&gt; &lt;br /&gt; Th&eacute;onyme ou d&eacute;signation divine &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt; ">Nwt</a> <a href="http://sith.huma-num.fr/vocable/170" class="i v170" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V170-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/170&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Comme &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">mj</a> rd<span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V8-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne pluriel">&#11799;sn</span> <a href="http://sith.huma-num.fr/vocable/188" class="i v188" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V188-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/188&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; À, pour &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">n</a><span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V4-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne masculin singulier">&#11799;f</span> <a href="http://sith.huma-num.fr/vocable/163" class="i v163" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V163-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/163&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Temps <i>neheh</i> &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nḥḥ</a> <a href="http://sith.huma-num.fr/vocable/168" class="i v168" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V168-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/168&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Dans, avec, comme, en tant que &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">m</a> <a href="http://sith.huma-num.fr/vocable/35" class="i v35" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V35-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/35&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Fête <i>sed</i>, cérémonie jubilaire &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ḥbw-sd</a> <a href="http://sith.huma-num.fr/vocable/172" class="i v172" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V172-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/172&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Éternité &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ḏt</a> <a href="http://sith.huma-num.fr/vocable/168" class="i v168" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V168-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/168&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Dans, avec, comme, en tant que &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">m</a> <a href="http://sith.huma-num.fr/vocable/75" class="i v75" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V75-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/75&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Année &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">rnpwt</a> ḥtpwt <a href="http://sith.huma-num.fr/vocable/204" class="i v204" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V204-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/204&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Particule énonciative&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme particule &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">jw</a> <a href="http://sith.huma-num.fr/vocable/203" class="i v203" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V203-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/203&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Pays, terre &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">tȝw</a> <a href="http://sith.huma-num.fr/vocable/150" class="i v150" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V150-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/150&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Tout, chaque &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé pour qualifier un substantif (adjectif épithète) &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nbw</a> <a href="http://sith.huma-num.fr/vocable/241" class="i v241" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V241-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/241&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Contrée étrangère &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ḫȝswt</a> <a href="http://sith.huma-num.fr/vocable/150" class="i v150" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V150-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/150&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Tout, chaque &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé pour qualifier un substantif (adjectif épithète) &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nbwt</a> <a href="http://sith.huma-num.fr/vocable/518" class="i v518" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V518-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/518&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Renverser (un ennemi) &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme verbe &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ẖtb</a> <a href="http://sith.huma-num.fr/vocable/293" class="i v293" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V293-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/293&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Sous, chargé de &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ẖr</a> <a href="http://sith.huma-num.fr/vocable/271" class="i v271" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V271-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/271&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Sandale, plante des pieds &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ṯbwty</a><span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V4-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne masculin singulier">&#11799;f</span> <a href="http://sith.huma-num.fr/vocable/171" class="i v171" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V171-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/171&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Éternellement &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme adverbe &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ḏt</a></em><br /><br />
<div id="l5" style="padding-bottom:10px;"><img src="http://sith.huma-num.fr/KIUH/KIU32-5.png" /></div> <b>5</b> <em><a href="http://sith.huma-num.fr/vocable/138" class="i v138" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V138-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/138&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Venir &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme verbe &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">jyt</a> <a href="http://sith.huma-num.fr/vocable/323" class="i v323" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V323-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/323&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Messager, envoyé &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">wpwty</a> <a href="http://sith.huma-num.fr/vocable/10" class="i v10" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V10-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/10&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Roi &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nswt</a> jdnw Nmty ? <span style="color:gray">[...]</span> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> tȝ-nyt-ḥtr <a href="http://sith.huma-num.fr/vocable/323" class="i v323" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V323-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/323&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Messager, envoyé &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">wpwty</a> <a href="http://sith.huma-num.fr/vocable/10" class="i v10" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V10-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/10&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Roi &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nswt</a> <span style="color:gray">[...]</span> <span style="color:gray">[...]</span> <span style="color:gray">[...]</span> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/203" class="i v203" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V203-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/203&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Pays, terre &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">tȝ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <span style="color:gray">[...]</span>tsb <a href="http://sith.huma-num.fr/vocable/323" class="i v323" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V323-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/323&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Messager, envoyé &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">wpwty</a> <span style="color:gray">[...]</span> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/22" class="i l22" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L22-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/22&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Hatti &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Territoire, localit&eacute; ou ethnique du « nord » &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Ḫt</a> Rʿ-ms <span style="color:gray">[...]</span>-myš Ypsr ẖr <span style="color:gray">[...]</span> n <span style="color:gray">[...]</span></em><br /><br />
<div id="l6" style="padding-bottom:10px;"><img src="http://sith.huma-num.fr/KIUH/KIU32-6.png" /></div> <b>6</b> <em><a href="http://sith.huma-num.fr/vocable/85" class="i v85" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V85-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/85&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand, prince, notable &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">wr</a> <a href="http://sith.huma-num.fr/vocable/156" class="i v156" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V156-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/156&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé pour qualifier un substantif (adjectif épithète) &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ʿȝ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/22" class="i l22" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L22-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/22&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Hatti &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Territoire, localit&eacute; ou ethnique du « nord » &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Ḫt</a> Ḫtsr <a href="http://sith.huma-num.fr/vocable/267" class="i v267" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V267-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/267&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Aller chercher, apporter, emporter &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme verbe &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">jn</a>⸗tw <a href="http://sith.huma-num.fr/vocable/184" class="i v184" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V184-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/184&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Pour, vers &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">r</a> <a href="http://sith.huma-num.fr/vocable/331" class="i v331" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V331-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/331&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Palais royal, pharaon &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pr-ʿȝ</a> <a href="http://sith.huma-num.fr/vocable/44" class="i v44" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V44-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/44&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Vie &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ʿnḫ</a> <a href="http://sith.huma-num.fr/vocable/840" class="i v840" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V840-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/840&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Bonne santé, bien-être &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">wḏȝ</a> <a href="http://sith.huma-num.fr/vocable/47" class="i v47" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V47-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/47&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; (Bonne) santé &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">snb</a> <a href="http://sith.huma-num.fr/vocable/184" class="i v184" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V184-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/184&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Pour, vers &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">r</a> <a href="http://sith.huma-num.fr/vocable/332" class="i v332" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V332-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/332&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Demander, réclamer &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme verbe &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">dbḥ</a> <a href="http://sith.huma-num.fr/vocable/333" class="i v333" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V333-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/333&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Paix, calme &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ḥtpw</a> <span style="color:gray">[...]</span> <a href="http://sith.huma-num.fr/titulature/20" class="i t20" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/titulatures/T20-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/titulature/20&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Ramsès II &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Nom de couronnement  &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Wsr-Mȝʿt-Rʿ-stp~n-Rʿ</a> <a href="http://sith.huma-num.fr/vocable/12" class="i v12" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V12-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/12&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Fils &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">sȝ</a> <a href="http://sith.huma-num.fr/theonyme/9" class="i d9" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/theonymes/D9-G1.png&quot; height=&quot;35px;&quot; /&gt;  &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/theonyme/9&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Rê &lt;/span&gt;&lt;/a&gt; &lt;br /&gt; Th&eacute;onyme ou d&eacute;signation divine &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt; ">Rʿ</a> <a href="http://sith.huma-num.fr/titulature/21" class="i t21" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/titulatures/T21-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/titulature/21&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Ramsès II &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Nom de fils de Rê  &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Rʿ-ms-sw-mry-Jmn</a> <a href="http://sith.huma-num.fr/vocable/30" class="i v30" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V30-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/30&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Donner, offrir, accorder &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme verbe &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">dw</a> <a href="http://sith.huma-num.fr/vocable/44" class="i v44" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V44-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/44&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Vie &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ʿnḫ</a> <a href="http://sith.huma-num.fr/vocable/171" class="i v171" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V171-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/171&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Éternellement &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme adverbe &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ḏt</a> <a href="http://sith.huma-num.fr/vocable/192" class="i v192" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V192-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/192&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Éternellement &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme adverbe &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nḥḥ</a> <a href="http://sith.huma-num.fr/vocable/170" class="i v170" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V170-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/170&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Comme &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">mj</a> <a href="http://sith.huma-num.fr/vocable/22" class="i v22" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V22-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/22&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Père &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">jt</a><span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V4-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne masculin singulier">&#11799;f</span> <a href="http://sith.huma-num.fr/theonyme/9" class="i d9" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/theonymes/D9-G1.png&quot; height=&quot;35px;&quot; /&gt;  &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/theonyme/9&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Rê &lt;/span&gt;&lt;/a&gt; &lt;br /&gt; Th&eacute;onyme ou d&eacute;signation divine &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt; ">Rʿ</a> <a href="http://sith.huma-num.fr/vocable/263" class="i v263" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V263-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/263&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Chaque jour, quotidiennement &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">rʿ nb</a> <a href="http://sith.huma-num.fr/vocable/521" class="i v521" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V521-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/521&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Copie, duplicata &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">mjty</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/319" class="i v319" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V319-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/319&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Tablette (pour l’écriture) &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ʿn</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/vocable/320" class="i v320" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V320-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/320&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Argent (métal) &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ḥḏ</a> <a href="http://sith.huma-num.fr/vocable/30" class="i v30" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V30-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/30&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Donner, offrir, accorder &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme verbe &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">rd~n</a> <a href="http://sith.huma-num.fr/vocable/85" class="i v85" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V85-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/85&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand, prince, notable &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">wr</a> <a href="http://sith.huma-num.fr/vocable/156" class="i v156" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V156-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/156&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé pour qualifier un substantif (adjectif épithète) &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ʿȝ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/22" class="i l22" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L22-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/22&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Hatti &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Territoire, localit&eacute; ou ethnique du « nord » &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Ḫt</a> Ḫtsr <a href="http://sith.huma-num.fr/vocable/267" class="i v267" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V267-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/267&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Aller chercher, apporter, emporter &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme verbe &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">jn</a>⸗tw <a href="http://sith.huma-num.fr/vocable/184" class="i v184" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V184-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/184&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Pour, vers &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">r</a> <a href="http://sith.huma-num.fr/vocable/331" class="i v331" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V331-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/331&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Palais royal, pharaon &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pr-ʿȝ</a> <a href="http://sith.huma-num.fr/vocable/44" class="i v44" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V44-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/44&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Vie &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ʿnḫ</a> <a href="http://sith.huma-num.fr/vocable/840" class="i v840" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V840-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/840&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Bonne santé, bien-être &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">wḏȝ</a> <a href="http://sith.huma-num.fr/vocable/47" class="i v47" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V47-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/47&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; (Bonne) santé &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">snb</a> m-ḏrt <a href="http://sith.huma-num.fr/vocable/323" class="i v323" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V323-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/323&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Messager, envoyé &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">wpwty</a><span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V4-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne masculin singulier">&#11799;f</span></em><br /><br />
<div id="l7" style="padding-bottom:10px;"><img src="http://sith.huma-num.fr/KIUH/KIU32-7.png" /></div> <b>7</b> <em>Trtsb <a href="http://sith.huma-num.fr/vocable/323" class="i v323" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V323-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/323&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Messager, envoyé &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">wpwty</a><span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V4-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne masculin singulier">&#11799;f</span> Rʿ-ms <a href="http://sith.huma-num.fr/vocable/184" class="i v184" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V184-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/184&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Pour, vers &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">r</a> <a href="http://sith.huma-num.fr/vocable/332" class="i v332" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V332-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/332&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Demander, réclamer &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme verbe &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">dbḥ</a> <a href="http://sith.huma-num.fr/vocable/333" class="i v333" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V333-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/333&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Paix, calme &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ḥtpw</a> <a href="http://sith.huma-num.fr/vocable/175" class="i v175" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V175-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/175&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Auprès de &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ḫr</a> <a href="http://sith.huma-num.fr/vocable/33" class="i v33" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V33-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/33&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Majesté &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ḥm</a> <span style="color:gray">[ny]</span> <span style="color:gray">[...]</span> <a href="http://sith.huma-num.fr/vocable/12" class="i v12" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V12-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/12&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Fils &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">sȝ</a> <a href="http://sith.huma-num.fr/theonyme/9" class="i d9" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/theonymes/D9-G1.png&quot; height=&quot;35px;&quot; /&gt;  &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/theonyme/9&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Rê &lt;/span&gt;&lt;/a&gt; &lt;br /&gt; Th&eacute;onyme ou d&eacute;signation divine &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt; ">Rʿ</a> <a href="http://sith.huma-num.fr/titulature/21" class="i t21" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/titulatures/T21-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/titulature/21&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Ramsès II &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Nom de fils de Rê  &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Rʿ-ms-sw-mry-Jmn</a> <a href="http://sith.huma-num.fr/vocable/147" class="i v147" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V147-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/147&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Taureau &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">kȝ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/vocable/247" class="i v247" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V247-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/247&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Souverain, roi &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ḥqȝw</a> <a href="http://sith.huma-num.fr/vocable/20" class="i v20" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V20-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/20&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Faire, agir &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme verbe &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">jr</a> <a href="http://sith.huma-num.fr/vocable/256" class="i v256" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V256-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/256&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Limite, frontière &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">tȝšw</a><span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V4-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne masculin singulier">&#11799;f</span> <a href="http://sith.huma-num.fr/vocable/184" class="i v184" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V184-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/184&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Pour, vers &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">r</a> <a href="http://sith.huma-num.fr/vocable/37" class="i v37" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V37-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/37&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Aimer, désirer &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme verbe &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">mr~n</a><span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V4-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne masculin singulier">&#11799;f</span> <a href="http://sith.huma-num.fr/vocable/168" class="i v168" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V168-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/168&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Dans, avec, comme, en tant que &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">m</a> <a href="http://sith.huma-num.fr/vocable/203" class="i v203" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V203-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/203&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Pays, terre &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">tȝ</a> <a href="http://sith.huma-num.fr/vocable/150" class="i v150" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V150-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/150&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Tout, chaque &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé pour qualifier un substantif (adjectif épithète) &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nb</a> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/315" class="i v315" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V315-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/315&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Traité, contrat, rituel &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nyt-ʿ</a> jrrw <a href="http://sith.huma-num.fr/vocable/85" class="i v85" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V85-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/85&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand, prince, notable &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">wr</a> <a href="http://sith.huma-num.fr/vocable/156" class="i v156" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V156-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/156&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé pour qualifier un substantif (adjectif épithète) &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ʿȝ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/22" class="i l22" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L22-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/22&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Hatti &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Territoire, localit&eacute; ou ethnique du « nord » &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Ḫt</a> Ḫtsr <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/522" class="i v522" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V522-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/522&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Puissant, vaillant &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ṯnr</a> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/325" class="i v325" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V325-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/325&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Enfant &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">šrj</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> Mrsr</em><br /><br />
<div id="l8" style="padding-bottom:10px;"><img src="http://sith.huma-num.fr/KIUH/KIU32-8.png" /></div> <b>8</b> <em><a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/85" class="i v85" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V85-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/85&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand, prince, notable &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">wr</a> <a href="http://sith.huma-num.fr/vocable/156" class="i v156" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V156-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/156&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé pour qualifier un substantif (adjectif épithète) &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ʿȝ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/22" class="i l22" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L22-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/22&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Hatti &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Territoire, localit&eacute; ou ethnique du « nord » &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Ḫt</a> <a href="http://sith.huma-num.fr/vocable/522" class="i v522" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V522-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/522&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Puissant, vaillant &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ṯnr</a> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/325" class="i v325" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V325-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/325&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Enfant &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">šrj</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/325" class="i v325" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V325-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/325&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Enfant &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">šrj</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> Spr<span style="color:gray">[...]</span> <a href="http://sith.huma-num.fr/vocable/522" class="i v522" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V522-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/522&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Puissant, vaillant &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ṯnr</a> <a href="http://sith.huma-num.fr/vocable/166" class="i v166" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V166-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/166&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Sur, concernant &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ḥr</a> <a href="http://sith.huma-num.fr/vocable/319" class="i v319" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V319-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/319&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Tablette (pour l’écriture) &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ʿn</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/vocable/320" class="i v320" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V320-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/320&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Argent (métal) &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ḥḏ</a> <a href="http://sith.huma-num.fr/vocable/188" class="i v188" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V188-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/188&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; À, pour &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">n</a> <a href="http://sith.huma-num.fr/titulature/20" class="i t20" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/titulatures/T20-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/titulature/20&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Ramsès II &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Nom de couronnement  &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Wsr-Mȝʿt-Rʿ-stp~n-Rʿ</a> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/247" class="i v247" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V247-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/247&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Souverain, roi &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ḥqȝ</a> <a href="http://sith.huma-num.fr/vocable/156" class="i v156" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V156-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/156&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé pour qualifier un substantif (adjectif épithète) &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ʿȝ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/15" class="i l15" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L15-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/15&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Terre noire, Égypte &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Aire g&eacute;ographique &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Kmt</a> <a href="http://sith.huma-num.fr/vocable/522" class="i v522" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V522-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/522&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Puissant, vaillant &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ṯnr</a> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/325" class="i v325" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V325-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/325&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Enfant &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">šrj</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/titulature/34" class="i t34" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/titulatures/T34-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/titulature/34&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Séthi I<sup>er</sup> &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Nom de couronnement  &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Mn-Mȝʿt-Rʿ</a> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/247" class="i v247" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V247-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/247&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Souverain, roi &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ḥqȝ</a> <a href="http://sith.huma-num.fr/vocable/156" class="i v156" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V156-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/156&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé pour qualifier un substantif (adjectif épithète) &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ʿȝ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/15" class="i l15" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L15-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/15&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Terre noire, Égypte &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Aire g&eacute;ographique &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Kmt</a> <a href="http://sith.huma-num.fr/vocable/522" class="i v522" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V522-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/522&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Puissant, vaillant &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ṯnr</a> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/325" class="i v325" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V325-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/325&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Enfant &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">šrj</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/325" class="i v325" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V325-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/325&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Enfant &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">šrj</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/titulature/58" class="i t58" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/titulatures/T58-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/titulature/58&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Ramsès I<sup>er</sup> &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Nom de couronnement  &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Mn-pḥty-Rʿ</a></em><br /><br />
<div id="l9" style="padding-bottom:10px;"><img src="http://sith.huma-num.fr/KIUH/KIU32-9.png" /></div> <b>9</b> <em><a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/247" class="i v247" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V247-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/247&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Souverain, roi &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ḥqȝ</a> <a href="http://sith.huma-num.fr/vocable/156" class="i v156" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V156-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/156&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé pour qualifier un substantif (adjectif épithète) &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ʿȝ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/15" class="i l15" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L15-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/15&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Terre noire, Égypte &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Aire g&eacute;ographique &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Kmt</a> <a href="http://sith.huma-num.fr/vocable/522" class="i v522" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V522-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/522&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Puissant, vaillant &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ṯnr</a> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/315" class="i v315" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V315-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/315&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Traité, contrat, rituel &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nyt-ʿ</a> <a href="http://sith.huma-num.fr/vocable/153" class="i v153" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V153-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/153&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Beau, bon, parfait, accompli &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé pour qualifier un substantif (adjectif épithète) &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nfr</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> ḥtp <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/vocable/523" class="i v523" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V523-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/523&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Fraternité &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">snsn</a> d ḥtp <span style="color:gray">[...]</span> <span style="color:gray">[...]</span> <span style="color:gray">[...]</span> nḥḥ jr r-ḥȝt n-ḏr nḥḥ jr <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/342" class="i v342" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V342-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/342&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Plan, conseil, avis, décision &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">sḫr</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/247" class="i v247" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V247-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/247&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Souverain, roi &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ḥqȝ</a> <a href="http://sith.huma-num.fr/vocable/156" class="i v156" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V156-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/156&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé pour qualifier un substantif (adjectif épithète) &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ʿȝ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/15" class="i l15" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L15-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/15&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Terre noire, Égypte &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Aire g&eacute;ographique &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Kmt</a> <a href="http://sith.huma-num.fr/vocable/318" class="i v318" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V318-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/318&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Avec, en compagnie de &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">jrm</a> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/85" class="i v85" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V85-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/85&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand, prince, notable &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">wr</a> <a href="http://sith.huma-num.fr/vocable/156" class="i v156" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V156-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/156&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé pour qualifier un substantif (adjectif épithète) &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ʿȝ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/22" class="i l22" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L22-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/22&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Hatti &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Territoire, localit&eacute; ou ethnique du « nord » &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Ḫt</a> <a href="http://sith.huma-num.fr/vocable/524" class="i v524" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V524-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/524&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Particule négative&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme particule &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">bw</a> d <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/14" class="i v14" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V14-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/14&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Dieu, divinité &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nṯr</a> <a href="http://sith.huma-num.fr/vocable/525" class="i v525" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V525-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/525&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Venir à exister, advenir, se transformer &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme verbe &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ḫpr</a> <a href="http://sith.huma-num.fr/vocable/526" class="i v526" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V526-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/526&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Guerre, rébellion &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ḫrwy</a> r-jwd<span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V8-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne pluriel">&#11799;sn</span> <a href="http://sith.huma-num.fr/vocable/168" class="i v168" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V168-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/168&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Dans, avec, comme, en tant que &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">m</a> <a href="http://sith.huma-num.fr/vocable/315" class="i v315" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V315-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/315&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Traité, contrat, rituel &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nyt-ʿ</a> ḫr jr m</em><br /><br />
<div id="l10" style="padding-bottom:10px;"><img src="http://sith.huma-num.fr/KIUH/KIU32-10.png" /></div> <b>10</b> <em>hȝw Mṯnr <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/85" class="i v85" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V85-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/85&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand, prince, notable &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">wr</a> <a href="http://sith.huma-num.fr/vocable/156" class="i v156" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V156-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/156&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé pour qualifier un substantif (adjectif épithète) &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ʿȝ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/22" class="i l22" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L22-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/22&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Hatti &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Territoire, localit&eacute; ou ethnique du « nord » &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Ḫt</a> <a href="http://sith.huma-num.fr/vocable/317" class="i v317" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V317-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/317&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Article possessif masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝy</a><span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V1-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 1<sup>re</sup> personne singulier">&#11799;j</span> <a href="http://sith.huma-num.fr/vocable/527" class="i v527" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V527-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/527&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Frère &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">sn</a> <a href="http://sith.huma-num.fr/vocable/204" class="i v204" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V204-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/204&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Particule énonciative&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme particule &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">jw</a><span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V4-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne masculin singulier">&#11799;f</span> ḥr <a href="http://sith.huma-num.fr/vocable/528" class="i v528" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V528-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/528&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Combattre &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme verbe &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ʿḥȝ</a> <a href="http://sith.huma-num.fr/vocable/318" class="i v318" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V318-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/318&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Avec, en compagnie de &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">jrm</a> <span style="color:gray">[...pȝ]</span> <a href="http://sith.huma-num.fr/vocable/247" class="i v247" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V247-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/247&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Souverain, roi &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ḥqȝ</a> <a href="http://sith.huma-num.fr/vocable/156" class="i v156" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V156-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/156&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé pour qualifier un substantif (adjectif épithète) &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ʿȝ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/15" class="i l15" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L15-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/15&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Terre noire, Égypte &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Aire g&eacute;ographique &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Kmt</a> ḫr jr ḥr-sȝ <a href="http://sith.huma-num.fr/vocable/555" class="i v555" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V555-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/555&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Depuis, à partir de &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">šȝʿ-m</a> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/322" class="i v322" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V322-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/322&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Jour &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">hrw</a> ptr Ḫtsr <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/85" class="i v85" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V85-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/85&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand, prince, notable &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">wr</a> <a href="http://sith.huma-num.fr/vocable/156" class="i v156" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V156-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/156&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé pour qualifier un substantif (adjectif épithète) &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ʿȝ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/22" class="i l22" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L22-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/22&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Hatti &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Territoire, localit&eacute; ou ethnique du « nord » &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Ḫt</a> <span style="color:gray">[...]</span> <a href="http://sith.huma-num.fr/vocable/315" class="i v315" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V315-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/315&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Traité, contrat, rituel &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nyt-ʿ</a> n dt mn <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/342" class="i v342" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V342-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/342&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Plan, conseil, avis, décision &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">sḫr</a> jrrw <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/theonyme/9" class="i d9" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/theonymes/D9-G1.png&quot; height=&quot;35px;&quot; /&gt;  &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/theonyme/9&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Rê &lt;/span&gt;&lt;/a&gt; &lt;br /&gt; Th&eacute;onyme ou d&eacute;signation divine &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt; ">Rʿ</a> jrrw <a href="http://sith.huma-num.fr/theonyme/25" class="i d25" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/theonymes/D25-G1.png&quot; height=&quot;35px;&quot; /&gt;  &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/theonyme/25&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Seth &lt;/span&gt;&lt;/a&gt; &lt;br /&gt; Th&eacute;onyme ou d&eacute;signation divine &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt; ">Stẖ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/203" class="i v203" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V203-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/203&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Pays, terre &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">tȝ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/15" class="i l15" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L15-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/15&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Terre noire, Égypte &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Aire g&eacute;ographique &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Kmt</a></em><br /><br />
<div id="l11" style="padding-bottom:10px;"><img src="http://sith.huma-num.fr/KIUH/KIU32-11.png" /></div> <b>11</b> <em><a href="http://sith.huma-num.fr/vocable/318" class="i v318" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V318-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/318&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Avec, en compagnie de &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">jrm</a> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/203" class="i v203" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V203-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/203&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Pays, terre &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">tȝ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/22" class="i l22" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L22-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/22&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Hatti &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Territoire, localit&eacute; ou ethnique du « nord » &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Ḫt</a> r <a href="http://sith.huma-num.fr/vocable/529" class="i v529" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V529-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/529&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Verbe négatif&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme verbe &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">tm</a> dt <a href="http://sith.huma-num.fr/vocable/525" class="i v525" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V525-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/525&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Venir à exister, advenir, se transformer &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme verbe &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ḫpr</a> <a href="http://sith.huma-num.fr/vocable/526" class="i v526" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V526-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/526&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Guerre, rébellion &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ḫrwy</a> <a href="http://sith.huma-num.fr/vocable/324" class="i v324" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V324-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/324&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Entre &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">r-jwd</a><span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V8-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne pluriel">&#11799;sn</span> r <a href="http://sith.huma-num.fr/vocable/163" class="i v163" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V163-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/163&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Temps <i>neheh</i> &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nḥḥ</a> ptr mḥw sw Ḫtsr <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/85" class="i v85" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V85-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/85&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand, prince, notable &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">wr</a> <a href="http://sith.huma-num.fr/vocable/156" class="i v156" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V156-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/156&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé pour qualifier un substantif (adjectif épithète) &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ʿȝ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/22" class="i l22" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L22-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/22&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Hatti &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Territoire, localit&eacute; ou ethnique du « nord » &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Ḫt</a> <a href="http://sith.huma-num.fr/vocable/168" class="i v168" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V168-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/168&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Dans, avec, comme, en tant que &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">m</a> <a href="http://sith.huma-num.fr/vocable/315" class="i v315" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V315-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/315&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Traité, contrat, rituel &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nyt-ʿ</a> <a href="http://sith.huma-num.fr/vocable/318" class="i v318" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V318-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/318&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Avec, en compagnie de &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">jrm</a> <a href="http://sith.huma-num.fr/titulature/20" class="i t20" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/titulatures/T20-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/titulature/20&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Ramsès II &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Nom de couronnement  &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Wsr-Mȝʿt-Rʿ-stp~n-Rʿ</a> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/247" class="i v247" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V247-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/247&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Souverain, roi &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ḥqȝ</a> <a href="http://sith.huma-num.fr/vocable/156" class="i v156" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V156-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/156&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé pour qualifier un substantif (adjectif épithète) &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ʿȝ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/15" class="i l15" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L15-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/15&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Terre noire, Égypte &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Aire g&eacute;ographique &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Kmt</a> <a href="http://sith.huma-num.fr/vocable/555" class="i v555" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V555-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/555&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Depuis, à partir de &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">šȝʿ-m</a> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/322" class="i v322" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V322-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/322&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Jour &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">hrw</a> r dt <a href="http://sith.huma-num.fr/vocable/525" class="i v525" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V525-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/525&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Venir à exister, advenir, se transformer &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme verbe &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ḫpr</a> ḥtp nfr <a href="http://sith.huma-num.fr/vocable/523" class="i v523" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V523-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/523&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Fraternité &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">snsn</a> nfr <a href="http://sith.huma-num.fr/vocable/324" class="i v324" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V324-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/324&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Entre &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">r-jwd</a><span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V6-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 1<sup>re</sup> personne pluriel">&#11799;n</span> r <a href="http://sith.huma-num.fr/vocable/163" class="i v163" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V163-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/163&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Temps <i>neheh</i> &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nḥḥ</a></em><br /><br />
<div id="l12" style="padding-bottom:10px;"><img src="http://sith.huma-num.fr/KIUH/KIU32-12.png" /></div> <b>12</b> <em><a href="http://sith.huma-num.fr/vocable/204" class="i v204" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V204-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/204&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Particule énonciative&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme particule &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">jw</a><span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V4-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne masculin singulier">&#11799;f</span> <a href="http://sith.huma-num.fr/vocable/530" class="i v530" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V530-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/530&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Fraterniser, s’unir, se joindre &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme verbe &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">snsn</a> <a href="http://sith.huma-num.fr/vocable/318" class="i v318" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V318-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/318&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Avec, en compagnie de &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">jrm</a><span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V1-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 1<sup>re</sup> personne singulier">&#11799;j</span> <a href="http://sith.huma-num.fr/vocable/204" class="i v204" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V204-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/204&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Particule énonciative&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme particule &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">jw</a><span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V4-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne masculin singulier">&#11799;f</span> <a href="http://sith.huma-num.fr/vocable/100" class="i v100" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V100-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/100&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Être en paix, se poser, faire halte, se reposer, être satisfait &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme verbe &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ḥtp</a> <a href="http://sith.huma-num.fr/vocable/318" class="i v318" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V318-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/318&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Avec, en compagnie de &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">jrm</a><span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V1-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 1<sup>re</sup> personne singulier">&#11799;j</span> <a href="http://sith.huma-num.fr/vocable/204" class="i v204" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V204-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/204&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Particule énonciative&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme particule &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">jw</a><span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V1-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 1<sup>re</sup> personne singulier">&#11799;j</span> <a href="http://sith.huma-num.fr/vocable/530" class="i v530" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V530-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/530&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Fraterniser, s’unir, se joindre &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme verbe &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">snsn</a><a href="http://sith.huma-num.fr/vocable/209" class="i v209" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V209-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/209&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Désinence du parfait, première personne du singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Désinence &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">⸗kw</a> <a href="http://sith.huma-num.fr/vocable/318" class="i v318" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V318-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/318&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Avec, en compagnie de &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">jrm</a><span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V4-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne masculin singulier">&#11799;f</span> <a href="http://sith.huma-num.fr/vocable/204" class="i v204" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V204-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/204&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Particule énonciative&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme particule &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">jw</a><span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V1-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 1<sup>re</sup> personne singulier">&#11799;j</span> <a href="http://sith.huma-num.fr/vocable/100" class="i v100" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V100-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/100&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Être en paix, se poser, faire halte, se reposer, être satisfait &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme verbe &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ḥtp</a><a href="http://sith.huma-num.fr/vocable/209" class="i v209" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V209-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/209&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Désinence du parfait, première personne du singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Désinence &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">⸗kw</a> <a href="http://sith.huma-num.fr/vocable/318" class="i v318" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V318-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/318&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Avec, en compagnie de &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">jrm</a><span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V4-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne masculin singulier">&#11799;f</span> <a href="http://sith.huma-num.fr/vocable/184" class="i v184" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V184-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/184&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Pour, vers &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">r</a> <a href="http://sith.huma-num.fr/vocable/163" class="i v163" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V163-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/163&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Temps <i>neheh</i> &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nḥḥ</a> jr ḏr <a href="http://sith.huma-num.fr/vocable/531" class="i v531" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V531-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/531&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; S’en aller &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme verbe &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ḥn</a> Mṯnr <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/85" class="i v85" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V85-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/85&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand, prince, notable &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">wr</a> <a href="http://sith.huma-num.fr/vocable/156" class="i v156" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V156-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/156&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé pour qualifier un substantif (adjectif épithète) &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ʿȝ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/22" class="i l22" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L22-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/22&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Hatti &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Territoire, localit&eacute; ou ethnique du « nord » &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Ḫt</a> <a href="http://sith.huma-num.fr/vocable/317" class="i v317" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V317-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/317&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Article possessif masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝy</a><span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V1-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 1<sup>re</sup> personne singulier">&#11799;j</span> <a href="http://sith.huma-num.fr/vocable/527" class="i v527" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V527-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/527&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Frère &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">sn</a> <a href="http://sith.huma-num.fr/vocable/1691" class="i v1691" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V1691-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/1691&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Après, derrière &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">m-sȝ</a> <a href="http://sith.huma-num.fr/vocable/317" class="i v317" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V317-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/317&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Article possessif masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝy</a><span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V4-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne masculin singulier">&#11799;f</span> šȝy <a href="http://sith.huma-num.fr/vocable/204" class="i v204" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V204-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/204&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Particule énonciative&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme particule &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">jw</a> Ḫtsr <a href="http://sith.huma-num.fr/vocable/166" class="i v166" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V166-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/166&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Sur, concernant &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ḥr</a> <a href="http://sith.huma-num.fr/vocable/53" class="i v53" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V53-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/53&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; S’asseoir &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme verbe &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ḥms</a> <a href="http://sith.huma-num.fr/vocable/168" class="i v168" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V168-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/168&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Dans, avec, comme, en tant que &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">m</a></em><br /><br />
<div id="l13" style="padding-bottom:10px;"><img src="http://sith.huma-num.fr/KIUH/KIU32-13.png" /></div> <b>13</b> <em><a href="http://sith.huma-num.fr/vocable/85" class="i v85" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V85-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/85&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand, prince, notable &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">wr</a> <a href="http://sith.huma-num.fr/vocable/156" class="i v156" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V156-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/156&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé pour qualifier un substantif (adjectif épithète) &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ʿȝ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/22" class="i l22" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L22-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/22&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Hatti &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Territoire, localit&eacute; ou ethnique du « nord » &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Ḫt</a> <a href="http://sith.huma-num.fr/vocable/166" class="i v166" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V166-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/166&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Sur, concernant &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ḥr</a> tȝ <a href="http://sith.huma-num.fr/vocable/532" class="i v532" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V532-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/532&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Siège, trône &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">jsbt</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/vocable/317" class="i v317" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V317-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/317&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Article possessif masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝy</a><span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V4-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne masculin singulier">&#11799;f</span> <a href="http://sith.huma-num.fr/vocable/22" class="i v22" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V22-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/22&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Père &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">jt</a> ptr <a href="http://sith.huma-num.fr/vocable/166" class="i v166" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V166-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/166&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Sur, concernant &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ḥr</a> ḫpr <a href="http://sith.huma-num.fr/vocable/318" class="i v318" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V318-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/318&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Avec, en compagnie de &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">jrm</a> <a href="http://sith.huma-num.fr/titulature/21" class="i t21" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/titulatures/T21-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/titulature/21&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Ramsès II &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Nom de fils de Rê  &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Rʿ-ms-sw-mry-Jmn</a> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/247" class="i v247" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V247-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/247&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Souverain, roi &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ḥqȝ</a> <a href="http://sith.huma-num.fr/vocable/156" class="i v156" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V156-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/156&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé pour qualifier un substantif (adjectif épithète) &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ʿȝ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/15" class="i l15" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L15-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/15&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Terre noire, Égypte &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Aire g&eacute;ographique &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Kmt</a> <a href="http://sith.huma-num.fr/vocable/204" class="i v204" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V204-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/204&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Particule énonciative&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme particule &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">jw</a> <span style="color:gray">[...]</span> ḥtp <span style="color:gray">[...]</span> <a href="http://sith.huma-num.fr/vocable/317" class="i v317" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V317-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/317&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Article possessif masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝy</a><span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V6-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 1<sup>re</sup> personne pluriel">&#11799;n</span> snsn <a href="http://sith.huma-num.fr/vocable/204" class="i v204" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V204-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/204&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Particule énonciative&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme particule &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">jw</a> nfr sw <a href="http://sith.huma-num.fr/vocable/184" class="i v184" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V184-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/184&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Pour, vers &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">r</a> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> ḥtp <a href="http://sith.huma-num.fr/vocable/184" class="i v184" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V184-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/184&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Pour, vers &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">r</a> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> snsn ḥȝwty wnw <a href="http://sith.huma-num.fr/vocable/168" class="i v168" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V168-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/168&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Dans, avec, comme, en tant que &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">m</a> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> tȝ ptr <a href="http://sith.huma-num.fr/vocable/184" class="i v184" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V184-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/184&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Pour, vers &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">r</a><span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V1-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 1<sup>re</sup> personne singulier">&#11799;j</span> <a href="http://sith.huma-num.fr/vocable/168" class="i v168" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V168-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/168&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Dans, avec, comme, en tant que &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">m</a> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/85" class="i v85" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V85-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/85&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand, prince, notable &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">wr</a> <a href="http://sith.huma-num.fr/vocable/156" class="i v156" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V156-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/156&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé pour qualifier un substantif (adjectif épithète) &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ʿȝ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/22" class="i l22" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L22-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/22&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Hatti &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Territoire, localit&eacute; ou ethnique du « nord » &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Ḫt</a> <a href="http://sith.huma-num.fr/vocable/318" class="i v318" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V318-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/318&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Avec, en compagnie de &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">jrm</a></em><br /><br />
<div id="l14" style="padding-bottom:10px;"><img src="http://sith.huma-num.fr/KIUH/KIU32-14.png" /></div> <b>14</b> <em><span style="color:gray">[...]</span> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/247" class="i v247" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V247-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/247&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Souverain, roi &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ḥqȝ</a> <a href="http://sith.huma-num.fr/vocable/156" class="i v156" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V156-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/156&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé pour qualifier un substantif (adjectif épithète) &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ʿȝ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/15" class="i l15" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L15-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/15&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Terre noire, Égypte &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Aire g&eacute;ographique &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Kmt</a> <a href="http://sith.huma-num.fr/vocable/168" class="i v168" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V168-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/168&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Dans, avec, comme, en tant que &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">m</a> <a href="http://sith.huma-num.fr/vocable/333" class="i v333" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V333-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/333&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Paix, calme &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ḥtpw</a> nfr <a href="http://sith.huma-num.fr/vocable/168" class="i v168" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V168-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/168&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Dans, avec, comme, en tant que &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">m</a> snsn nfr <a href="http://sith.huma-num.fr/vocable/414" class="i v414" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V414-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/414&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article pluriel&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nȝ</a> <a href="http://sith.huma-num.fr/vocable/413" class="i v413" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V413-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/413&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Enfant &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ẖrdw</a> <a href="http://sith.huma-num.fr/vocable/414" class="i v414" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V414-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/414&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article pluriel&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nȝ</a> <a href="http://sith.huma-num.fr/vocable/413" class="i v413" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V413-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/413&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Enfant &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ẖrdw</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nyw</a> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/85" class="i v85" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V85-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/85&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand, prince, notable &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">wr</a> <a href="http://sith.huma-num.fr/vocable/156" class="i v156" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V156-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/156&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé pour qualifier un substantif (adjectif épithète) &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ʿȝ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/22" class="i l22" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L22-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/22&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Hatti &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Territoire, localit&eacute; ou ethnique du « nord » &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Ḫt</a> snsn ḥtp <a href="http://sith.huma-num.fr/vocable/318" class="i v318" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V318-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/318&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Avec, en compagnie de &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">jrm</a> <a href="http://sith.huma-num.fr/vocable/414" class="i v414" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V414-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/414&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article pluriel&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nȝ</a> <a href="http://sith.huma-num.fr/vocable/413" class="i v413" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V413-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/413&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Enfant &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ẖrdw</a> <a href="http://sith.huma-num.fr/vocable/414" class="i v414" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V414-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/414&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article pluriel&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nȝ</a> <a href="http://sith.huma-num.fr/vocable/413" class="i v413" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V413-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/413&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Enfant &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ẖrdw</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nyw</a> <a href="http://sith.huma-num.fr/titulature/21" class="i t21" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/titulatures/T21-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/titulature/21&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Ramsès II &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Nom de fils de Rê  &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Rʿ-ms-sw-mry-Jmn</a> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/247" class="i v247" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V247-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/247&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Souverain, roi &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ḥqȝ</a> <a href="http://sith.huma-num.fr/vocable/156" class="i v156" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V156-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/156&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé pour qualifier un substantif (adjectif épithète) &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ʿȝ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/15" class="i l15" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L15-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/15&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Terre noire, Égypte &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Aire g&eacute;ographique &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Kmt</a> <a href="http://sith.huma-num.fr/vocable/204" class="i v204" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V204-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/204&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Particule énonciative&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme particule &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">jw</a> <a href="http://sith.huma-num.fr/vocable/168" class="i v168" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V168-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/168&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Dans, avec, comme, en tant que &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">m</a> <a href="http://sith.huma-num.fr/vocable/317" class="i v317" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V317-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/317&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Article possessif masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝy</a><span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V6-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 1<sup>re</sup> personne pluriel">&#11799;n</span> <a href="http://sith.huma-num.fr/vocable/342" class="i v342" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V342-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/342&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Plan, conseil, avis, décision &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">sḫr</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> snsn <a href="http://sith.huma-num.fr/vocable/317" class="i v317" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V317-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/317&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Article possessif masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝy</a><span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V6-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 1<sup>re</sup> personne pluriel">&#11799;n</span> <a href="http://sith.huma-num.fr/vocable/342" class="i v342" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V342-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/342&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Plan, conseil, avis, décision &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">sḫr</a></em><br /><br />
<div id="l15" style="padding-bottom:10px;"><img src="http://sith.huma-num.fr/KIUH/KIU32-15.png" /></div> <b>15</b> <em><span style="color:gray">[...]</span> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/15" class="i l15" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L15-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/15&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Terre noire, Égypte &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Aire g&eacute;ographique &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Kmt</a> <a href="http://sith.huma-num.fr/vocable/318" class="i v318" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V318-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/318&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Avec, en compagnie de &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">jrm</a> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/203" class="i v203" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V203-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/203&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Pays, terre &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">tȝ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/22" class="i l22" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L22-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/22&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Hatti &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Territoire, localit&eacute; ou ethnique du « nord » &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Ḫt</a> ḥtp snsn <a href="http://sith.huma-num.fr/vocable/170" class="i v170" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V170-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/170&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Comme &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">mj</a> qd<span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V6-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 1<sup>re</sup> personne pluriel">&#11799;n</span> <a href="http://sith.huma-num.fr/vocable/184" class="i v184" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V184-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/184&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Pour, vers &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">r</a> <a href="http://sith.huma-num.fr/vocable/163" class="i v163" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V163-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/163&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Temps <i>neheh</i> &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nḥḥ</a> <a href="http://sith.huma-num.fr/vocable/204" class="i v204" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V204-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/204&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Particule énonciative&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme particule &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">jw</a> <a href="http://sith.huma-num.fr/vocable/524" class="i v524" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V524-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/524&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Particule négative&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme particule &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">bw</a> <a href="http://sith.huma-num.fr/vocable/525" class="i v525" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V525-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/525&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Venir à exister, advenir, se transformer &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme verbe &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ḫpr~n</a> <a href="http://sith.huma-num.fr/vocable/526" class="i v526" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V526-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/526&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Guerre, rébellion &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ḫrwy</a> <a href="http://sith.huma-num.fr/vocable/324" class="i v324" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V324-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/324&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Entre &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">r-jwd</a><span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V8-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne pluriel">&#11799;sn</span> <a href="http://sith.huma-num.fr/vocable/184" class="i v184" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V184-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/184&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Pour, vers &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">r</a> <a href="http://sith.huma-num.fr/vocable/163" class="i v163" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V163-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/163&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Temps <i>neheh</i> &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nḥḥ</a> <a href="http://sith.huma-num.fr/vocable/204" class="i v204" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V204-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/204&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Particule énonciative&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme particule &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">jw</a> <a href="http://sith.huma-num.fr/vocable/524" class="i v524" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V524-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/524&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Particule négative&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme particule &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">bw</a> jr <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/85" class="i v85" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V85-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/85&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand, prince, notable &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">wr</a> <a href="http://sith.huma-num.fr/vocable/156" class="i v156" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V156-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/156&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé pour qualifier un substantif (adjectif épithète) &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ʿȝ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/22" class="i l22" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L22-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/22&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Hatti &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Territoire, localit&eacute; ou ethnique du « nord » &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Ḫt</a> th <a href="http://sith.huma-num.fr/vocable/184" class="i v184" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V184-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/184&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Pour, vers &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">r</a> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/203" class="i v203" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V203-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/203&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Pays, terre &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">tȝ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/15" class="i l15" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L15-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/15&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Terre noire, Égypte &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Aire g&eacute;ographique &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Kmt</a> <a href="http://sith.huma-num.fr/vocable/184" class="i v184" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V184-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/184&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Pour, vers &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">r</a> <a href="http://sith.huma-num.fr/vocable/163" class="i v163" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V163-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/163&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Temps <i>neheh</i> &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nḥḥ</a> <a href="http://sith.huma-num.fr/vocable/184" class="i v184" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V184-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/184&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Pour, vers &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">r</a> <a href="http://sith.huma-num.fr/vocable/1622" class="i v1622" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V1622-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/1622&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Voler, s’emparer &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme verbe &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">jṯȝ</a> <a href="http://sith.huma-num.fr/vocable/1248" class="i v1248" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V1248-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/1248&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Quelque chose &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nkt</a> <a href="http://sith.huma-num.fr/vocable/168" class="i v168" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V168-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/168&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Dans, avec, comme, en tant que &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">jm</a><span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V4-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne masculin singulier">&#11799;f</span> <a href="http://sith.huma-num.fr/vocable/204" class="i v204" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V204-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/204&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Particule énonciative&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme particule &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">jw</a> <a href="http://sith.huma-num.fr/vocable/524" class="i v524" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V524-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/524&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Particule négative&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme particule &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">bw</a> jr <a href="http://sith.huma-num.fr/titulature/20" class="i t20" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/titulatures/T20-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/titulature/20&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Ramsès II &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Nom de couronnement  &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Wsr-Mȝʿt-Rʿ-stp~n-Rʿ</a> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/247" class="i v247" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V247-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/247&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Souverain, roi &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ḥqȝ</a> <a href="http://sith.huma-num.fr/vocable/156" class="i v156" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V156-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/156&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé pour qualifier un substantif (adjectif épithète) &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ʿȝ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/15" class="i l15" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L15-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/15&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Terre noire, Égypte &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Aire g&eacute;ographique &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Kmt</a> th <a href="http://sith.huma-num.fr/vocable/184" class="i v184" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V184-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/184&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Pour, vers &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">r</a> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/203" class="i v203" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V203-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/203&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Pays, terre &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">tȝ</a></em><br /><br />
<div id="l16" style="padding-bottom:10px;"><img src="http://sith.huma-num.fr/KIUH/KIU32-16.png" /></div> <b>16</b> <em><span style="color:gray">[...]</span> jm<span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V4-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne masculin singulier">&#11799;f</span> <a href="http://sith.huma-num.fr/vocable/184" class="i v184" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V184-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/184&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Pour, vers &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">r</a> <a href="http://sith.huma-num.fr/vocable/163" class="i v163" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V163-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/163&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Temps <i>neheh</i> &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nḥḥ</a> jr <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/315" class="i v315" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V315-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/315&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Traité, contrat, rituel &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nyt-ʿ</a> mty wnw dy <a href="http://sith.huma-num.fr/vocable/168" class="i v168" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V168-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/168&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Dans, avec, comme, en tant que &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">m</a> hȝw Sprr <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/85" class="i v85" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V85-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/85&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand, prince, notable &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">wr</a> <a href="http://sith.huma-num.fr/vocable/156" class="i v156" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V156-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/156&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé pour qualifier un substantif (adjectif épithète) &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ʿȝ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/22" class="i l22" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L22-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/22&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Hatti &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Territoire, localit&eacute; ou ethnique du « nord » &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Ḫt</a> <a href="http://sith.huma-num.fr/vocable/965" class="i v965" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V965-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/965&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Pareillement, de même &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme adverbe &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">m-mjtt</a> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/315" class="i v315" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V315-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/315&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Traité, contrat, rituel &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nyt-ʿ</a> mty wnw <a href="http://sith.huma-num.fr/vocable/168" class="i v168" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V168-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/168&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Dans, avec, comme, en tant que &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">m</a> hȝw Mṯnr <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/85" class="i v85" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V85-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/85&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand, prince, notable &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">wr</a> <a href="http://sith.huma-num.fr/vocable/156" class="i v156" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V156-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/156&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé pour qualifier un substantif (adjectif épithète) &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ʿȝ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/22" class="i l22" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L22-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/22&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Hatti &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Territoire, localit&eacute; ou ethnique du « nord » &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Ḫt</a> <a href="http://sith.huma-num.fr/vocable/317" class="i v317" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V317-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/317&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Article possessif masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝy</a><span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V1-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 1<sup>re</sup> personne singulier">&#11799;j</span> <a href="http://sith.huma-num.fr/vocable/22" class="i v22" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V22-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/22&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Père &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">jt</a> mḥw<span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V1-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 1<sup>re</sup> personne singulier">&#11799;j</span> jm<span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V4-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne masculin singulier">&#11799;f</span> ptr mḥw <a href="http://sith.huma-num.fr/titulature/21" class="i t21" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/titulatures/T21-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/titulature/21&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Ramsès II &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Nom de fils de Rê  &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Rʿ-ms-sw-mry-Jmn</a> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/247" class="i v247" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V247-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/247&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Souverain, roi &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ḥqȝ</a> <a href="http://sith.huma-num.fr/vocable/156" class="i v156" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V156-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/156&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé pour qualifier un substantif (adjectif épithète) &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ʿȝ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/15" class="i l15" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L15-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/15&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Terre noire, Égypte &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Aire g&eacute;ographique &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Kmt</a></em><br /><br />
<div id="l17" style="padding-bottom:10px;"><img src="http://sith.huma-num.fr/KIUH/KIU32-17.png" /></div> <b>17</b> <em><span style="color:gray">[...]</span> <a href="http://sith.huma-num.fr/vocable/318" class="i v318" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V318-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/318&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Avec, en compagnie de &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">jrm</a><span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V6-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 1<sup>re</sup> personne pluriel">&#11799;n</span> n sp <a href="http://sith.huma-num.fr/vocable/555" class="i v555" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V555-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/555&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Depuis, à partir de &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">šȝʿ-m</a> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/322" class="i v322" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V322-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/322&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Jour &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">hrw</a> mḥw<span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V6-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 1<sup>re</sup> personne pluriel">&#11799;n</span> jm<span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V4-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne masculin singulier">&#11799;f</span> jrr<span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V6-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 1<sup>re</sup> personne pluriel">&#11799;n</span> <a href="http://sith.huma-num.fr/vocable/168" class="i v168" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V168-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/168&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Dans, avec, comme, en tant que &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">m</a> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/342" class="i v342" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V342-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/342&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Plan, conseil, avis, décision &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">sḫr</a> mty jr jw <a href="http://sith.huma-num.fr/vocable/1187" class="i v1187" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V1187-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/1187&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Autre &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ky</a> ḫrwy <a href="http://sith.huma-num.fr/vocable/184" class="i v184" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V184-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/184&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Pour, vers &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">r</a> <a href="http://sith.huma-num.fr/vocable/414" class="i v414" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V414-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/414&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article pluriel&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nȝ</a> <a href="http://sith.huma-num.fr/vocable/203" class="i v203" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V203-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/203&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Pays, terre &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">tȝw</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nyw</a> <a href="http://sith.huma-num.fr/titulature/20" class="i t20" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/titulatures/T20-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/titulature/20&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Ramsès II &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Nom de couronnement  &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Wsr-Mȝʿt-Rʿ-stp~n-Rʿ</a> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/247" class="i v247" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V247-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/247&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Souverain, roi &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ḥqȝ</a> <a href="http://sith.huma-num.fr/vocable/156" class="i v156" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V156-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/156&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé pour qualifier un substantif (adjectif épithète) &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ʿȝ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/15" class="i l15" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L15-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/15&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Terre noire, Égypte &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Aire g&eacute;ographique &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Kmt</a> mtw<span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V4-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne masculin singulier">&#11799;f</span> <a href="http://sith.huma-num.fr/vocable/981" class="i v981" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V981-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/981&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Envoyer &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme verbe &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">hȝb</a> n <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/85" class="i v85" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V85-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/85&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand, prince, notable &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">wr</a> <a href="http://sith.huma-num.fr/vocable/156" class="i v156" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V156-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/156&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé pour qualifier un substantif (adjectif épithète) &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ʿȝ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/22" class="i l22" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L22-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/22&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Hatti &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Territoire, localit&eacute; ou ethnique du « nord » &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Ḫt</a> <a href="http://sith.huma-num.fr/vocable/184" class="i v184" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V184-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/184&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Pour, vers &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">r</a> ḏd my m-d<span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V1-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 1<sup>re</sup> personne singulier">&#11799;j</span> <a href="http://sith.huma-num.fr/vocable/168" class="i v168" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V168-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/168&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Dans, avec, comme, en tant que &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">m</a> nḫtw <a href="http://sith.huma-num.fr/vocable/184" class="i v184" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V184-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/184&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Pour, vers &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">r</a><span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V4-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne masculin singulier">&#11799;f</span> jr <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/85" class="i v85" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V85-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/85&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand, prince, notable &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">wr</a> <a href="http://sith.huma-num.fr/vocable/156" class="i v156" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V156-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/156&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé pour qualifier un substantif (adjectif épithète) &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ʿȝ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/22" class="i l22" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L22-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/22&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Hatti &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Territoire, localit&eacute; ou ethnique du « nord » &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Ḫt</a></em><br /><br />
<div id="l18" style="padding-bottom:10px;"><img src="http://sith.huma-num.fr/KIUH/KIU32-18.png" /></div> <b>18</b> <em><span style="color:gray">[...]</span> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/85" class="i v85" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V85-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/85&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand, prince, notable &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">wr</a> <a href="http://sith.huma-num.fr/vocable/156" class="i v156" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V156-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/156&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé pour qualifier un substantif (adjectif épithète) &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ʿȝ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/22" class="i l22" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L22-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/22&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Hatti &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Territoire, localit&eacute; ou ethnique du « nord » &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Ḫt</a> <a href="http://sith.huma-num.fr/vocable/1633" class="i v1633" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V1633-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/1633&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Tuer &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme verbe &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ẖdb</a> <a href="http://sith.huma-num.fr/vocable/317" class="i v317" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V317-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/317&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Article possessif masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝy</a><span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V4-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne masculin singulier">&#11799;f</span> ḫrwy ḫr jr <a href="http://sith.huma-num.fr/vocable/204" class="i v204" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V204-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/204&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Particule énonciative&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme particule &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">jw</a> <a href="http://sith.huma-num.fr/vocable/1029" class="i v1029" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V1029-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/1029&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Particule négative&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme particule &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">bn</a> jb <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/85" class="i v85" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V85-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/85&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand, prince, notable &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">wr</a> <a href="http://sith.huma-num.fr/vocable/156" class="i v156" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V156-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/156&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé pour qualifier un substantif (adjectif épithète) &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ʿȝ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/22" class="i l22" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L22-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/22&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Hatti &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Territoire, localit&eacute; ou ethnique du « nord » &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Ḫt</a> šmt <a href="http://sith.huma-num.fr/vocable/204" class="i v204" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V204-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/204&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Particule énonciative&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme particule &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">jw</a><span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V4-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne masculin singulier">&#11799;f</span> <a href="http://sith.huma-num.fr/vocable/166" class="i v166" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V166-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/166&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Sur, concernant &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ḥr</a> dt ḥnn <a href="http://sith.huma-num.fr/vocable/317" class="i v317" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V317-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/317&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Article possessif masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝy</a><span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V4-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne masculin singulier">&#11799;f</span> <a href="http://sith.huma-num.fr/vocable/517" class="i v517" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V517-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/517&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Troupe, armée &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">mšʿ</a> tȝ-nyt-ḥtr mtw<span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V4-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne masculin singulier">&#11799;f</span> <a href="http://sith.huma-num.fr/vocable/1633" class="i v1633" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V1633-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/1633&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Tuer &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme verbe &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ẖdb</a> <a href="http://sith.huma-num.fr/vocable/317" class="i v317" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V317-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/317&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Article possessif masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝy</a><span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V4-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne masculin singulier">&#11799;f</span> ḫrwy <a href="http://sith.huma-num.fr/vocable/1553" class="i v1553" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V1553-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/1553&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Ou bien &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme particule &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">rȝ-pw</a> jr qnd <a href="http://sith.huma-num.fr/titulature/21" class="i t21" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/titulatures/T21-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/titulature/21&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Ramsès II &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Nom de fils de Rê  &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Rʿ-ms-sw-mry-Jmn</a></em><br /><br />
<div id="l19" style="padding-bottom:10px;"><img src="http://sith.huma-num.fr/KIUH/KIU32-19.png" /></div> <b>19</b> <em><span style="color:gray">[...]</span> r bȝkw swt <a href="http://sith.huma-num.fr/vocable/204" class="i v204" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V204-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/204&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Particule énonciative&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme particule &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">jw</a> jr<span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V8-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne pluriel">&#11799;sn</span> ky ṯȝy r<span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V4-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne masculin singulier">&#11799;f</span> mtw<span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V4-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne masculin singulier">&#11799;f</span> šmt r ẖdb⸗w jr <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/85" class="i v85" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V85-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/85&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand, prince, notable &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">wr</a> <a href="http://sith.huma-num.fr/vocable/156" class="i v156" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V156-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/156&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé pour qualifier un substantif (adjectif épithète) &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ʿȝ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/22" class="i l22" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L22-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/22&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Hatti &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Territoire, localit&eacute; ou ethnique du « nord » &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Ḫt</a> <a href="http://sith.huma-num.fr/vocable/318" class="i v318" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V318-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/318&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Avec, en compagnie de &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">jrm</a><span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V4-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne masculin singulier">&#11799;f</span> <span style="color:gray">[...]</span> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> nty nb <a href="http://sith.huma-num.fr/vocable/204" class="i v204" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V204-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/204&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Particule énonciative&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme particule &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">jw</a>⸗w r qnd r⸗w ḫr <span style="color:gray">[...]</span> ky ḫrwy r <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/85" class="i v85" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V85-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/85&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand, prince, notable &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">wr</a> <span style="color:gray">[...]</span> <a href="http://sith.huma-num.fr/titulature/20" class="i t20" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/titulatures/T20-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/titulature/20&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Ramsès II &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Nom de couronnement  &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Wsr-Mȝʿt-Rʿ-stp~n-Rʿ</a> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a></em><br /><br />
<div id="l20" style="padding-bottom:10px;"><img src="http://sith.huma-num.fr/KIUH/KIU32-20.png" /></div> <b>20</b> <em><span style="color:gray">[...]</span> <a href="http://sith.huma-num.fr/vocable/138" class="i v138" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V138-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/138&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Venir &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme verbe &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">jyt</a> <a href="http://sith.huma-num.fr/vocable/188" class="i v188" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V188-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/188&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; À, pour &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">n</a><span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V4-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne masculin singulier">&#11799;f</span> <a href="http://sith.huma-num.fr/vocable/168" class="i v168" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V168-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/168&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Dans, avec, comme, en tant que &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">m</a> <a href="http://sith.huma-num.fr/vocable/313" class="i v313" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V313-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/313&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Force, vigueur, victoire &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nḫt</a> <a href="http://sith.huma-num.fr/vocable/184" class="i v184" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V184-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/184&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Pour, vers &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">r</a> <a href="http://sith.huma-num.fr/vocable/1633" class="i v1633" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V1633-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/1633&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Tuer &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme verbe &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ẖdb</a> <a href="http://sith.huma-num.fr/vocable/317" class="i v317" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V317-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/317&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Article possessif masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝy</a><span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V4-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne masculin singulier">&#11799;f</span> ḫrwy jr <a href="http://sith.huma-num.fr/vocable/204" class="i v204" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V204-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/204&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Particule énonciative&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme particule &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">jw</a> jb <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/titulature/21" class="i t21" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/titulatures/T21-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/titulature/21&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Ramsès II &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Nom de fils de Rê  &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Rʿ-ms-sw-mry-Jmn</a> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/247" class="i v247" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V247-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/247&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Souverain, roi &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ḥqȝ</a> <a href="http://sith.huma-num.fr/vocable/156" class="i v156" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V156-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/156&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé pour qualifier un substantif (adjectif épithète) &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ʿȝ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/15" class="i l15" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L15-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/15&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Terre noire, Égypte &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Aire g&eacute;ographique &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Kmt</a> <a href="http://sith.huma-num.fr/vocable/184" class="i v184" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V184-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/184&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Pour, vers &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">r</a> <a href="http://sith.huma-num.fr/vocable/138" class="i v138" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V138-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/138&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Venir &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme verbe &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">jyt</a> <a href="http://sith.huma-num.fr/vocable/204" class="i v204" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V204-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/204&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Particule énonciative&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme particule &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">jw</a><span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V4-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne masculin singulier">&#11799;f</span> <span style="color:gray">[...]</span> <a href="http://sith.huma-num.fr/vocable/184" class="i v184" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V184-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/184&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Pour, vers &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">r</a> <a href="http://sith.huma-num.fr/vocable/204" class="i v204" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V204-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/204&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Particule énonciative&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme particule &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">jw</a> <span style="color:gray">[...]</span> <span style="color:gray">[...]</span> <span style="color:gray">[...]</span></em><br /><br />
<div id="l21" style="padding-bottom:10px;"><img src="http://sith.huma-num.fr/KIUH/KIU32-21.png" /></div> <b>21</b> <em><span style="color:gray">[...]</span> tȝ-nyt-ḥtr m-d ʿnn wšb n <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/203" class="i v203" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V203-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/203&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Pays, terre &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">tȝ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/22" class="i l22" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L22-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/22&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Hatti &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Territoire, localit&eacute; ou ethnique du « nord » &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Ḫt</a> ḫr jr th bȝkw n <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/85" class="i v85" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V85-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/85&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand, prince, notable &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">wr</a> <a href="http://sith.huma-num.fr/vocable/156" class="i v156" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V156-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/156&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé pour qualifier un substantif (adjectif épithète) &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ʿȝ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/22" class="i l22" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L22-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/22&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Hatti &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Territoire, localit&eacute; ou ethnique du « nord » &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Ḫt</a> <a href="http://sith.huma-num.fr/vocable/184" class="i v184" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V184-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/184&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Pour, vers &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">r</a><span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V4-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne masculin singulier">&#11799;f</span> mtw <a href="http://sith.huma-num.fr/titulature/21" class="i t21" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/titulatures/T21-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/titulature/21&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Ramsès II &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Nom de fils de Rê  &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Rʿ-ms-sw-mry-Jmn</a> <span style="color:gray">[...]</span> <span style="color:gray">[...]</span> <span style="color:gray">[...]</span> <a href="http://sith.huma-num.fr/vocable/247" class="i v247" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V247-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/247&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Souverain, roi &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ḥqȝ</a> <span style="color:gray">[...]</span> <span style="color:gray">[...]</span> <span style="color:gray">[...]</span> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/22" class="i l22" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L22-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/22&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Hatti &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Territoire, localit&eacute; ou ethnique du « nord » &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Ḫt</a> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <span style="color:gray">[...]</span></em><br /><br />
<div id="l22" style="padding-bottom:10px;"><img src="http://sith.huma-num.fr/KIUH/KIU32-22.png" /></div> <b>22</b> <em><span style="color:gray">[...]</span> pȝ ʿnḫ kw ḏd <a href="http://sith.huma-num.fr/vocable/204" class="i v204" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V204-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/204&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Particule énonciative&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme particule &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">jw</a><span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V1-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 1<sup>re</sup> personne singulier">&#11799;j</span> <a href="http://sith.huma-num.fr/vocable/184" class="i v184" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V184-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/184&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Pour, vers &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">r</a> šmt m sȝ <a href="http://sith.huma-num.fr/vocable/317" class="i v317" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V317-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/317&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Article possessif masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝy</a><span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V1-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 1<sup>re</sup> personne singulier">&#11799;j</span> šȝy ḫr <a href="http://sith.huma-num.fr/titulature/21" class="i t21" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/titulatures/T21-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/titulature/21&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Ramsès II &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Nom de fils de Rê  &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Rʿ-ms-sw-mry-Jmn</a> pȝ <a href="http://sith.huma-num.fr/vocable/247" class="i v247" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V247-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/247&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Souverain, roi &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ḥqȝ</a> <a href="http://sith.huma-num.fr/vocable/156" class="i v156" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V156-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/156&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé pour qualifier un substantif (adjectif épithète) &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ʿȝ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/15" class="i l15" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L15-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/15&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Terre noire, Égypte &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Aire g&eacute;ographique &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Kmt</a> <a href="http://sith.huma-num.fr/vocable/9" class="i v9" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V9-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/9&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Vivre &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme verbe &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ʿnḫ</a> <a href="http://sith.huma-num.fr/vocable/184" class="i v184" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V184-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/184&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Pour, vers &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">r</a> <a href="http://sith.huma-num.fr/vocable/163" class="i v163" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V163-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/163&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Temps <i>neheh</i> &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nḥḥ</a> mtw⸗tw <span style="color:gray">[...]</span> pȝ <span style="color:gray">[tȝ]</span> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/22" class="i l22" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L22-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/22&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Hatti &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Territoire, localit&eacute; ou ethnique du « nord » &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Ḫt</a> <span style="color:gray">[...]</span> pȝ <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <span style="color:gray">[...]</span> nb <a href="http://sith.huma-num.fr/vocable/184" class="i v184" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V184-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/184&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Pour, vers &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">r</a> <a href="http://sith.huma-num.fr/vocable/529" class="i v529" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V529-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/529&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Verbe négatif&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme verbe &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">tm</a> dt jry m <span style="color:gray">[...]</span> <span style="color:gray">[...]</span> <span style="color:gray">[...]</span></em><br /><br />
<div id="l23" style="padding-bottom:10px;"><img src="http://sith.huma-num.fr/KIUH/KIU32-23.png" /></div> <b>23</b> <em><span style="color:gray">[...]</span><span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V8-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne pluriel">&#11799;sn</span> <a href="http://sith.huma-num.fr/vocable/184" class="i v184" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V184-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/184&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Pour, vers &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">r</a> d⸗tw<span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V4-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne masculin singulier">&#11799;f</span> <a href="http://sith.huma-num.fr/vocable/188" class="i v188" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V188-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/188&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; À, pour &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">n</a><span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V8-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne pluriel">&#11799;sn</span> <a href="http://sith.huma-num.fr/vocable/184" class="i v184" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V184-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/184&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Pour, vers &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">r</a> nb m dt gr <a href="http://sith.huma-num.fr/titulature/20" class="i t20" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/titulatures/T20-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/titulature/20&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Ramsès II &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Nom de couronnement  &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Wsr-Mȝʿt-Rʿ-stp~n-Rʿ</a> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/247" class="i v247" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V247-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/247&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Souverain, roi &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ḥqȝ</a> <a href="http://sith.huma-num.fr/vocable/156" class="i v156" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V156-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/156&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé pour qualifier un substantif (adjectif épithète) &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ʿȝ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/15" class="i l15" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L15-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/15&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Terre noire, Égypte &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Aire g&eacute;ographique &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Kmt</a> m rȝ<span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V4-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne masculin singulier">&#11799;f</span> <a href="http://sith.huma-num.fr/vocable/184" class="i v184" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V184-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/184&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Pour, vers &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">r</a> <a href="http://sith.huma-num.fr/vocable/163" class="i v163" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V163-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/163&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Temps <i>neheh</i> &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nḥḥ</a> jmm jwt<span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V4-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne masculin singulier">&#11799;f</span> <a href="http://sith.huma-num.fr/vocable/188" class="i v188" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V188-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/188&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; À, pour &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">n</a><span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V4-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne masculin singulier">&#11799;f</span> mtw<span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V4-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne masculin singulier">&#11799;f</span> <span style="color:gray">[...]</span> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/203" class="i v203" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V203-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/203&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Pays, terre &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">tȝ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/22" class="i l22" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L22-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/22&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Hatti &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Territoire, localit&eacute; ou ethnique du « nord » &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Ḫt</a> mtw<span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V4-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne masculin singulier">&#11799;f</span> Ȝʿnn wšb <a href="http://sith.huma-num.fr/vocable/188" class="i v188" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V188-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/188&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; À, pour &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">n</a> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/85" class="i v85" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V85-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/85&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand, prince, notable &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">wr</a> <a href="http://sith.huma-num.fr/vocable/156" class="i v156" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V156-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/156&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé pour qualifier un substantif (adjectif épithète) &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ʿȝ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/22" class="i l22" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L22-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/22&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Hatti &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Territoire, localit&eacute; ou ethnique du « nord » &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Ḫt</a> <a href="http://sith.huma-num.fr/vocable/965" class="i v965" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V965-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/965&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Pareillement, de même &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme adverbe &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">m-mjtt</a> <span style="color:gray">[...]</span> <span style="color:gray">[...]</span> <span style="color:gray">[...]</span> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/85" class="i v85" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V85-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/85&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand, prince, notable &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">wr</a> <a href="http://sith.huma-num.fr/vocable/156" class="i v156" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V156-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/156&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé pour qualifier un substantif (adjectif épithète) &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ʿȝ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/22" class="i l22" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L22-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/22&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Hatti &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Territoire, localit&eacute; ou ethnique du « nord » &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Ḫt</a> m-rȝ-pw wʿ dmj</em><br /><br />
<div id="l24" style="padding-bottom:10px;"><img src="http://sith.huma-num.fr/KIUH/KIU32-24.png" /></div> <b>24</b> <em><span style="color:gray">[...]</span> ny <a href="http://sith.huma-num.fr/vocable/203" class="i v203" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V203-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/203&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Pays, terre &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">tȝw</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nyw</a> <a href="http://sith.huma-num.fr/titulature/21" class="i t21" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/titulatures/T21-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/titulature/21&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Ramsès II &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Nom de fils de Rê  &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Rʿ-ms-sw-mry-Jmn</a> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/247" class="i v247" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V247-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/247&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Souverain, roi &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ḥqȝ</a> <a href="http://sith.huma-num.fr/vocable/156" class="i v156" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V156-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/156&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé pour qualifier un substantif (adjectif épithète) &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ʿȝ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/15" class="i l15" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L15-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/15&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Terre noire, Égypte &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Aire g&eacute;ographique &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Kmt</a> mtw<span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V8-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne pluriel">&#11799;sn</span> <a href="http://sith.huma-num.fr/vocable/138" class="i v138" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V138-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/138&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Venir &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme verbe &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">jyt</a> <a href="http://sith.huma-num.fr/vocable/188" class="i v188" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V188-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/188&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; À, pour &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">n</a> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/85" class="i v85" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V85-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/85&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand, prince, notable &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">wr</a> <a href="http://sith.huma-num.fr/vocable/156" class="i v156" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V156-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/156&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé pour qualifier un substantif (adjectif épithète) &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ʿȝ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/22" class="i l22" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L22-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/22&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Hatti &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Territoire, localit&eacute; ou ethnique du « nord » &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Ḫt</a> <a href="http://sith.huma-num.fr/vocable/1029" class="i v1029" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V1029-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/1029&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Particule négative&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme particule &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">bn</a> jr <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/85" class="i v85" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V85-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/85&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand, prince, notable &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">wr</a> <a href="http://sith.huma-num.fr/vocable/156" class="i v156" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V156-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/156&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé pour qualifier un substantif (adjectif épithète) &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ʿȝ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/22" class="i l22" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L22-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/22&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Hatti &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Territoire, localit&eacute; ou ethnique du « nord » &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Ḫt</a> <a href="http://sith.huma-num.fr/vocable/184" class="i v184" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V184-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/184&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Pour, vers &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">r</a> šsp⸗w jr <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/85" class="i v85" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V85-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/85&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand, prince, notable &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">wr</a> <a href="http://sith.huma-num.fr/vocable/156" class="i v156" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V156-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/156&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé pour qualifier un substantif (adjectif épithète) &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ʿȝ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/22" class="i l22" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L22-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/22&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Hatti &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Territoire, localit&eacute; ou ethnique du « nord » &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Ḫt</a> dt jn⸗tw⸗w <a href="http://sith.huma-num.fr/vocable/188" class="i v188" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V188-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/188&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; À, pour &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">n</a> <a href="http://sith.huma-num.fr/titulature/20" class="i t20" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/titulatures/T20-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/titulature/20&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Ramsès II &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Nom de couronnement  &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Wsr-Mȝʿt-Rʿ-stp~n-Rʿ</a> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/247" class="i v247" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V247-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/247&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Souverain, roi &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ḥqȝ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/15" class="i l15" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L15-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/15&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Terre noire, Égypte &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Aire g&eacute;ographique &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Kmt</a> <a href="http://sith.huma-num.fr/vocable/317" class="i v317" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V317-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/317&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Article possessif masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝy</a><span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V8-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne pluriel">&#11799;sn</span> <a href="http://sith.huma-num.fr/vocable/56" class="i v56" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V56-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/56&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Seigneur, maître, responsable &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nb</a> <a href="http://sith.huma-num.fr/vocable/44" class="i v44" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V44-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/44&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Vie &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ʿnḫ</a> <a href="http://sith.huma-num.fr/vocable/840" class="i v840" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V840-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/840&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Bonne santé, bien-être &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">wḏȝ</a> <a href="http://sith.huma-num.fr/vocable/47" class="i v47" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V47-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/47&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; (Bonne) santé &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">snb</a> <a href="http://sith.huma-num.fr/vocable/1553" class="i v1553" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V1553-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/1553&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Ou bien &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme particule &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">rȝ-pw</a> jr <a href="http://sith.huma-num.fr/vocable/416" class="i v416" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V416-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/416&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; S’enfuir &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme verbe &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">wʿr</a> wʿ <a href="http://sith.huma-num.fr/vocable/415" class="i v415" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V415-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/415&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Hommes, gens &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">rmṯ</a> <a href="http://sith.huma-num.fr/vocable/1553" class="i v1553" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V1553-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/1553&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Ou bien &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme particule &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">rȝ-pw</a> <a href="http://sith.huma-num.fr/vocable/415" class="i v415" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V415-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/415&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Hommes, gens &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">rmṯ</a> 2 <a href="http://sith.huma-num.fr/vocable/204" class="i v204" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V204-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/204&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Particule énonciative&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme particule &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">jw</a> <a href="http://sith.huma-num.fr/vocable/524" class="i v524" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V524-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/524&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Particule négative&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme particule &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">bw</a> rḫ⸗tw⸗w</em><br /><br />
<div id="l25" style="padding-bottom:10px;"><img src="http://sith.huma-num.fr/KIUH/KIU32-25.png" /></div> <b>25</b> <em><span style="color:gray">[...]</span> mtw<span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V8-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne pluriel">&#11799;sn</span> <a href="http://sith.huma-num.fr/vocable/138" class="i v138" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V138-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/138&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Venir &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme verbe &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">jyt</a> <a href="http://sith.huma-num.fr/vocable/184" class="i v184" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V184-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/184&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Pour, vers &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">r</a> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/203" class="i v203" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V203-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/203&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Pays, terre &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">tȝ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/22" class="i l22" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L22-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/22&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Hatti &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Territoire, localit&eacute; ou ethnique du « nord » &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Ḫt</a> <a href="http://sith.huma-num.fr/vocable/184" class="i v184" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V184-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/184&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Pour, vers &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">r</a> <a href="http://sith.huma-num.fr/vocable/20" class="i v20" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V20-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/20&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Faire, agir &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme verbe &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">jrt</a> bȝkw n ky <a href="http://sith.huma-num.fr/vocable/1029" class="i v1029" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V1029-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/1029&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Particule négative&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme particule &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">bn</a> <a href="http://sith.huma-num.fr/vocable/204" class="i v204" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V204-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/204&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Particule énonciative&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme particule &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">jw</a>⸗tw <a href="http://sith.huma-num.fr/vocable/184" class="i v184" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V184-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/184&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Pour, vers &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">r</a> wȝḥ⸗w <a href="http://sith.huma-num.fr/vocable/168" class="i v168" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V168-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/168&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Dans, avec, comme, en tant que &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">m</a> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/203" class="i v203" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V203-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/203&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Pays, terre &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">tȝ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/22" class="i l22" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L22-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/22&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Hatti &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Territoire, localit&eacute; ou ethnique du « nord » &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Ḫt</a> <a href="http://sith.huma-num.fr/vocable/204" class="i v204" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V204-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/204&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Particule énonciative&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme particule &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">jw</a>⸗tw <a href="http://sith.huma-num.fr/vocable/184" class="i v184" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V184-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/184&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Pour, vers &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">r</a> jnt⸗w n <a href="http://sith.huma-num.fr/titulature/21" class="i t21" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/titulatures/T21-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/titulature/21&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Ramsès II &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Nom de fils de Rê  &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Rʿ-ms-sw-mry-Jmn</a> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/247" class="i v247" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V247-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/247&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Souverain, roi &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ḥqȝ</a> <a href="http://sith.huma-num.fr/vocable/156" class="i v156" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V156-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/156&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé pour qualifier un substantif (adjectif épithète) &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ʿȝ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/15" class="i l15" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L15-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/15&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Terre noire, Égypte &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Aire g&eacute;ographique &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Kmt</a> <a href="http://sith.huma-num.fr/vocable/1553" class="i v1553" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V1553-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/1553&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Ou bien &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme particule &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">rȝ-pw</a> <a href="http://sith.huma-num.fr/vocable/417" class="i v417" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V417-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/417&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Si &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme conjonction &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">jr</a> <a href="http://sith.huma-num.fr/vocable/416" class="i v416" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V416-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/416&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; S’enfuir &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme verbe &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">wʿr</a> wʿ <a href="http://sith.huma-num.fr/vocable/415" class="i v415" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V415-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/415&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Hommes, gens &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">rmṯ</a> <a href="http://sith.huma-num.fr/vocable/156" class="i v156" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V156-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/156&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé pour qualifier un substantif (adjectif épithète) &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ʿȝ</a> <a href="http://sith.huma-num.fr/vocable/168" class="i v168" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V168-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/168&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Dans, avec, comme, en tant que &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">m</a> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/203" class="i v203" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V203-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/203&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Pays, terre &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">tȝ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/22" class="i l22" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L22-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/22&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Hatti &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Territoire, localit&eacute; ou ethnique du « nord » &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Ḫt</a> mtw<span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V4-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne masculin singulier">&#11799;f</span> <span style="color:gray">[...]</span> <a href="http://sith.huma-num.fr/titulature/20" class="i t20" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/titulatures/T20-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/titulature/20&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Ramsès II &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Nom de couronnement  &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Wsr-Mȝʿt-Rʿ-stp~n-Rʿ</a> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/247" class="i v247" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V247-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/247&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Souverain, roi &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ḥqȝ</a> <a href="http://sith.huma-num.fr/vocable/156" class="i v156" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V156-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/156&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé pour qualifier un substantif (adjectif épithète) &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ʿȝ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/15" class="i l15" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L15-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/15&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Terre noire, Égypte &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Aire g&eacute;ographique &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Kmt</a> <a href="http://sith.huma-num.fr/vocable/1553" class="i v1553" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V1553-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/1553&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Ou bien &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme particule &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">rȝ-pw</a> wʿ <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/vocable/321" class="i v321" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V321-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/321&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Ville, localité, port &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">dmj</a> <a href="http://sith.huma-num.fr/vocable/1553" class="i v1553" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V1553-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/1553&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Ou bien &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme particule &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">rȝ-pw</a> wʿ qʿḥ <a href="http://sith.huma-num.fr/vocable/1553" class="i v1553" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V1553-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/1553&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Ou bien &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme particule &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">rȝ-pw</a></em><br /><br />
<div id="l26" style="padding-bottom:10px;"><img src="http://sith.huma-num.fr/KIUH/KIU32-26.png" /></div> <b>26</b> <em><span style="color:gray">[...]</span> nȝy <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/203" class="i v203" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V203-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/203&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Pays, terre &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">tȝ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/22" class="i l22" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L22-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/22&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Hatti &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Territoire, localit&eacute; ou ethnique du « nord » &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Ḫt</a> mtw⸗w <a href="http://sith.huma-num.fr/vocable/138" class="i v138" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V138-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/138&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Venir &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme verbe &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">jyt</a> <a href="http://sith.huma-num.fr/vocable/188" class="i v188" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V188-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/188&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; À, pour &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">n</a> <a href="http://sith.huma-num.fr/titulature/21" class="i t21" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/titulatures/T21-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/titulature/21&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Ramsès II &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Nom de fils de Rê  &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Rʿ-ms-sw-mry-Jmn</a> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/247" class="i v247" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V247-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/247&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Souverain, roi &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ḥqȝ</a> <a href="http://sith.huma-num.fr/vocable/156" class="i v156" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V156-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/156&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé pour qualifier un substantif (adjectif épithète) &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ʿȝ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/15" class="i l15" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L15-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/15&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Terre noire, Égypte &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Aire g&eacute;ographique &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Kmt</a> <a href="http://sith.huma-num.fr/vocable/1029" class="i v1029" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V1029-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/1029&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Particule négative&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme particule &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">bn</a> jr <a href="http://sith.huma-num.fr/titulature/20" class="i t20" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/titulatures/T20-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/titulature/20&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Ramsès II &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Nom de couronnement  &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Wsr-Mȝʿt-Rʿ-stp~n-Rʿ</a> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/247" class="i v247" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V247-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/247&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Souverain, roi &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ḥqȝ</a> <a href="http://sith.huma-num.fr/vocable/156" class="i v156" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V156-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/156&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé pour qualifier un substantif (adjectif épithète) &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ʿȝ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/15" class="i l15" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L15-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/15&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Terre noire, Égypte &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Aire g&eacute;ographique &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Kmt</a> <a href="http://sith.huma-num.fr/vocable/184" class="i v184" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V184-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/184&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Pour, vers &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">r</a> šsp⸗w jr <a href="http://sith.huma-num.fr/titulature/21" class="i t21" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/titulatures/T21-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/titulature/21&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Ramsès II &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Nom de fils de Rê  &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Rʿ-ms-sw-mry-Jmn</a> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/247" class="i v247" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V247-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/247&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Souverain, roi &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ḥqȝ</a> <a href="http://sith.huma-num.fr/vocable/156" class="i v156" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V156-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/156&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé pour qualifier un substantif (adjectif épithète) &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ʿȝ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/15" class="i l15" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L15-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/15&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Terre noire, Égypte &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Aire g&eacute;ographique &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Kmt</a> dt jn⸗tw⸗w <a href="http://sith.huma-num.fr/vocable/188" class="i v188" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V188-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/188&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; À, pour &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">n</a> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/85" class="i v85" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V85-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/85&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand, prince, notable &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">wr</a> <a href="http://sith.huma-num.fr/vocable/156" class="i v156" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V156-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/156&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé pour qualifier un substantif (adjectif épithète) &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ʿȝ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/22" class="i l22" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L22-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/22&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Hatti &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Territoire, localit&eacute; ou ethnique du « nord » &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Ḫt</a> <a href="http://sith.huma-num.fr/vocable/1029" class="i v1029" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V1029-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/1029&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Particule négative&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme particule &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">bn</a> <a href="http://sith.huma-num.fr/vocable/204" class="i v204" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V204-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/204&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Particule énonciative&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme particule &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">jw</a>⸗tw <a href="http://sith.huma-num.fr/vocable/184" class="i v184" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V184-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/184&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Pour, vers &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">r</a> wȝḥ⸗w <a href="http://sith.huma-num.fr/vocable/965" class="i v965" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V965-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/965&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Pareillement, de même &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme adverbe &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">m-mjtt</a> jr <a href="http://sith.huma-num.fr/vocable/416" class="i v416" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V416-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/416&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; S’enfuir &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme verbe &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">wʿr</a> wʿ <a href="http://sith.huma-num.fr/vocable/415" class="i v415" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V415-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/415&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Hommes, gens &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">rmṯ</a> <a href="http://sith.huma-num.fr/vocable/1553" class="i v1553" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V1553-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/1553&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Ou bien &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme particule &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">rȝ-pw</a> <a href="http://sith.huma-num.fr/vocable/415" class="i v415" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V415-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/415&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Hommes, gens &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">rmṯ</a> 2</em><br /><br />
<div id="l27" style="padding-bottom:10px;"><img src="http://sith.huma-num.fr/KIUH/KIU32-27.png" /></div> <b>27</b> <em><span style="color:gray">[...]</span> rḫ⸗tw⸗w mtw<span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V8-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne pluriel">&#11799;sn</span> <a href="http://sith.huma-num.fr/vocable/138" class="i v138" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V138-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/138&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Venir &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme verbe &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">jyt</a> r <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/203" class="i v203" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V203-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/203&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Pays, terre &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">tȝ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/15" class="i l15" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L15-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/15&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Terre noire, Égypte &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Aire g&eacute;ographique &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Kmt</a> r <a href="http://sith.huma-num.fr/vocable/20" class="i v20" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V20-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/20&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Faire, agir &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme verbe &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">jrt</a> bȝkw n ktḫw <a href="http://sith.huma-num.fr/vocable/1029" class="i v1029" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V1029-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/1029&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Particule négative&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme particule &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">bn</a> jr <a href="http://sith.huma-num.fr/titulature/20" class="i t20" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/titulatures/T20-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/titulature/20&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Ramsès II &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Nom de couronnement  &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Wsr-Mȝʿt-Rʿ-stp~n-Rʿ</a> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/247" class="i v247" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V247-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/247&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Souverain, roi &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ḥqȝ</a> <a href="http://sith.huma-num.fr/vocable/156" class="i v156" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V156-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/156&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé pour qualifier un substantif (adjectif épithète) &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ʿȝ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/15" class="i l15" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L15-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/15&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Terre noire, Égypte &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Aire g&eacute;ographique &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Kmt</a> wȝḥ⸗w <a href="http://sith.huma-num.fr/vocable/204" class="i v204" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V204-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/204&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Particule énonciative&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme particule &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">jw</a><span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V4-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne masculin singulier">&#11799;f</span> r dt jn⸗tw⸗w n <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/85" class="i v85" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V85-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/85&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand, prince, notable &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">wr</a> <a href="http://sith.huma-num.fr/vocable/156" class="i v156" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V156-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/156&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé pour qualifier un substantif (adjectif épithète) &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ʿȝ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/22" class="i l22" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L22-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/22&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Hatti &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Territoire, localit&eacute; ou ethnique du « nord » &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Ḫt</a> jr nȝ <a href="http://sith.huma-num.fr/vocable/1228" class="i v1228" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V1228-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/1228&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Parole, discours, affaire &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">mdwt</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/315" class="i v315" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V315-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/315&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Traité, contrat, rituel &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nyt-ʿ</a> jrrw <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/85" class="i v85" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V85-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/85&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand, prince, notable &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">wr</a> <a href="http://sith.huma-num.fr/vocable/156" class="i v156" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V156-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/156&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé pour qualifier un substantif (adjectif épithète) &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ʿȝ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/22" class="i l22" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L22-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/22&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Hatti &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Territoire, localit&eacute; ou ethnique du « nord » &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Ḫt</a> <a href="http://sith.huma-num.fr/vocable/318" class="i v318" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V318-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/318&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Avec, en compagnie de &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">jrm</a> [...] <span style="color:gray">[pȝ]</span> <a href="http://sith.huma-num.fr/vocable/247" class="i v247" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V247-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/247&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Souverain, roi &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ḥqȝ</a> <a href="http://sith.huma-num.fr/vocable/156" class="i v156" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V156-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/156&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé pour qualifier un substantif (adjectif épithète) &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ʿȝ</a></em><br /><br />
<div id="l28" style="padding-bottom:10px;"><img src="http://sith.huma-num.fr/KIUH/KIU32-28.png" /></div> <b>28</b> <em><span style="color:gray">[...]</span> <a href="http://sith.huma-num.fr/vocable/168" class="i v168" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V168-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/168&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Dans, avec, comme, en tant que &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">m</a> sšw <a href="http://sith.huma-num.fr/vocable/166" class="i v166" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V166-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/166&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Sur, concernant &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ḥr</a> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/319" class="i v319" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V319-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/319&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Tablette (pour l’écriture) &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ʿn</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/vocable/320" class="i v320" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V320-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/320&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Argent (métal) &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ḥḏ</a> jr nȝ <a href="http://sith.huma-num.fr/vocable/1228" class="i v1228" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V1228-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/1228&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Parole, discours, affaire &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">mdwt</a> <a href="http://sith.huma-num.fr/vocable/432" class="i v432" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V432-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/432&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Mille &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ḫȝ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/vocable/14" class="i v14" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V14-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/14&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Dieu, divinité &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nṯr</a> <a href="http://sith.huma-num.fr/vocable/168" class="i v168" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V168-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/168&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Dans, avec, comme, en tant que &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">m</a> <a href="http://sith.huma-num.fr/vocable/14" class="i v14" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V14-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/14&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Dieu, divinité &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nṯrw</a> ʿḥȝwty <a href="http://sith.huma-num.fr/vocable/168" class="i v168" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V168-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/168&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Dans, avec, comme, en tant que &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">m</a> <a href="http://sith.huma-num.fr/vocable/14" class="i v14" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V14-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/14&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Dieu, divinité &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nṯrw</a> ḥmwt <a href="http://sith.huma-num.fr/vocable/168" class="i v168" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V168-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/168&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Dans, avec, comme, en tant que &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">m</a> nȝ <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/203" class="i v203" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V203-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/203&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Pays, terre &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">tȝ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/22" class="i l22" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L22-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/22&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Hatti &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Territoire, localit&eacute; ou ethnique du « nord » &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Ḫt</a> m-d <a href="http://sith.huma-num.fr/vocable/432" class="i v432" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V432-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/432&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Mille &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ḫȝ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/vocable/14" class="i v14" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V14-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/14&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Dieu, divinité &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nṯr</a> <a href="http://sith.huma-num.fr/vocable/168" class="i v168" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V168-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/168&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Dans, avec, comme, en tant que &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">m</a> <a href="http://sith.huma-num.fr/vocable/14" class="i v14" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V14-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/14&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Dieu, divinité &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nṯrw</a> ʿḥȝwty <a href="http://sith.huma-num.fr/vocable/168" class="i v168" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V168-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/168&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Dans, avec, comme, en tant que &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">m</a> <a href="http://sith.huma-num.fr/vocable/14" class="i v14" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V14-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/14&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Dieu, divinité &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nṯrw</a> ḥmwt <a href="http://sith.huma-num.fr/vocable/168" class="i v168" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V168-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/168&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Dans, avec, comme, en tant que &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">m</a> nȝy <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/203" class="i v203" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V203-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/203&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Pays, terre &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">tȝ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/15" class="i l15" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L15-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/15&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Terre noire, Égypte &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Aire g&eacute;ographique &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Kmt</a> st m-d<span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V1-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 1<sup>re</sup> personne singulier">&#11799;j</span> mtr⸗w <span style="color:gray">[...]</span> nȝy <a href="http://sith.huma-num.fr/vocable/1228" class="i v1228" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V1228-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/1228&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Parole, discours, affaire &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">mdwt</a> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/theonyme/9" class="i d9" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/theonymes/D9-G1.png&quot; height=&quot;35px;&quot; /&gt;  &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/theonyme/9&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Rê &lt;/span&gt;&lt;/a&gt; &lt;br /&gt; Th&eacute;onyme ou d&eacute;signation divine &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt; ">Rʿ</a> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/56" class="i v56" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V56-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/56&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Seigneur, maître, responsable &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nb</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/vocable/203" class="i v203" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V203-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/203&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Pays, terre &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">tȝ</a> <a href="http://sith.huma-num.fr/vocable/90" class="i v90" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V90-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/90&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Ciel &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pt</a> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/theonyme/9" class="i d9" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/theonymes/D9-G1.png&quot; height=&quot;35px;&quot; /&gt;  &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/theonyme/9&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Rê &lt;/span&gt;&lt;/a&gt; &lt;br /&gt; Th&eacute;onyme ou d&eacute;signation divine &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt; ">Rʿ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/vocable/321" class="i v321" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V321-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/321&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Ville, localité, port &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">dmj</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/57" class="i l57" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L57-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/57&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Irenen &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Toponyme, ethnique ou lieu de culte &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Jrnn</a></em><br /><br />
<div id="l29" style="padding-bottom:10px;"><img src="http://sith.huma-num.fr/KIUH/KIU32-29.png" /></div> <b>29</b> <em><a href="http://sith.huma-num.fr/theonyme/25" class="i d25" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/theonymes/D25-G1.png&quot; height=&quot;35px;&quot; /&gt;  &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/theonyme/25&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Seth &lt;/span&gt;&lt;/a&gt; &lt;br /&gt; Th&eacute;onyme ou d&eacute;signation divine &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt; ">Stẖ</a> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/56" class="i v56" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V56-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/56&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Seigneur, maître, responsable &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nb</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/vocable/203" class="i v203" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V203-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/203&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Pays, terre &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">tȝ</a> <a href="http://sith.huma-num.fr/vocable/90" class="i v90" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V90-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/90&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Ciel &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pt</a> <a href="http://sith.huma-num.fr/theonyme/25" class="i d25" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/theonymes/D25-G1.png&quot; height=&quot;35px;&quot; /&gt;  &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/theonyme/25&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Seth &lt;/span&gt;&lt;/a&gt; &lt;br /&gt; Th&eacute;onyme ou d&eacute;signation divine &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt; ">Stẖ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/22" class="i l22" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L22-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/22&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Hatti &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Territoire, localit&eacute; ou ethnique du « nord » &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Ḫt</a> <a href="http://sith.huma-num.fr/theonyme/25" class="i d25" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/theonymes/D25-G1.png&quot; height=&quot;35px;&quot; /&gt;  &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/theonyme/25&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Seth &lt;/span&gt;&lt;/a&gt; &lt;br /&gt; Th&eacute;onyme ou d&eacute;signation divine &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt; ">Stẖ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/vocable/321" class="i v321" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V321-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/321&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Ville, localité, port &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">dmj</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/57" class="i l57" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L57-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/57&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Irenen &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Toponyme, ethnique ou lieu de culte &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Jrnn</a> <a href="http://sith.huma-num.fr/theonyme/25" class="i d25" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/theonymes/D25-G1.png&quot; height=&quot;35px;&quot; /&gt;  &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/theonyme/25&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Seth &lt;/span&gt;&lt;/a&gt; &lt;br /&gt; Th&eacute;onyme ou d&eacute;signation divine &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt; ">Stẖ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/vocable/321" class="i v321" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V321-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/321&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Ville, localité, port &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">dmj</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/58" class="i l58" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L58-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/58&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Djepirened &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Territoire, localit&eacute; ou ethnique du « nord » &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Ḏpjrnd</a> <a href="http://sith.huma-num.fr/theonyme/25" class="i d25" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/theonymes/D25-G1.png&quot; height=&quot;35px;&quot; /&gt;  &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/theonyme/25&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Seth &lt;/span&gt;&lt;/a&gt; &lt;br /&gt; Th&eacute;onyme ou d&eacute;signation divine &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt; ">Stẖ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/vocable/321" class="i v321" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V321-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/321&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Ville, localité, port &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">dmj</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> Pyrq <a href="http://sith.huma-num.fr/theonyme/25" class="i d25" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/theonymes/D25-G1.png&quot; height=&quot;35px;&quot; /&gt;  &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/theonyme/25&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Seth &lt;/span&gt;&lt;/a&gt; &lt;br /&gt; Th&eacute;onyme ou d&eacute;signation divine &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt; ">Stẖ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/vocable/321" class="i v321" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V321-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/321&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Ville, localité, port &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">dmj</a> Ḫssp <a href="http://sith.huma-num.fr/theonyme/25" class="i d25" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/theonymes/D25-G1.png&quot; height=&quot;35px;&quot; /&gt;  &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/theonyme/25&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Seth &lt;/span&gt;&lt;/a&gt; &lt;br /&gt; Th&eacute;onyme ou d&eacute;signation divine &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt; ">Stẖ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/vocable/321" class="i v321" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V321-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/321&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Ville, localité, port &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">dmj</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/56" class="i l56" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L56-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/56&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Seres &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Territoire, localit&eacute; ou ethnique du « nord » &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Srs</a> <a href="http://sith.huma-num.fr/theonyme/25" class="i d25" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/theonymes/D25-G1.png&quot; height=&quot;35px;&quot; /&gt;  &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/theonyme/25&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Seth &lt;/span&gt;&lt;/a&gt; &lt;br /&gt; Th&eacute;onyme ou d&eacute;signation divine &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt; ">Stẖ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/vocable/321" class="i v321" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V321-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/321&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Ville, localité, port &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">dmj</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/55" class="i l55" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L55-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/55&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Kherep &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Toponyme, ethnique ou lieu de culte &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Ḫrp</a> <a href="http://sith.huma-num.fr/theonyme/25" class="i d25" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/theonymes/D25-G1.png&quot; height=&quot;35px;&quot; /&gt;  &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/theonyme/25&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Seth &lt;/span&gt;&lt;/a&gt; &lt;br /&gt; Th&eacute;onyme ou d&eacute;signation divine &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt; ">Stẖ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/vocable/321" class="i v321" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V321-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/321&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Ville, localité, port &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">dmj</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/54" class="i l54" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L54-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/54&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Rekhesen &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Territoire, localit&eacute; ou ethnique du « nord » &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Rḫsn</a> <a href="http://sith.huma-num.fr/theonyme/25" class="i d25" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/theonymes/D25-G1.png&quot; height=&quot;35px;&quot; /&gt;  &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/theonyme/25&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Seth &lt;/span&gt;&lt;/a&gt; &lt;br /&gt; Th&eacute;onyme ou d&eacute;signation divine &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt; ">Stẖ</a></em><br /><br />
<div id="l30" style="padding-bottom:10px;"><img src="http://sith.huma-num.fr/KIUH/KIU32-30.png" /></div> <b>30</b> <em>[...] ... [...] <a href="http://sith.huma-num.fr/theonyme/25" class="i d25" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/theonymes/D25-G1.png&quot; height=&quot;35px;&quot; /&gt;  &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/theonyme/25&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Seth &lt;/span&gt;&lt;/a&gt; &lt;br /&gt; Th&eacute;onyme ou d&eacute;signation divine &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt; ">Stẖ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/vocable/321" class="i v321" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V321-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/321&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Ville, localité, port &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">dmj</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &llt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/59" class="i l59" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L59-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/59&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Sekhepen &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Territoire, localit&eacute; ou ethnique du « nord » &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Sḫpn</a> ʿnṯrt <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nyt</a> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/203" class="i v203" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V203-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/203&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Pays, terre &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">tȝ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/22" class="i l22" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L22-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/22&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Hatti &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Territoire, localit&eacute; ou ethnique du « nord » &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Ḫt</a> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/14" class="i v14" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V14-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/14&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Dieu, divinité &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nṯr</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> Ḏytṯḫrr <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/14" class="i v14" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V14-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/14&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Dieu, divinité &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nṯr</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> Krḏy<span style="color:gray">[...]</span> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/14" class="i v14" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V14-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/14&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Dieu, divinité &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nṯr</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> Ḫrpntrs</em><br /><br />
<div id="l31" style="padding-bottom:10px;"><img src="http://sith.huma-num.fr/KIUH/KIU32-31.png" /></div> <b>31</b> <em>tȝ <a href="http://sith.huma-num.fr/vocable/359" class="i v359" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V359-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/359&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Déesse &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nṯrt</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nyt</a> <a href="http://sith.huma-num.fr/vocable/321" class="i v321" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V321-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/321&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Ville, localité, port &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">dmj</a> Krḫn tȝ <a href="http://sith.huma-num.fr/vocable/359" class="i v359" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V359-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/359&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Déesse &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nṯrt</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nyt</a> Ḏr tȝ <a href="http://sith.huma-num.fr/vocable/359" class="i v359" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V359-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/359&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Déesse &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nṯrt</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nyt</a> <a href="http://sith.huma-num.fr/toponyme/61" class="i l61" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L61-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/61&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Nenou, Ninive &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Territoire, localit&eacute; ou ethnique du « nord » &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Nnw</a> tȝ <a href="http://sith.huma-num.fr/vocable/359" class="i v359" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V359-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/359&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Déesse &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nṯrt</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nyt</a> Ḏn<span style="color:gray">[...]</span> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/14" class="i v14" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V14-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/14&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Dieu, divinité &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nṯr</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> Nynt <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/14" class="i v14" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V14-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/14&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Dieu, divinité &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nṯr</a> <span style="color:gray">[...]</span>rt <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/14" class="i v14" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V14-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/14&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Dieu, divinité &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nṯr</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> Ḫbt tȝ ḥmt <a href="http://sith.huma-num.fr/vocable/10" class="i v10" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V10-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/10&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Roi &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nswt</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nyt</a> tȝ <a href="http://sith.huma-num.fr/vocable/90" class="i v90" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V90-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/90&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Ciel &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pt</a> <a href="http://sith.huma-num.fr/vocable/414" class="i v414" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V414-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/414&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article pluriel&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nȝ</a> <a href="http://sith.huma-num.fr/vocable/14" class="i v14" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V14-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/14&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Dieu, divinité &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nṯrw</a> <a href="http://sith.huma-num.fr/vocable/150" class="i v150" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V150-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/150&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Tout, chaque &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé pour qualifier un substantif (adjectif épithète) &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nbw</a> ʿnḫ tȝ <a href="http://sith.huma-num.fr/vocable/359" class="i v359" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V359-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/359&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Déesse &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nṯrt</a> tȝ <a href="http://sith.huma-num.fr/vocable/244" class="i v244" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V244-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/244&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Souveraine, maîtresse &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ḥnwt</a> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> jwtn tȝ <a href="http://sith.huma-num.fr/vocable/244" class="i v244" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V244-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/244&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Souveraine, maîtresse &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ḥnwt</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nyt</a> ʿnḫ Tsḫr tȝ <a href="http://sith.huma-num.fr/vocable/244" class="i v244" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V244-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/244&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Souveraine, maîtresse &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ḥnwt</a></em><br /><br />
<div id="l32" style="padding-bottom:10px;"><img src="http://sith.huma-num.fr/KIUH/KIU32-32.png" /></div> <b>32</b> <em>ḏw <a href="http://sith.huma-num.fr/vocable/414" class="i v414" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V414-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/414&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article pluriel&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nȝ</a> <a href="http://sith.huma-num.fr/vocable/386" class="i v386" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V386-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/386&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Fleuve &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">jtrw</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/203" class="i v203" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V203-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/203&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Pays, terre &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">tȝ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/22" class="i l22" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L22-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/22&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Hatti &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Territoire, localit&eacute; ou ethnique du « nord » &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Ḫt</a> <a href="http://sith.huma-num.fr/vocable/414" class="i v414" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V414-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/414&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article pluriel&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nȝ</a> <a href="http://sith.huma-num.fr/vocable/14" class="i v14" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V14-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/14&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Dieu, divinité &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nṯrw</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nyw</a> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/203" class="i v203" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V203-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/203&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Pays, terre &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">tȝ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/60" class="i l60" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L60-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/60&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Qedjouden &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Territoire, localit&eacute; ou ethnique du « nord » &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Qḏwdn</a> <a href="http://sith.huma-num.fr/theonyme/1" class="i d1" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/theonymes/D1-G1.png&quot; height=&quot;35px;&quot; /&gt;  &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/theonyme/1&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Amon &lt;/span&gt;&lt;/a&gt; &lt;br /&gt; Th&eacute;onyme ou d&eacute;signation divine &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt; ">Jmn</a> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/theonyme/9" class="i d9" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/theonymes/D9-G1.png&quot; height=&quot;35px;&quot; /&gt;  &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/theonyme/9&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Rê &lt;/span&gt;&lt;/a&gt; &lt;br /&gt; Th&eacute;onyme ou d&eacute;signation divine &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt; ">Rʿ</a> <a href="http://sith.huma-num.fr/theonyme/25" class="i d25" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/theonymes/D25-G1.png&quot; height=&quot;35px;&quot; /&gt;  &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/theonyme/25&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Seth &lt;/span&gt;&lt;/a&gt; &lt;br /&gt; Th&eacute;onyme ou d&eacute;signation divine &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt; ">Stẖ</a> <a href="http://sith.huma-num.fr/vocable/414" class="i v414" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V414-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/414&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article pluriel&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nȝ</a> <a href="http://sith.huma-num.fr/vocable/14" class="i v14" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V14-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/14&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Dieu, divinité &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nṯrw</a> ʿḥȝwty <a href="http://sith.huma-num.fr/vocable/414" class="i v414" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V414-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/414&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article pluriel&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nȝ</a> <a href="http://sith.huma-num.fr/vocable/14" class="i v14" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V14-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/14&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Dieu, divinité &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nṯrw</a> <a href="http://sith.huma-num.fr/vocable/414" class="i v414" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V414-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/414&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article pluriel&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nȝ</a> ḏw <a href="http://sith.huma-num.fr/vocable/414" class="i v414" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V414-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/414&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article pluriel&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nȝ</a> <a href="http://sith.huma-num.fr/vocable/386" class="i v386" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V386-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/386&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Fleuve &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">jtrw</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/203" class="i v203" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V203-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/203&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Pays, terre &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">tȝ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/15" class="i l15" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L15-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/15&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Terre noire, Égypte &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Aire g&eacute;ographique &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Kmt</a> tȝ <a href="http://sith.huma-num.fr/vocable/90" class="i v90" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V90-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/90&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Ciel &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pt</a> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> jwtn <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/1109" class="i v1109" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V1109-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/1109&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Mer &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ym</a> <a href="http://sith.huma-num.fr/vocable/156" class="i v156" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V156-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/156&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé pour qualifier un substantif (adjectif épithète) &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ʿȝ</a> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> ṯȝw <a href="http://sith.huma-num.fr/vocable/414" class="i v414" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V414-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/414&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article pluriel&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nȝ</a> šnʿw jr nȝ <a href="http://sith.huma-num.fr/vocable/1228" class="i v1228" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V1228-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/1228&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Parole, discours, affaire &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">mdwt</a></em><br /><br />
<div id="l33" style="padding-bottom:10px;"><img src="http://sith.huma-num.fr/KIUH/KIU32-33.png" /></div> <b>33</b> <em>nty <a href="http://sith.huma-num.fr/vocable/166" class="i v166" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V166-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/166&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Sur, concernant &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ḥr</a> pȝ <a href="http://sith.huma-num.fr/vocable/319" class="i v319" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V319-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/319&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Tablette (pour l’écriture) &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ʿn</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/vocable/320" class="i v320" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V320-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/320&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Argent (métal) &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ḥḏ</a> <a href="http://sith.huma-num.fr/vocable/188" class="i v188" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V188-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/188&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; À, pour &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">n</a> pȝ <a href="http://sith.huma-num.fr/vocable/203" class="i v203" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V203-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/203&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Pays, terre &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">tȝ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/22" class="i l22" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L22-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/22&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Hatti &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Territoire, localit&eacute; ou ethnique du « nord » &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Ḫt</a> <a href="http://sith.huma-num.fr/vocable/188" class="i v188" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V188-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/188&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; À, pour &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">n</a> pȝ <a href="http://sith.huma-num.fr/vocable/203" class="i v203" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V203-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/203&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Pays, terre &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">tȝ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/15" class="i l15" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L15-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/15&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Terre noire, Égypte &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Aire g&eacute;ographique &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Kmt</a> jr pȝ nty <a href="http://sith.huma-num.fr/vocable/1029" class="i v1029" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V1029-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/1029&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Particule négative&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme particule &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">bn</a> jw<span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V4-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne masculin singulier">&#11799;f</span> <a href="http://sith.huma-num.fr/vocable/184" class="i v184" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V184-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/184&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Pour, vers &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">r</a> sȝw<span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V8-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne pluriel">&#11799;sn</span> jr <a href="http://sith.huma-num.fr/vocable/432" class="i v432" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V432-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/432&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Mille &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ḫȝ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/vocable/14" class="i v14" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V14-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/14&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Dieu, divinité &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nṯr</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> pȝ <a href="http://sith.huma-num.fr/vocable/203" class="i v203" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V203-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/203&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Pays, terre &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">tȝ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/22" class="i l22" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L22-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/22&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Hatti &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Territoire, localit&eacute; ou ethnique du « nord » &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Ḫt</a> m-d <a href="http://sith.huma-num.fr/vocable/432" class="i v432" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V432-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/432&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Mille &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ḫȝ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/vocable/14" class="i v14" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V14-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/14&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Dieu, divinité &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nṯr</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> pȝ <a href="http://sith.huma-num.fr/vocable/203" class="i v203" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V203-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/203&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Pays, terre &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">tȝ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/15" class="i l15" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L15-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/15&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Terre noire, Égypte &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Aire g&eacute;ographique &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Kmt</a> <a href="http://sith.huma-num.fr/vocable/184" class="i v184" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V184-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/184&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Pour, vers &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">r</a> fḫ <a href="http://sith.huma-num.fr/vocable/317" class="i v317" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V317-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/317&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Article possessif masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝy</a><span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V4-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne masculin singulier">&#11799;f</span> <a href="http://sith.huma-num.fr/vocable/62" class="i v62" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V62-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/62&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Maison, domaine, temple &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pr</a> <a href="http://sith.huma-num.fr/vocable/317" class="i v317" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V317-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/317&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Article possessif masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝy</a><span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V4-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne masculin singulier">&#11799;f</span> <a href="http://sith.huma-num.fr/vocable/203" class="i v203" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V203-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/203&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Pays, terre &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">tȝ</a> nȝy<span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V4-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne masculin singulier">&#11799;f</span> bȝkw ḫr jr pȝ nty jw<span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V4-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne masculin singulier">&#11799;f</span> <a href="http://sith.huma-num.fr/vocable/184" class="i v184" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V184-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/184&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Pour, vers &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">r</a> sȝwty nȝ <a href="http://sith.huma-num.fr/vocable/1228" class="i v1228" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V1228-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/1228&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Parole, discours, affaire &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">mdwt</a> nty <a href="http://sith.huma-num.fr/vocable/166" class="i v166" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V166-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/166&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Sur, concernant &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ḥr</a> pȝ <a href="http://sith.huma-num.fr/vocable/319" class="i v319" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V319-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/319&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Tablette (pour l’écriture) &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ʿn</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/vocable/320" class="i v320" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V320-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/320&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Argent (métal) &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ḥḏ</a> jw⸗w <a href="http://sith.huma-num.fr/vocable/168" class="i v168" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V168-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/168&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Dans, avec, comme, en tant que &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">m</a> <a href="http://sith.huma-num.fr/toponyme/22" class="i l22" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L22-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/22&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Hatti &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Territoire, localit&eacute; ou ethnique du « nord » &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Ḫt</a> jw⸗w <a href="http://sith.huma-num.fr/vocable/168" class="i v168" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V168-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/168&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Dans, avec, comme, en tant que &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">m</a> <a href="http://sith.huma-num.fr/vocable/415" class="i v415" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V415-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/415&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Hommes, gens &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">rmṯ</a></em><br /><br />
<div id="l34" style="padding-bottom:10px;"><img src="http://sith.huma-num.fr/KIUH/KIU32-34.png" /></div> <b>34</b> <em><a href="http://sith.huma-num.fr/toponyme/15" class="i l15" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L15-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/15&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Terre noire, Égypte &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Aire g&eacute;ographique &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Kmt</a> mtw⸗w tm jr<span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V4-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne masculin singulier">&#11799;f</span> ḫmḫm r<span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V8-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne pluriel">&#11799;sn</span> jr <a href="http://sith.huma-num.fr/vocable/432" class="i v432" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V432-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/432&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Mille &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ḫȝ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/vocable/14" class="i v14" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V14-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/14&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Dieu, divinité &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nṯr</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/203" class="i v203" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V203-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/203&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Pays, terre &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">tȝ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/22" class="i l22" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L22-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/22&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Hatti &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Territoire, localit&eacute; ou ethnique du « nord » &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Ḫt</a> m-d <a href="http://sith.huma-num.fr/vocable/432" class="i v432" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V432-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/432&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Mille &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ḫȝ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/vocable/14" class="i v14" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V14-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/14&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Dieu, divinité &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nṯr</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/203" class="i v203" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V203-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/203&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Pays, terre &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">tȝ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/15" class="i l15" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L15-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/15&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Terre noire, Égypte &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Aire g&eacute;ographique &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Kmt</a> r dt snb<span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V4-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne masculin singulier">&#11799;f</span> r dt <a href="http://sith.huma-num.fr/vocable/9" class="i v9" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V9-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/9&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Vivre &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme verbe &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ʿnḫ</a><span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V4-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne masculin singulier">&#11799;f</span> <a href="http://sith.huma-num.fr/vocable/318" class="i v318" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V318-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/318&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Avec, en compagnie de &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">jrm</a> <a href="http://sith.huma-num.fr/vocable/1019" class="i v1019" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V1019-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/1019&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Article possessif pluriel&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nȝy</a><span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V4-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne masculin singulier">&#11799;f</span> pryt <a href="http://sith.huma-num.fr/vocable/318" class="i v318" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V318-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/318&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Avec, en compagnie de &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">jrm</a> <a href="http://sith.huma-num.fr/vocable/317" class="i v317" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V317-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/317&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Article possessif masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝy</a><span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V4-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne masculin singulier">&#11799;f</span> <a href="http://sith.huma-num.fr/vocable/203" class="i v203" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V203-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/203&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Pays, terre &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">tȝ</a> <a href="http://sith.huma-num.fr/vocable/318" class="i v318" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V318-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/318&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Avec, en compagnie de &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">jrm</a> <a href="http://sith.huma-num.fr/vocable/1019" class="i v1019" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V1019-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/1019&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Article possessif pluriel&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nȝy</a><span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V4-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne masculin singulier">&#11799;f</span> bȝkw jr <a href="http://sith.huma-num.fr/vocable/416" class="i v416" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V416-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/416&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; S’enfuir &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme verbe &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">wʿr</a> wʿ <a href="http://sith.huma-num.fr/vocable/415" class="i v415" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V415-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/415&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Hommes, gens &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">rmṯ</a> <a href="http://sith.huma-num.fr/vocable/168" class="i v168" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V168-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/168&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Dans, avec, comme, en tant que &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">m</a> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/203" class="i v203" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V203-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/203&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Pays, terre &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">tȝ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/15" class="i l15" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L15-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/15&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Terre noire, Égypte &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Aire g&eacute;ographique &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Kmt</a> <a href="http://sith.huma-num.fr/vocable/1553" class="i v1553" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V1553-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/1553&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Ou bien &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme particule &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">rȝ-pw</a> 2 <a href="http://sith.huma-num.fr/vocable/1553" class="i v1553" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V1553-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/1553&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Ou bien &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme particule &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">rȝ-pw</a> 3</em><br /><br />
<div id="l35" style="padding-bottom:10px;"><img src="http://sith.huma-num.fr/KIUH/KIU32-35.png" /></div> <b>35</b> <em>mtw<span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V8-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne pluriel">&#11799;sn</span> <a href="http://sith.huma-num.fr/vocable/138" class="i v138" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V138-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/138&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Venir &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme verbe &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">jyt</a> <a href="http://sith.huma-num.fr/vocable/188" class="i v188" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V188-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/188&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; À, pour &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">n</a> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/85" class="i v85" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V85-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/85&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand, prince, notable &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">wr</a> <a href="http://sith.huma-num.fr/vocable/156" class="i v156" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V156-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/156&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé pour qualifier un substantif (adjectif épithète) &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ʿȝ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/22" class="i l22" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L22-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/22&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Hatti &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Territoire, localit&eacute; ou ethnique du « nord » &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Ḫt</a> jr <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/85" class="i v85" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V85-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/85&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand, prince, notable &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">wr</a> <a href="http://sith.huma-num.fr/vocable/156" class="i v156" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V156-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/156&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé pour qualifier un substantif (adjectif épithète) &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ʿȝ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/22" class="i l22" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L22-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/22&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Hatti &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Territoire, localit&eacute; ou ethnique du « nord » &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Ḫt</a> mḥ jm<span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V8-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne pluriel">&#11799;sn</span> mtw<span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V4-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne masculin singulier">&#11799;f</span> dt jn⸗tw⸗w ʿn <a href="http://sith.huma-num.fr/vocable/188" class="i v188" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V188-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/188&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; À, pour &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">n</a> <a href="http://sith.huma-num.fr/titulature/20" class="i t20" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/titulatures/T20-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/titulature/20&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Ramsès II &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Nom de couronnement  &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Wsr-Mȝʿt-Rʿ-stp~n-Rʿ</a> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/247" class="i v247" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V247-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/247&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Souverain, roi &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ḥqȝ</a> <a href="http://sith.huma-num.fr/vocable/156" class="i v156" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V156-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/156&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé pour qualifier un substantif (adjectif épithète) &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ʿȝ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/15" class="i l15" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L15-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/15&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Terre noire, Égypte &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Aire g&eacute;ographique &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Kmt</a> ḫr jr <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/415" class="i v415" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V415-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/415&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Hommes, gens &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">rmṯ</a> nty <a href="http://sith.huma-num.fr/vocable/204" class="i v204" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V204-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/204&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Particule énonciative&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme particule &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">jw</a>⸗tw <a href="http://sith.huma-num.fr/vocable/184" class="i v184" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V184-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/184&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Pour, vers &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">r</a> jnt<span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V4-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne masculin singulier">&#11799;f</span> <a href="http://sith.huma-num.fr/vocable/188" class="i v188" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V188-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/188&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; À, pour &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">n</a> <a href="http://sith.huma-num.fr/titulature/21" class="i t21" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/titulatures/T21-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/titulature/21&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Ramsès II &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Nom de fils de Rê  &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Rʿ-ms-sw-mry-Jmn</a> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/247" class="i v247" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V247-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/247&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Souverain, roi &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ḥqȝ</a> <a href="http://sith.huma-num.fr/vocable/156" class="i v156" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V156-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/156&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé pour qualifier un substantif (adjectif épithète) &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ʿȝ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/15" class="i l15" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L15-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/15&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Terre noire, Égypte &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Aire g&eacute;ographique &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Kmt</a> m dy jry⸗tw sʿḥʿ <a href="http://sith.huma-num.fr/vocable/317" class="i v317" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V317-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/317&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Article possessif masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝy</a><span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V4-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne masculin singulier">&#11799;f</span> <a href="http://sith.huma-num.fr/vocable/1018" class="i v1018" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V1018-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/1018&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Faute, crime &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">btȝ</a> <a href="http://sith.huma-num.fr/vocable/184" class="i v184" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V184-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/184&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Pour, vers &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">r</a><span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V4-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne masculin singulier">&#11799;f</span> m dy</em><br /><br />
<div id="l36" style="padding-bottom:10px;"><img src="http://sith.huma-num.fr/KIUH/KIU32-36.png" /></div> <b>36</b> <em>fḫ⸗tw <a href="http://sith.huma-num.fr/vocable/317" class="i v317" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V317-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/317&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Article possessif masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝy</a><span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V4-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne masculin singulier">&#11799;f</span> <a href="http://sith.huma-num.fr/vocable/62" class="i v62" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V62-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/62&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Maison, domaine, temple &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pr</a> <a href="http://sith.huma-num.fr/vocable/1019" class="i v1019" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V1019-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/1019&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Article possessif pluriel&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nȝy</a><span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V4-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne masculin singulier">&#11799;f</span> <a href="http://sith.huma-num.fr/vocable/825" class="i v825" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V825-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/825&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Femme, épouse &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ḥmwt</a> <a href="http://sith.huma-num.fr/vocable/1019" class="i v1019" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V1019-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/1019&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Article possessif pluriel&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nȝy</a><span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V4-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne masculin singulier">&#11799;f</span> <a href="http://sith.huma-num.fr/vocable/413" class="i v413" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V413-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/413&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Enfant &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ẖrdw</a> <span style="color:gray">[...]</span>⸗tw<span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V4-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne masculin singulier">&#11799;f</span> m dy th⸗tw <a href="http://sith.huma-num.fr/vocable/184" class="i v184" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V184-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/184&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Pour, vers &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">r</a> jrty<span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V4-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne masculin singulier">&#11799;f</span> <a href="http://sith.huma-num.fr/vocable/184" class="i v184" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V184-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/184&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Pour, vers &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">r</a> msḏrwy<span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V4-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne masculin singulier">&#11799;f</span> <a href="http://sith.huma-num.fr/vocable/184" class="i v184" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V184-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/184&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Pour, vers &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">r</a> rȝ<span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V4-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne masculin singulier">&#11799;f</span> <a href="http://sith.huma-num.fr/vocable/184" class="i v184" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V184-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/184&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Pour, vers &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">r</a> <a href="http://sith.huma-num.fr/vocable/98" class="i v98" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V98-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/98&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Pied, jambe &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">rdwy</a><span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V4-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne masculin singulier">&#11799;f</span> m dy jry⸗tw <span style="color:gray">[...]</span> <a href="http://sith.huma-num.fr/vocable/150" class="i v150" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V150-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/150&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Tout, chaque &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé pour qualifier un substantif (adjectif épithète) &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nb</a> <a href="http://sith.huma-num.fr/vocable/184" class="i v184" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V184-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/184&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Pour, vers &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">r</a><span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V4-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne masculin singulier">&#11799;f</span> <a href="http://sith.huma-num.fr/vocable/965" class="i v965" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V965-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/965&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Pareillement, de même &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme adverbe &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">m-mjtt</a> jr <a href="http://sith.huma-num.fr/vocable/416" class="i v416" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V416-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/416&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; S’enfuir &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme verbe &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">wʿr</a> <a href="http://sith.huma-num.fr/vocable/415" class="i v415" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V415-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/415&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Hommes, gens &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">rmṯ</a> m <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/203" class="i v203" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V203-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/203&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Pays, terre &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">tȝ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/22" class="i l22" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L22-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/22&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Hatti &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Territoire, localit&eacute; ou ethnique du « nord » &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Ḫt</a> jw<span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V4-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne masculin singulier">&#11799;f</span> m wʿ jw<span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V4-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne masculin singulier">&#11799;f</span> m 2 jw<span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V4-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne masculin singulier">&#11799;f</span> m 3 mtw⸗w <a href="http://sith.huma-num.fr/vocable/138" class="i v138" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V138-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/138&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Venir &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme verbe &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">jyt</a> <a href="http://sith.huma-num.fr/vocable/188" class="i v188" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V188-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/188&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; À, pour &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">n</a> <a href="http://sith.huma-num.fr/titulature/20" class="i t20" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/titulatures/T20-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/titulature/20&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Ramsès II &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Nom de couronnement  &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Wsr-Mȝʿt-Rʿ-stp~n-Rʿ</a></em><br /><br />
<div id="l37" style="padding-bottom:10px;"><img src="http://sith.huma-num.fr/KIUH/KIU32-37.png" /></div> <b>37</b> <em><a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/247" class="i v247" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V247-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/247&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Souverain, roi &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ḥqȝ</a> <a href="http://sith.huma-num.fr/vocable/156" class="i v156" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V156-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/156&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé pour qualifier un substantif (adjectif épithète) &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ʿȝ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/15" class="i l15" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L15-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/15&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Terre noire, Égypte &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Aire g&eacute;ographique &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Kmt</a> jmy mḥ <a href="http://sith.huma-num.fr/titulature/21" class="i t21" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/titulatures/T21-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/titulature/21&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Ramsès II &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Nom de fils de Rê  &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Rʿ-ms-sw-mry-Jmn</a> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/247" class="i v247" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V247-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/247&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Souverain, roi &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ḥqȝ</a> <span style="color:gray">[...]</span> jn⸗tw⸗w <a href="http://sith.huma-num.fr/vocable/188" class="i v188" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V188-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/188&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; À, pour &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">n</a> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/85" class="i v85" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V85-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/85&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand, prince, notable &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">wr</a> <a href="http://sith.huma-num.fr/vocable/156" class="i v156" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V156-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/156&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé pour qualifier un substantif (adjectif épithète) &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ʿȝ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/22" class="i l22" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L22-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/22&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Hatti &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Territoire, localit&eacute; ou ethnique du « nord » &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Ḫt</a> mtw tm <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/85" class="i v85" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V85-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/85&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand, prince, notable &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">wr</a> <a href="http://sith.huma-num.fr/vocable/156" class="i v156" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V156-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/156&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé pour qualifier un substantif (adjectif épithète) &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ʿȝ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/22" class="i l22" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L22-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/22&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Hatti &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Territoire, localit&eacute; ou ethnique du « nord » &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Ḫt</a> <span style="color:gray">[...]</span> <a href="http://sith.huma-num.fr/vocable/1018" class="i v1018" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V1018-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/1018&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Faute, crime &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">btȝ</a> <a href="http://sith.huma-num.fr/vocable/184" class="i v184" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V184-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/184&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Pour, vers &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">r</a><span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V8-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne pluriel">&#11799;sn</span> mtw⸗tw tm fḫ <a href="http://sith.huma-num.fr/vocable/317" class="i v317" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V317-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/317&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Article possessif masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝy</a><span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V4-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne masculin singulier">&#11799;f</span> <a href="http://sith.huma-num.fr/vocable/1019" class="i v1019" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V1019-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/1019&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Article possessif pluriel&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nȝy</a><span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V4-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne masculin singulier">&#11799;f</span> <a href="http://sith.huma-num.fr/vocable/825" class="i v825" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V825-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/825&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Femme, épouse &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ḥmwt</a> <a href="http://sith.huma-num.fr/vocable/1019" class="i v1019" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V1019-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/1019&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Article possessif pluriel&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nȝy</a><span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V4-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne masculin singulier">&#11799;f</span> <a href="http://sith.huma-num.fr/vocable/413" class="i v413" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V413-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/413&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Enfant &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ẖrdw</a> mtw⸗tw tm ẖdb⸗tw<span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V4-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne masculin singulier">&#11799;f</span> mtw⸗tw tm th <a href="http://sith.huma-num.fr/vocable/184" class="i v184" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V184-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/184&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Pour, vers &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">r</a> msḏrwy<span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V4-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne masculin singulier">&#11799;f</span></em><br /><br />
<div id="l38" style="padding-bottom:10px;"><img src="http://sith.huma-num.fr/KIUH/KIU32-38.png" /></div> <b>38</b> <em>r jrty<span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V4-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne masculin singulier">&#11799;f</span> <a href="http://sith.huma-num.fr/vocable/184" class="i v184" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V184-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/184&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Pour, vers &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">r</a> rȝ<span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V4-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne masculin singulier">&#11799;f</span> <a href="http://sith.huma-num.fr/vocable/184" class="i v184" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V184-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/184&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Pour, vers &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">r</a> <a href="http://sith.huma-num.fr/vocable/98" class="i v98" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V98-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/98&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Pied, jambe &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">rdwy</a><span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V4-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne masculin singulier">&#11799;f</span> mtw⸗tw tm sʿḥʿ <a href="http://sith.huma-num.fr/vocable/1018" class="i v1018" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V1018-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/1018&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Faute, crime &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">btȝ</a> <a href="http://sith.huma-num.fr/vocable/150" class="i v150" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V150-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/150&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Tout, chaque &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé pour qualifier un substantif (adjectif épithète) &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nb</a> <a href="http://sith.huma-num.fr/vocable/184" class="i v184" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V184-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/184&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Pour, vers &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">r</a><span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V4-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne masculin singulier">&#11799;f</span> nty m ḥry-jb <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/319" class="i v319" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V319-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/319&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Tablette (pour l’écriture) &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ʿn</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/vocable/320" class="i v320" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V320-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/320&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Argent (métal) &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ḥḏ</a> ḥr <a href="http://sith.huma-num.fr/vocable/1226" class="i v1226" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V1226-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/1226&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Article possessif féminin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">tȝy</a><span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V4-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne masculin singulier">&#11799;f</span> <a href="http://sith.huma-num.fr/vocable/1703" class="i v1703" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V1703-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/1703&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Côté, bord &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">rjt</a> ḥȝwt <a href="http://sith.huma-num.fr/vocable/1632" class="i v1632" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V1632-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/1632&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Incrustations, gravures &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ẖpw</a> m <a href="http://sith.huma-num.fr/vocable/451" class="i v451" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V451-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/451&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Statue, image &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">twt</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/theonyme/25" class="i d25" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/theonymes/D25-G1.png&quot; height=&quot;35px;&quot; /&gt;  &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/theonyme/25&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Seth &lt;/span&gt;&lt;/a&gt; &lt;br /&gt; Th&eacute;onyme ou d&eacute;signation divine &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt; ">Stẖ</a> ḥr <span style="color:gray">[...]</span> <a href="http://sith.huma-num.fr/vocable/188" class="i v188" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V188-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/188&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; À, pour &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">n</a> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/85" class="i v85" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V85-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/85&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand, prince, notable &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">wr</a> <a href="http://sith.huma-num.fr/vocable/156" class="i v156" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V156-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/156&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé pour qualifier un substantif (adjectif épithète) &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ʿȝ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/22" class="i l22" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L22-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/22&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Hatti &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Territoire, localit&eacute; ou ethnique du « nord » &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Ḫt</a> <a href="http://sith.huma-num.fr/vocable/865" class="i v865" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V865-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/865&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Entourer, ceindre &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme verbe &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">jnḥw</a> m <a href="http://sith.huma-num.fr/vocable/1423" class="i v1423" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V1423-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/1423&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Contour &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">smdt</a> mdw <span style="color:gray">[...]</span> m ḏd <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> ḫtm <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/theonyme/25" class="i d25" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/theonymes/D25-G1.png&quot; height=&quot;35px;&quot; /&gt;  &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/theonyme/25&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Seth &lt;/span&gt;&lt;/a&gt; &lt;br /&gt; Th&eacute;onyme ou d&eacute;signation divine &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt; ">Stẖ</a> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/247" class="i v247" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V247-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/247&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Souverain, roi &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ḥqȝ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> tȝ <a href="http://sith.huma-num.fr/vocable/90" class="i v90" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V90-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/90&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Ciel &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pt</a> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> ḫtm <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/315" class="i v315" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V315-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/315&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Traité, contrat, rituel &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nyt-ʿ</a> jrrw Ḫtsr <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/85" class="i v85" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V85-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/85&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand, prince, notable &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">wr</a> <a href="http://sith.huma-num.fr/vocable/156" class="i v156" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V156-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/156&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé pour qualifier un substantif (adjectif épithète) &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ʿȝ</a></em><br /><br />
<div id="l39" style="padding-bottom:10px;"><img src="http://sith.huma-num.fr/KIUH/KIU32-39.png" /></div> <b>39</b> <em><a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/22" class="i l22" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L22-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/22&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Hatti &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Territoire, localit&eacute; ou ethnique du « nord » &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Ḫt</a> <a href="http://sith.huma-num.fr/vocable/522" class="i v522" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V522-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/522&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Puissant, vaillant &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ṯnr</a> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/325" class="i v325" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V325-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/325&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Enfant &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">šrj</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> Mrsr <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/85" class="i v85" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V85-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/85&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand, prince, notable &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">wr</a> <a href="http://sith.huma-num.fr/vocable/156" class="i v156" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V156-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/156&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grand &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé pour qualifier un substantif (adjectif épithète) &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ʿȝ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/22" class="i l22" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L22-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/22&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Hatti &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Territoire, localit&eacute; ou ethnique du « nord » &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Ḫt</a> <a href="http://sith.huma-num.fr/vocable/522" class="i v522" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V522-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/522&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Puissant, vaillant &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ṯnr</a> m-ẖnw <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> jnḥw <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/1632" class="i v1632" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V1632-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/1632&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Incrustations, gravures &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ẖpw</a> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> ḫtm [...] ... [...] <a href="http://sith.huma-num.fr/vocable/1226" class="i v1226" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V1226-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/1226&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Article possessif féminin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">tȝy</a><span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V4-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne masculin singulier">&#11799;f</span> ky <a href="http://sith.huma-num.fr/vocable/1703" class="i v1703" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V1703-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/1703&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Côté, bord &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">rjt</a> <a href="http://sith.huma-num.fr/vocable/1632" class="i v1632" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V1632-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/1632&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Incrustations, gravures &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ẖpw</a> <a href="http://sith.huma-num.fr/vocable/1704" class="i v1704" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V1704-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/1704&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Statue féminine, représentation féminine &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">rpyt</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nyt</a> <a href="http://sith.huma-num.fr/vocable/1227" class="i v1227" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V1227-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/1227&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini féminin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">tȝ</a> <a href="http://sith.huma-num.fr/vocable/359" class="i v359" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V359-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/359&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Déesse &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nṯrt</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nyt</a> <a href="http://sith.huma-num.fr/toponyme/22" class="i l22" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L22-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/22&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Hatti &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Territoire, localit&eacute; ou ethnique du « nord » &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Ḫt</a> ḥr qn <a href="http://sith.huma-num.fr/vocable/1704" class="i v1704" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V1704-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/1704&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Statue féminine, représentation féminine &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">rpyt</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nyt</a> <a href="http://sith.huma-num.fr/vocable/423" class="i v423" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V423-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/423&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grande &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">wrt</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nyt</a> <a href="http://sith.huma-num.fr/toponyme/22" class="i l22" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L22-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/22&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Hatti &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Territoire, localit&eacute; ou ethnique du « nord » &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Ḫt</a> <a href="http://sith.huma-num.fr/vocable/865" class="i v865" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V865-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/865&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Entourer, ceindre &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme verbe &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">jnḥw</a> m <a href="http://sith.huma-num.fr/vocable/1423" class="i v1423" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V1423-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/1423&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Contour &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">smdt</a> mdw m ḏd <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> ḫtm <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a></em><br /><br />
<div id="l40" style="padding-bottom:10px;"><img src="http://sith.huma-num.fr/KIUH/KIU32-40.png" /></div> <b>40</b> <em><a href="http://sith.huma-num.fr/theonyme/9" class="i d9" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/theonymes/D9-G1.png&quot; height=&quot;35px;&quot; /&gt;  &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/theonyme/9&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Rê &lt;/span&gt;&lt;/a&gt; &lt;br /&gt; Th&eacute;onyme ou d&eacute;signation divine &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt; ">Rʿ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/vocable/321" class="i v321" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V321-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/321&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Ville, localité, port &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">dmj</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/57" class="i l57" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L57-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/57&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Irenen &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Toponyme, ethnique ou lieu de culte &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Jrnn</a> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/56" class="i v56" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V56-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/56&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Seigneur, maître, responsable &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nb</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> tȝ <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> ḫtm <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> Ptḫp tȝ <a href="http://sith.huma-num.fr/vocable/423" class="i v423" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V423-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/423&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Grande &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">wrt</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nyt</a> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> tȝ <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/22" class="i l22" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L22-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/22&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Hatti &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Territoire, localit&eacute; ou ethnique du « nord » &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Ḫt</a> tȝ <a href="http://sith.huma-num.fr/vocable/1682" class="i v1682" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V1682-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/1682&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Petite, enfant &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">šrjt</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nyt</a> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> tȝ <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/60" class="i l60" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L60-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/60&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Qedjouden &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Territoire, localit&eacute; ou ethnique du « nord » &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Qḏwdn</a> <span style="color:gray">[...]</span> <a href="http://sith.huma-num.fr/toponyme/57" class="i l57" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L57-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/57&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Irenen &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Toponyme, ethnique ou lieu de culte &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Jrnn</a> tȝ <a href="http://sith.huma-num.fr/vocable/244" class="i v244" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V244-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/244&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Souveraine, maîtresse &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ḥnwt</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nyt</a> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> tȝ bȝkt tȝ <a href="http://sith.huma-num.fr/vocable/359" class="i v359" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V359-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/359&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Déesse &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nṯrt</a> m-ẖnw <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> jnḥw <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/1632" class="i v1632" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V1632-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/1632&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Incrustations, gravures &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ẖpw</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> ḫtm <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/theonyme/9" class="i d9" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/theonymes/D9-G1.png&quot; height=&quot;35px;&quot; /&gt;  &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/theonyme/9&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Rê &lt;/span&gt;&lt;/a&gt; &lt;br /&gt; Th&eacute;onyme ou d&eacute;signation divine &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt; ">Rʿ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/toponyme/57" class="i l57" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L57-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/57&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Irenen &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Toponyme, ethnique ou lieu de culte &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Jrnn</a> <a href="http://sith.huma-num.fr/vocable/316" class="i v316" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V316-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/316&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Pronom démonstratif, article défini masculin singulier&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; pronom, article &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pȝ</a> <a href="http://sith.huma-num.fr/vocable/56" class="i v56" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V56-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/56&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Seigneur, maître, responsable &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nb</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> tȝ <a href="http://sith.huma-num.fr/vocable/150" class="i v150" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V150-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/150&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Tout, chaque &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé pour qualifier un substantif (adjectif épithète) &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nb</a></em><br /><br />

<br /><b>Colonne de gauche</b><br />
<div id="l41" style="padding-bottom:10px;"><img src="http://sith.huma-num.fr/KIUH/KIU32-41.png" /></div> <b>41</b> <em>[...] ... [...] <span style="color:gray">[nb]</span> <a href="http://sith.huma-num.fr/toponyme/1" class="i l1" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L1-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/1&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Double Pays, Égypte &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Aire g&eacute;ographique &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Tȝwy</a> <a href="http://sith.huma-num.fr/vocable/56" class="i v56" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V56-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/56&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Seigneur, maître, responsable &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nb</a> <a href="http://sith.huma-num.fr/vocable/20" class="i v20" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V20-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/20&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Faire, agir &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme verbe &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">jrt</a> <a href="http://sith.huma-num.fr/vocable/82" class="i v82" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V82-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/82&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Biens, possessions, rites &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ḫt</a> <a href="http://sith.huma-num.fr/titulature/20" class="i t20" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/titulatures/T20-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/titulature/20&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Ramsès II &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Nom de couronnement  &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Wsr-Mȝʿt-Rʿ-stp~n-Rʿ</a> <a href="http://sith.huma-num.fr/vocable/12" class="i v12" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V12-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/12&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Fils &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">sȝ</a> <a href="http://sith.huma-num.fr/theonyme/9" class="i d9" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/theonymes/D9-G1.png&quot; height=&quot;35px;&quot; /&gt;  &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/theonyme/9&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Rê &lt;/span&gt;&lt;/a&gt; &lt;br /&gt; Th&eacute;onyme ou d&eacute;signation divine &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt; ">Rʿ</a> <a href="http://sith.huma-num.fr/vocable/173" class="i v173" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V173-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/173&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; De &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme préposition &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ny</a> <a href="http://sith.huma-num.fr/vocable/120" class="i v120" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V120-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/120&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Corps, ventre &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ẖt</a><span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V4-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne masculin singulier">&#11799;f</span> <a href="http://sith.huma-num.fr/vocable/79" class="i v79" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V79-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/79&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Aimé de &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">mry</a><span class="i" title=" &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V4-G1.png&quot; height=&quot;35px;&quot; /&gt; Pronom suffixe &lt;br /&gt; 3<sup>e</sup> personne masculin singulier">&#11799;f</span> <a href="http://sith.huma-num.fr/vocable/56" class="i v56" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V56-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/56&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Seigneur, maître, responsable &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nb</a> <a href="http://sith.huma-num.fr/vocable/78" class="i v78" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V78-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/78&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Couronne, apparition &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ḫʿw</a> <a href="http://sith.huma-num.fr/titulature/21" class="i t21" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/titulatures/T21-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/titulature/21&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Ramsès II &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Nom de fils de Rê  &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Rʿ-ms-sw-mry-Jmn</a> <a href="http://sith.huma-num.fr/vocable/79" class="i v79" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V79-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/79&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Aimé de &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">mry</a> <a href="http://sith.huma-num.fr/theonyme/10" class="i d10" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/theonymes/D10-G1.png&quot; height=&quot;35px;&quot; /&gt;  &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/theonyme/10&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Amon-Rê &lt;/span&gt;&lt;/a&gt; &lt;br /&gt; Th&eacute;onyme ou d&eacute;signation divine &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt; ">Jmn-Rʿ</a> <a href="http://sith.huma-num.fr/vocable/56" class="i v56" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V56-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/56&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Seigneur, maître, responsable &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nb</a> <a href="http://sith.huma-num.fr/vocable/13" class="i v13" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V13-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/13&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Trône, siège, support &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nswt</a> <a href="http://sith.huma-num.fr/toponyme/1" class="i l1" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L1-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/1&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Double Pays, Égypte &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Aire g&eacute;ographique &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Tȝwy</a> <a href="http://sith.huma-num.fr/vocable/30" class="i v30" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V30-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/30&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Donner, offrir, accorder &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme verbe &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">dw</a> <a href="http://sith.huma-num.fr/vocable/44" class="i v44" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V44-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/44&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Vie &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ʿnḫ</a></em><br /><br />

<br /><b>Colonne de droite</b><br />
<div id="l42" style="padding-bottom:10px;"><img src="http://sith.huma-num.fr/KIUH/KIU32-42.png" /></div> <b>42</b> <em>[...] ... [...] <a href="http://sith.huma-num.fr/titulature/81" class="i t81" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/titulatures/T81-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/titulature/81&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Ramsès II &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Nom d’Horus  &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Kȝ-nḫt-mry-Mȝʿt</a> <a href="http://sith.huma-num.fr/vocable/10" class="i v10" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V10-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/10&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Roi &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nswt</a> <a href="http://sith.huma-num.fr/vocable/11" class="i v11" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V11-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/11&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Roi, roi de Basse-Égypte &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">bjty</a> <a href="http://sith.huma-num.fr/vocable/56" class="i v56" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V56-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/56&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Seigneur, maître, responsable &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nb</a> <a href="http://sith.huma-num.fr/toponyme/1" class="i l1" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/toponymes/L1-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/toponyme/1&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Double Pays, Égypte &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Aire g&eacute;ographique &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Tȝwy</a> <a href="http://sith.huma-num.fr/vocable/56" class="i v56" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V56-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/56&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Seigneur, maître, responsable &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nb</a> <a href="http://sith.huma-num.fr/vocable/20" class="i v20" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V20-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/20&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Faire, agir &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme verbe &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">jrt</a> <a href="http://sith.huma-num.fr/vocable/82" class="i v82" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V82-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/82&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Biens, possessions, rites &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ḫt</a> <a href="http://sith.huma-num.fr/titulature/20" class="i t20" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/titulatures/T20-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/titulature/20&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Ramsès II &lt;/span&gt; &lt;/a&gt; &lt;br /&gt; Nom de couronnement  &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">Wsr-Mȝʿt-Rʿ-stp~n-Rʿ</a> [...] ... [...] <a href="http://sith.huma-num.fr/vocable/79" class="i v79" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V79-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/79&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Aimé de &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">mry</a> <a href="http://sith.huma-num.fr/theonyme/10" class="i d10" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/theonymes/D10-G1.png&quot; height=&quot;35px;&quot; /&gt;  &lt;/div&gt;
 &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt; &lt;a href=&quot;http://sith.huma-num.fr/theonyme/10&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; Amon-Rê &lt;/span&gt;&lt;/a&gt; &lt;br /&gt; Th&eacute;onyme ou d&eacute;signation divine &lt;/div&gt;&lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt; ">Jmn-Rʿ</a> <a href="http://sith.huma-num.fr/vocable/10" class="i v10" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V10-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/10&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Roi &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nswt</a> <a href="http://sith.huma-num.fr/vocable/14" class="i v14" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V14-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/14&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Dieu, divinité &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nṯrw</a> <a href="http://sith.huma-num.fr/vocable/56" class="i v56" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V56-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/56&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Seigneur, maître, responsable &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">nb</a> <a href="http://sith.huma-num.fr/vocable/90" class="i v90" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V90-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/90&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Ciel &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">pt</a> <a href="http://sith.huma-num.fr/vocable/30" class="i v30" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V30-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/30&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Donner, offrir, accorder &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme verbe &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">dw</a> <a href="http://sith.huma-num.fr/vocable/44" class="i v44" title=" &lt;div style=&quot;float:left;&quot;&gt; &lt;img src=&quot;http://sith.huma-num.fr/api/vocables/V44-G1.png&quot; height=&quot;35px;&quot; /&gt; &lt;/div&gt; &lt;div style=&quot;float:left;padding-left:20px;line-height:20px;&quot; &gt;  &lt;a href=&quot;http://sith.huma-num.fr/vocable/44&quot;&gt; &lt;span style=&quot;color:#06e;&quot;&gt; &laquo; Vie &raquo;&lt;/a&gt;&lt;/span&gt; &lt;br /&gt; Employé comme substantif &lt;/div&gt; &lt;div style=&quot;clear:both;&quot;&gt;&lt;/div&gt;">ʿnḫ</a></em><br /><br /><hr /><a name="bibliographie"></a> 

In [ ]:
## Works in Python 3.10 or higher

import csv # Create csv files
import html
import re
import requests

from bs4 import BeautifulSoup, Tag, NavigableString


# =========================
# Configuration
# =========================

#INPUT_HTML_FILE = "input.html"
INPUT_URL = "http://sith.huma-num.fr/karnak/32" # exchange for URL/Id number of text of interest from "Projet Karnak" (U Montpellier III)
METADATA_CSV_FILE = "metadata.csv"


# =========================
# Regular expressions
# =========================

# RegEx for lemma type (different lists for general vocabulary and types of proper nouns as defined by the project)
LEMMA_HREF_PATTERN = re.compile(r"/(vocable|theonyme|toponyme|titulature)/(\d+)/?$", re.IGNORECASE)
# RegEx for translation (in French quotation marks) of the lemmata
TRANSLATION_PATTERN = re.compile(r"«\s*(.*?)\s*»")
# RegEx for finding numbers in the transliteration
NUMERAL_RE = re.compile(r"\d+")


# =========================
# General helper functions
# =========================

def fetch_html(url: str) -> str:
    response = requests.get(url) # optional: timeout=30
    response.raise_for_status()
    response.encoding = "utf-8"
    return response.text


def normalize_whitespace(text: str) -> str:
    """Collapse whitespace and strip ends."""
    return re.sub(r"\s+", " ", text).strip()


def filename_by_id(url: str) -> str:
    """
    Extract last number from URL and build filename:
    http://.../karnak/32  -> KIU32.csv
    """
    match = re.search(r"/(\d+)/?$", url)
    if not match:
        raise ValueError("No numeric ID found in URL")

    number = match.group(1)
    return f"KIU{number}" # string for output file name


# =========================
# Text parsing helpers
# =========================

def extract_transcription(node: Tag) -> str:
    """Get clean visible text of a token."""
    return normalize_whitespace(node.get_text(" ", strip=True))


# For PoS labels that are preceded by "Employé comme" ("used as")
def extract_pos(title_unescaped: str) -> str | None:
    """
    Extract grammatical category from SITH title text.
    Examples:
    - Employé comme substantif -> substantif
    - Employé pour qualifier un substantif (adjectif épithète) -> adjectif épithète
    """
    patterns = [
        r"Employé comme\s+([^<]+)",
        r"Employé pour qualifier un substantif\s*\(([^)]+)\)",
    ]

    for pattern in patterns:
        match = re.search(pattern, title_unescaped, flags=re.IGNORECASE)
        if match:
            return normalize_whitespace(match.group(1))

    return None


# Other PoS labels
def extract_pos_fallback(title_unescaped: str) -> str | None:
    """
    Fallback PoS extraction for cases where the title does not use
    'Employé comme ...' or 'Employé pour ...' but simply gives a category line such as
    'pronom, article'.
    """
    title_text = re.sub(r"<[^>]+>", "\n", title_unescaped)
    lines = [normalize_whitespace(line) for line in title_text.split("\n")]
    lines = [line for line in lines if line]

    # Last non-empty line is often the PoS line in SITH titles
    if lines:
        candidate = lines[-1]
        # Avoid returning the same text as the lemma translation line
        if not candidate.lower().startswith(("pronom suffixe",)):
            return candidate

    return None


# Get all information on the respective lemma
def parse_lemma_info(anchor: Tag) -> tuple[int | None, str | None, str | None, str | None]:
    """
    Extract:
    - LemmaID from href
    - LemmaType from href (vocable, theonyme, toponyme, titulature)
    - LemmaTranslation from title
    - PoS from title, if available
    """
    href = anchor.get("href", "")
    title_raw = anchor.get("title", "")

    lemma_id = None
    lemma_type = None
    lemma_translation = None
    pos = None

    href_match = LEMMA_HREF_PATTERN.search(href)
    if href_match:
        lemma_type = href_match.group(1).lower()
        lemma_id = int(href_match.group(2))

    if not title_raw:
        return lemma_id, lemma_type, lemma_translation, pos

    title_unescaped = html.unescape(title_raw)
    title_soup = BeautifulSoup(title_unescaped, "html.parser")

    if lemma_type == "vocable":
        # 1. Preferred pattern: translation in guillemets
        translation_match = TRANSLATION_PATTERN.search(title_unescaped)
        if translation_match:
            lemma_translation = normalize_whitespace(translation_match.group(1)).lower()

        # 2. Fallback: take the blue span text
        # for all lemma translations that are not in French "guillmets" (quotation marks)
        if not lemma_translation:
            span = title_soup.find("span", style=lambda s: s and "#06e" in s)
            if span:
                lemma_translation = normalize_whitespace(span.get_text(" ", strip=True)).lower()

        # 3. Preferred PoS extraction
        # PoS for clear cases containing "Employé comme"
        pos = extract_pos(title_unescaped)

        # 4. Fallback PoS extraction
        if not pos:
            pos = extract_pos_fallback(title_unescaped)

    elif lemma_type in {"theonyme", "toponyme", "titulature"}:
        span = title_soup.find("span", style=lambda s: s and "#06e" in s)
        if span:
            lemma_translation = normalize_whitespace(span.get_text(" ", strip=True))

    return lemma_id, lemma_type, lemma_translation, pos


# Identify destroyed and partially damaged tokens
def is_gray_reconstruction(node: Tag) -> bool:
    """Return True for spans like <span style='color:gray'>[...]</span>."""
    if node.name != "span":
        return False
    style = (node.get("style") or "").lower()
    text = normalize_whitespace(node.get_text(" ", strip=True))
    return "gray" in style and text.startswith("[")


# Check if an element is a "suffix pronoun" (PoS)
def is_suffix_pronoun(node: Tag) -> bool:
    """
    SITH encodes suffix pronouns as:
    <span class="i" title="...Pronom suffixe...">f</span>
    """
    if node.name != "span":
        return False
    title = html.unescape(node.get("title", ""))
    return "Pronom suffixe" in title # return True if "Pronom suffixe" in attribute "title"


# Get morphological analysis for suffix pronouns (e.g. 3rd p. fem. sg.) and use it instead of a translation ("she")
def parse_suffix_info(span: Tag) -> tuple[str | None, str | None]:
    """
    Extract info from suffix pronoun span.
    Returns:
    - lemma_translation, e.g. '3e personne masculin singulier'
    - pos, e.g. 'pronom suffixe'
    """
    title_unescaped = html.unescape(span.get("title", ""))

    pos = "pronom suffixe" if re.search(r"Pronom suffixe", title_unescaped, re.IGNORECASE) else None

    # Remove HTML tags from the title content
    title_text = re.sub(r"<[^>]+>", " ", title_unescaped)
    title_text = re.sub(r"\s+", " ", title_text).strip()

    # Take everything after "Pronom suffixe"
    match = re.search(r"Pronom suffixe\s*(.+)$", title_text, re.IGNORECASE)
    lemma_translation = match.group(1).strip() if match else None
    if lemma_translation:
        lemma_translation = re.sub(r"(\d)\s+([a-zA-Zéèêû]+)", r"\1\2", lemma_translation) # no space after ordinal number (1ere, 2e etc.)

    return lemma_translation, pos


# Dealing with tokens that are not lemmatized, e.g. numerals, lacunae, unknown words
def add_unannotated_tokens(
    text: str,
    rows: list[dict[str, object]],
    text_id: str,
    line_no: int,
    word_counter: int,
    hiero_url: str,
) -> int:
    """
    Add tokens from plain text outside annotated <a> elements.

    Ensures that the output reflects the full linear token stream,
    including items without lemma annotation (e.g. numerals, lacunae, unlemmatized words).
    """
    
    parts = re.findall(r"\S+", text)

    for part in parts:
        word_counter += 1

        # Processing of numbers in the transcription
        if NUMERAL_RE.fullmatch(part):
            rows.append(
                {
                    "TextID": text_id,
                    "Line/Column": line_no,
                    "Word": word_counter,
                    "Transcription": part,
                    "Hieroglyphs": hiero_url,
                    "LemmaID": None,
                    "LemmaType": "nombre",
                    "LemmaTranslation": part,   # alternativ: "1"
                    "PoS": "numéral",
                }
            )
        # Processing of lacunae in the transcription    
        else:
            rows.append(
                {
                    "TextID": text_id,
                    "Line/Column": line_no,
                    "Word": word_counter,
                    "Transcription": part,
                    "Hieroglyphs": hiero_url,
                    "LemmaID": None,
                    "LemmaType": None,
                    "LemmaTranslation": None,
                    "PoS": None,
                }
            )

    return word_counter



# =========================
# Main extraction
# =========================

def parse_words(soup: BeautifulSoup, url: str) -> list[dict[str, object]]:
    """
    Parse inscription lines and return row dicts for words.csv.

    Rules:
    - TextID comes from the page URL, e.g. .../32 -> KIU32
    - <a> tokens are treated as lemmatized tokens when possible
    - suffix pronouns stay separate tokens
    - gray reconstructions are kept as transcription
    - gray reconstructions can merge with immediately following plain text,
      e.g. <span style="color:gray">[...]</span>tsb -> [...]tsb
    - plain text outside tags is split on whitespace into separate tokens
    """
    text_id = filename_by_id(url)
    rows: list[dict[str, object]] = []

    # All inscription line divs look like <div id="l1">...</div>, <div id="l2">...</div>, etc.
    line_divs = soup.find_all("div", id=re.compile(r"^l\d+$"))

    # Processing each line ("div")
    for line_div in line_divs:
        line_id = line_div.get("id", "")
        line_no = int(line_id[1:])  # "l1" -> 1

        # Getting the link to image of the hieroglyphs of the line
        img = line_div.find("img")
        hiero_url = img.get("src", "").strip() if img else ""

        word_counter = 0
        current = line_div.next_sibling

        while current is not None:
            # isinstance() is a built-in Python function used to distinguish between
            # different node types returned by BeautifulSoup. The parsed HTML consists
            # of Tag objects (e.g. <a>, <span>) and NavigableString objects (plain text).
            # We need this check because these types require different handling:
            # Tag nodes may carry linguistic annotation, while NavigableString nodes
            # represent unannotated text (e.g. numbers or plain words).
            if isinstance(current, Tag):
                # Stop at next line div
                if current.name == "div" and re.fullmatch(r"l\d+", current.get("id", "")):
                    break

                # Stop at hard section boundary
                if current.name == "hr":
                    break

                # Extract tokens from <em> blocks
                # <em> encloses the text that is rendered in italics -> Egyptological transcription
                if current.name == "em":
                    children = list(current.contents)
                    i = 0

                    while i < len(children):
                        node = children[i]

                        # 1. Lemmatized tokens in <a> tag
                        if isinstance(node, Tag) and node.name == "a":
                            word_counter += 1
                            transcription = extract_transcription(node)
                            lemma_id, lemma_type, lemma_translation, pos = parse_lemma_info(node)

                            rows.append(
                                {
                                    "TextID": text_id,
                                    "Line/Column": line_no,
                                    "Word": word_counter,
                                    "Transcription": transcription,
                                    "Hieroglyphs": hiero_url,
                                    "LemmaID": lemma_id,
                                    "LemmaType": lemma_type,
                                    "LemmaTranslation": lemma_translation,
                                    "PoS": pos,
                                }
                            )

                        # 2. Treat suffix pronouns as separate tokens
                        elif isinstance(node, Tag) and is_suffix_pronoun(node):
                            word_counter += 1
                            transcription = extract_transcription(node)
                            lemma_translation, pos = parse_suffix_info(node)

                            rows.append(
                                {
                                    "TextID": text_id,
                                    "Line/Column": line_no,
                                    "Word": word_counter,
                                    "Transcription": transcription,
                                    "Hieroglyphs": hiero_url,
                                    "LemmaID": None,
                                    "LemmaType": None,
                                    "LemmaTranslation": lemma_translation,
                                    "PoS": pos,
                                }
                            )

                        # 3. Combine partial destructions (rendered in gray) with preserved parts of words
                        # e.g. [...]tsb encoded as <span style="color:gray">[...]</span>tsb <a> 
                        elif isinstance(node, Tag) and is_gray_reconstruction(node):
                            transcription = extract_transcription(node)

                            if i + 1 < len(children) and isinstance(children[i + 1], NavigableString):
                                next_text = str(children[i + 1])
                                match = re.match(r"(\S+)(.*)", next_text, flags=re.DOTALL)
                                if match:
                                    transcription += match.group(1)
                                    children[i + 1] = NavigableString(match.group(2))

                            word_counter += 1
                            rows.append(
                                {
                                    "TextID": text_id,
                                    "Line/Column": line_no,
                                    "Word": word_counter,
                                    "Transcription": transcription,
                                    "Hieroglyphs": hiero_url,
                                    "LemmaID": None,
                                    "LemmaType": None,
                                    "LemmaTranslation": None,
                                    "PoS": None,
                                }
                            )

                        # 4. Text not framed by tags and not lemmatized, z. B. tȝ-nyt-ḥtr mtw
                        elif isinstance(node, NavigableString):
                            text = str(node)
                            word_counter = add_unannotated_tokens(
                                text=text,
                                rows=rows,
                                text_id=text_id,
                                line_no=line_no,
                                word_counter=word_counter,
                                hiero_url=hiero_url,
                            )

                        i += 1

            current = current.next_sibling

    return rows


# =========================
# Metadata extraction
# =========================

def extract_editors(full_text: str) -> str:
    """
    Extract editors from the bottom part:
    - Auteur(s) de la notice : Edwin Dalino.
    - Avec des contributions de Elena Panaite, Sébastien Biston-Moulin
    Returns one comma-separated string.
    """
    editors: list[str] = []

    notice_match = re.search(
        r"Auteur\(s\) de la notice\s*:\s*(.*?)<br\s*/?>",
        full_text,
        flags=re.IGNORECASE | re.DOTALL,
    )
    if notice_match:
        fragment = notice_match.group(1)
        names = re.findall(r'>([^<>]+)</a>', fragment)
        editors.extend([normalize_whitespace(n) for n in names if normalize_whitespace(n)])

    contrib_match = re.search(
        r"Avec des contributions de\s*(.*?)<br\s*/?>",
        full_text,
        flags=re.IGNORECASE | re.DOTALL,
    )
    if contrib_match:
        fragment = contrib_match.group(1)
        names = re.findall(r'>([^<>]+)</a>', fragment)
        editors.extend([normalize_whitespace(n) for n in names if normalize_whitespace(n)])

    # Deduplicate while preserving order
    seen = set()
    deduped = []
    for name in editors:
        if name not in seen:
            seen.add(name)
            deduped.append(name)

    return ", ".join(deduped)


def extract_file_dates(full_text: str) -> tuple[str | None, str | None]:
    created_match = re.search(
        r"Création de la fiche\s*:\s*([0-9]{2}/[0-9]{2}/[0-9]{4})",
        full_text,
        flags=re.IGNORECASE,
    )
    modified_match = re.search(
        r"Dernière modification\s*:\s*([0-9]{2}/[0-9]{2}/[0-9]{4})",
        full_text,
        flags=re.IGNORECASE,
    )

    created = created_match.group(1) if created_match else None
    modified = modified_match.group(1) if modified_match else None
    return created, modified


def parse_metadata(raw_html: str, url: str) -> dict[str, str | None]:
    text_id = filename_by_id(url)

    editors = extract_editors(raw_html)
    created, modified = extract_file_dates(raw_html)

    return {
        "ID": text_id,
        "Editors": editors,
        "FileCreated": created,
        "FileLastModified": modified,
    }


# =========================
# CSV output
# =========================

def write_words_csv(rows: list[dict[str, object]], output_file: str) -> None:
    fieldnames = [
        "TextID",
        "Line/Column",
        "Word",
        "Transcription",
        "Hieroglyphs",
        "LemmaID",
        "LemmaType",
        "LemmaTranslation",
        "PoS",
    ]

    with open(output_file, "w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def write_metadata_csv(row: dict[str,str | None], output_file: str) -> None:
    fieldnames = ["ID", "Editors", "FileCreated", "FileLastModified"]

    with open(output_file, "w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerow(row)


# =========================
# Main
# =========================

def main() -> None:
    #input_path = Path(INPUT_HTML_FILE)
    #raw_html = input_path.read_text(encoding="utf-8")
    raw_html = fetch_html(INPUT_URL)

    soup = BeautifulSoup(raw_html, "html.parser")

    words = parse_words(soup, INPUT_URL)
    metadata = parse_metadata(raw_html, INPUT_URL)

    tokens_filename = f"data/egyptian/{filename_by_id(INPUT_URL)}.csv"
    write_words_csv(words, tokens_filename)
    write_metadata_csv(metadata, "data/egyptian/" + METADATA_CSV_FILE)

    print(f"Wrote {len(words)} rows to {tokens_filename}")
    print(f"Wrote metadata to {METADATA_CSV_FILE}")


if __name__ == "__main__":
    main()

## Hittite Example 1: TLHdig Administrative Letters (SimTex XML format)

The *Thesaurus Linguarum Hethaeorum digitalis* (TLHdig) is a digital repository
containing transliterations of more than 98% of all published Hittite cuneiform
fragments. Published on Zenodo:

> TLHdig Project Team (2025). *Thesaurus Linguarum Hethaeorum digitalis (TLHdig)
> Beta Version 0.2*. Zenodo.
> [https://doi.org/10.5281/zenodo.15459133](https://doi.org/10.5281/zenodo.15459133)

Each tablet is a separate **XML** file, organised in folders by CTH number
(the standard Hittite text catalogue). The format is called *SimTex*. Its key
structural elements are:

| Element / Attribute | Content |
|---------------------|---------|
| `<AOxml>` | Root element (multiple namespaces) |
| `<AOHeader><docID>` | Tablet identifier (e.g. `HKM 9`) |
| `<lb lnr="..." lg="..." cu="..."/>` | Line: number, language code, cuneiform Unicode |
| `<w trans="..." mrp0sel="..." mrp1="...">` | Word: normalised transliteration; morphological parsing |
| `<sGr>` | Sumerogram |
| `<d>` | Determinative |
| `<aGr>` | Akkadogram |
| `<del_fin/>` / `<del_in/>` | Start / end of damaged section |
| `<parsep/>` | Paragraph separator |

The `mrp1` attribute encodes morphological information in a single `@`-delimited
string: `lemma@German_meaning@morphological_form@HZL_reference@`.
For example: `"uwa=te-@(her)bringen@3PL.PST@I.6.1@"` →
lemma `uwa=te-`, meaning *bring (here)*, form `3PL.PST`.

We work with **CTH 186** — the administrative letters from Maşat Höyük (ancient
Tapikka), 14th century BCE. These are short, well-preserved royal correspondence
tablets, ideal for learning to navigate a complex XML structure.

**Task:** Parse the XML of tablet **HKM 9**, extract lines and words, and produce
a DataFrame that includes line number, language, cuneiform Unicode, transliteration,
and the first morphological parsing option.


In [ ]:
# HKM 9 — CTH 186 (Hittite administrative letter, Maşat Höyük, 13th c. BCE)
# Source: TLHdig Beta 0.2, Zenodo https://doi.org/10.5281/zenodo.15459133
#
# To download the full archive and extract this file in Colab:
#
#   import requests, zipfile, io
#   url = 'https://zenodo.org/api/records/15459134/files/TLHdig_0.2.0-beta.zip/content'
#   r = requests.get(url, stream=True)
#   with open('TLHdig.zip', 'wb') as f:
#       for chunk in r.iter_content(8192): f.write(chunk)
#   z = zipfile.ZipFile('TLHdig.zip')
#   hkm9_xml = z.read('TLHbasisONLINE25.1_ZENODO/CTH 186_XML/HKM 9.xml').decode('utf-8')
#
# XML embedded below (extracted directly from the archive — cuneiform is correct).

hkm9_xml = """<?xml-stylesheet href="HPMxml.css" type="text/css"?><AOxml xmlns:hpm="http://hethiter.net/ns/hpm/1.0" xmlns:AO="http://hethiter.net/ns/AO/1.0" xmlns:dc="http://purl.org/dc/elements/1.1/" xmlns:meta="urn:oasis:names:tc:opendocument:xmlns:meta:1.0" xmlns:text="urn:oasis:names:tc:opendocument:xmlns:text:1.0" xmlns:table="urn:oasis:names:tc:opendocument:xmlns:table:1.0" xmlns:draw="urn:oasis:names:tc:opendocument:xmlns:drawing:1.0" xmlns:xlink="http://www.w3.org/1999/xlink" xml:space="preserve" ><AOHeader><docID>HKM 9</docID><meta><creation-date date="2016-04-15T16:55:36.58"/><kor2 date="2023-07-26T16:00:33"/><AOxml-creation date="2023-07-26T16:00:33"/><annotation> <annot editor="auto" date=""/> <annot editor="" date=""/></annotation></meta></AOHeader><body> <div1 type="transliteration"> <text xml:lang="Hit">
<AO:Manuscripts><AO:TxtPubl>HKM 9</AO:TxtPubl> </AO:Manuscripts>
 
<lb txtid="HKM 9" lnr="Vs. 1" lg="Hit" cu="𒌝𒈠𒀭𒌓𒅆𒈠"/> <w trans="UM-MA" mrp0sel=" 1 " mrp1="UMMA@folgendermaßen@ADV@@ " ><aGr>UM-MA</aGr></w> <w trans="UTU-ŠI-MA" mrp0sel=" " mrp1="UTU-ŠI-MA@‘Meine Sonne’@{a → NOM.SG(UNM)_QUOT} {b → ACC.SG(UNM)_QUOT} {c → NOM.PL(UNM)_QUOT} {d → ACC.PL(UNM)_QUOT} {e → GEN.SG(UNM)_QUOT} {f → GEN.PL(UNM)_QUOT} {g → D/L.SG(UNM)_QUOT} {h → D/L.PL(UNM)_QUOT} {i → ALL(UNM)_QUOT} {j → ABL(UNM)_QUOT} {k → INS(UNM)_QUOT} {l → VOC.SG(UNM)_QUOT} {m → VOC.PL(UNM)_QUOT}@@ D" ><d>D</d><sGr>UTU</sGr><aGr>-ŠI-MA</aGr></w> 
<lb txtid="HKM 9" lnr="Vs. 2" lg="Hit" cu="𒀀𒈾𒁹𒅗𒀸𒋗𒌑𒆠𒉈𒈠"/> <w trans="A-NA kaššu" mrp0sel=" " mrp1="kašš=u-@Kaš(š)u@{ a → …:D/L.SG} { b → …:D/L.PL} { c → …:ALL}@38.4@m"><aGr>A-NA</aGr> <d>m</d>ka-aš-šu-ú</w> <w trans="QÍ-BÍ-MA" mrp0sel=" 1 " mrp1="QABÛ@sagen@2SG.IMP_CNJ@@ " ><aGr>QÍ-BÍ-MA</aGr></w> <parsep/> 
<lb txtid="HKM 9" lnr="Vs. 3" lg="Hit" cu="𒌋𒁹𒁹𒁹𒇽𒈨𒌍𒁁𒋼𒀭𒁺𒍑𒃷"/> <w trans="13" mrp0sel=" 1 " mrp1="13@13@@ QUANcar@" ><num>13</num></w> <w trans="pitteantuškan" mrp0sel="???" ><d>LÚ.MEŠ</d>pít-te-an-du-uš-kán</w> 
<lb txtid="HKM 9" lnr="Vs. 4" lg="Hit" cu="𒂉𒀉𒉺𒊏𒀀𒈾𒀉𒋫"/> <w trans="kuit" mrp0sel=" " mrp1="kui-@welcher@{ a → REL.NOM.SG.N} { b → REL.ACC.SG.N}@22@" mrp2="kui-@wer?@{ a → INT.NOM.SG.N} { b → INT.ACC.SG.N}@23@" mrp3="kuit@weil@@ CNJ@" >ku-it</w> <w trans="para" mrp0sel=" " mrp1="parā@außerdem@@ ADV@" mrp2="parā@heraus aus@@ POSP@" mrp3="parā@aus-@@ PREV@" mrp4="par=a-@Luft@{ a → NOM.COLL.C} { b → ACC.COLL.C}@1.1@" mrp5="par=a-@Luft@{ a → VOC.SG} { b → ALL} { c → STF}@1.1@" mrp6="par=a-@Para(?)@{ a → DN.STF} { b → DN.HURR.ABS} { c → DN.NOM.SG(UNM)} { d → DN.ACC.SG(UNM)} { e → DN.GEN.SG(UNM)} { f → DN.D/L.SG(UNM)} { g → DN.ABL(UNM)} { h → DN.INS(UNM)} { i → DN.VOC.SG}@35.1.1@D" mrp7="par=a-@Para(?)@{ a → DN.NOM.SG(UNM)} { b → DN.ACC.SG(UNM)} { c → DN.GEN.SG(UNM)} { d → DN.D/L.SG(UNM)} { e → DN.ABL(UNM)} { f → DN.INS(UNM)} { g → DN.VOC.SG(UNM)}@35.1.1 += ma@CNJctr@@ D" >pa-ra-a</w> <w trans="naitta" mrp0sel=" " mrp1="n=ai-@(sich) drehen@{ a → 2SG.PST} { b → 3SG.PST}@II.6.3@" mrp2="n=ai-@(sich) drehen@3SG.PST@II.6.3@ += ya@CNJadd@" >na-it-ta</w> 
<lb txtid="HKM 9" lnr="Vs. 5" lg="Hit" cu="𒈾𒀸𒌑𒉿𒋼𒅕"/> <w trans="naš" mrp0sel=" " mrp1="n=aš@@ { a → CONNn=PPRO.3SG.C.NOM} { b → CONNn=PPRO.3PL.C.ACC}@@ " >na-aš</w> <w trans="uater" mrp0sel=" 1 " mrp1="uwa=te-@(her)bringen@3PL.PST@I.6.1@" >ú-wa-te-er</w> <parsep/> 
<lb txtid="HKM 9" lnr="Vs. 6" lg="Hit" cu="𒊭𒀲𒆳𒊏𒄭𒀀▒𒈠𒈬"/> <w trans="ŠA ANŠE.KUR.RAmamu" mrp0sel=" " mrp1="ANŠE.KUR.RA@Pferd@{ a → …:GEN.SG} { b → …:GEN.PL}@28.3.1.1 += ma=mu@{ R → CNJctr=PPRO.1SG.ACC} { S → CNJctr=PPRO.1SG.DAT}@@ "><aGr>ŠA</aGr> <sGr>ANŠE.KUR.RA</sGr><d>ḪI.A</d>-ma-mu</w> 
<lb txtid="HKM 9" lnr="Vs. 7" lg="Hit" cu="𒂉𒀉𒌓𒋻𒄩𒀜𒊏𒀀𒌍"/> <w trans="kuit" mrp0sel=" " mrp1="kui-@welcher@{ a → REL.NOM.SG.N} { b → REL.ACC.SG.N}@22@" mrp2="kui-@wer?@{ a → INT.NOM.SG.N} { b → INT.ACC.SG.N}@23@" mrp3="kuit@weil@@ CNJ@" >ku-it</w> <w trans="uttar" mrp0sel=" " mrp1="udd=ar@Wort; Sache@{ a → NOM.SG.N} { b → ACC.SG.N} { c → NOM.PL.N} { d → ACC.PL.N} { e → STF}@14.1.1@" mrp2="utt=ar@Wort; Sache@{ a → NOM.SG.N} { b → ACC.SG.N} { c → NOM.PL.N} { d → ACC.PL.N} { e → STF}@14.1.1@" >ut-tar</w> <w trans="ḫatraiš" mrp0sel=" 1 " mrp1="ḫatr=ā(e)-@mitteilen@2SG.PST@I.9@" >ḫa-at-ra-a-eš</w> 
<lb txtid="HKM 9" lnr="u. Rd. 8" lg="Hit" cu="𒈾𒀜𒀸𒈨"/> <w trans="nat" mrp0sel=" " mrp1="n=at@@ { a → CONNn=PPRO.3SG.N.NOM} { b → CONNn=PPRO.3SG.N.ACC} { c → CONNn=PPRO.3PL.N.NOM} { d → CONNn=PPRO.3PL.N.ACC} { e → CONNn=PPRO.3PL.C.NOM}@@ " >na-at</w> <w trans="AŠ-ME" mrp0sel=" 1 " mrp1="ŠEMÛ@hören@1SG.PST@@ " ><aGr>AŠ-ME</aGr></w> <parsep/> 
<gap t="line" c="Textende"/> 
</text></div1></body></AOxml> """

print(hkm9_xml[:400])


In [ ]:
# Parse the XML with Python's built-in ElementTree module.
# The SimTex format uses multiple XML namespaces; we register them
# so we can use short prefixes in our XPath queries.

import xml.etree.ElementTree as ET

NS = {
    'AO':   'http://hethiter.net/ns/AO/1.0',
    'hpm':  'http://hethiter.net/ns/hpm/1.0',
    'dc':   'http://purl.org/dc/elements/1.1/',
}

root = ET.fromstring(hkm9_xml)

# Extract document ID from the header
doc_id = root.find('AOHeader/docID').text
print(f'Document: {doc_id}')

# The text content lives in body > div1 > text
text_node = root.find('body/div1/text')
lang = text_node.get('{http://www.w3.org/XML/1998/namespace}lang', 'unknown')
print(f'Primary language: {lang}')

# Count lines and words
lines = text_node.findall('lb')
words = text_node.findall('w')
print(f'Lines: {len(lines)}')
print(f'Words: {len(words)}')


In [ ]:
# Extract lines and words from the XML into the course standard CSV format.
# Mapping SimTex XML attributes to course CSV columns:
#
#  ref          doc_id.line_label.word_index  (e.g. HKM 9.Vs. 1.1)
#  inst         (not available — SimTex has no ORACC-style inst)
#  frag         trans attribute (normalised transliteration)
#  norm         trans attribute (same — no separate norm in SimTex)
#  cf           lemma from mrp1 (1st @ field)
#  sense        German meaning from mrp1 (2nd @ field)
#  pos          morphological form from mrp1 (3rd @ field)
#  unicode      (not per-word — cu= is per-line in SimTex)
#  unicode_word (not available at word level)
#  reading      inner text of <w> (syllabic sign-level reading)
#  break        'damaged' if <del_fin>/<del_in> inside <w>, else 'complete'
#  break_perc   0.5 if damaged (cannot compute precisely from SimTex)
#  lang         lg= attribute of enclosing <lb/>
#  text         docID
#  line         lnr= attribute of enclosing <lb/>
#  word         word index within line

import re

def get_inner_text(elem):
    parts = [elem.text or '']
    for child in elem:
        parts.append(child.text or '')
        parts.append(child.tail or '')
    return ''.join(parts).strip()

def parse_mrp(mrp_str):
    if not mrp_str:
        return '', '', ''
    parts = mrp_str.split('@')
    return (parts[0].strip() if len(parts) > 0 else '',
            parts[1].strip() if len(parts) > 1 else '',
            parts[2].strip() if len(parts) > 2 else '')

def has_damage(w_elem):
    return any(child.tag in ('del_fin', 'del_in', 'laes_in', 'laes_fin')
               for child in w_elem)

rows = []
current_lnr = ''
current_lg  = ''
word_idx    = 0

for child in text_node:
    if child.tag == 'lb':
        current_lnr = child.get('lnr', '')
        current_lg  = child.get('lg', '')
        word_idx    = 0
    elif child.tag == 'w':
        word_idx += 1
        trans    = child.get('trans', '').strip()
        mrp1     = child.get('mrp1', '')
        cf, sense, morph = parse_mrp(mrp1)
        reading  = get_inner_text(child)
        damaged  = has_damage(child)

        rows.append({
            'ref':          f"{doc_id}.{current_lnr}.{word_idx}",
            'inst':         '',
            'frag':         trans if trans else reading,
            'norm':         trans if trans else reading,
            'cf':           cf,
            'sense':        sense,
            'pos':          morph,
            'unicode':      '',
            'unicode_word': '',
            'reading':      reading,
            'break':        "['damaged']" if damaged else "['complete']",
            'break_perc':   '0.5' if damaged else '0.0',
            'mask':         '',
            'lang':         current_lg,
            'text':         doc_id,
            'line':         current_lnr,
            'word':         str(word_idx),
        })

print(f"Extracted {len(rows)} word tokens")
for r in rows[:4]:
    print(f"  ref={r['ref']:<28}  frag={r['frag']:<22}  cf={r['cf']:<18}  pos={r['pos']}")


In [ ]:
import pandas as pd

df_hit = pd.DataFrame(rows)
print(df_hit.to_string(index=False))

import os
os.makedirs('data/hittite', exist_ok=True)
df_hit.to_csv('data/hittite/TLHdig_HKM9_CTH186.csv', index=False)
print('\nSaved to data/hittite/TLHdig_HKM9_CTH186.csv')

# Discussion questions:
# - Which fields are we unable to fill for Hittite vs. the Akkadian ORACC data?
#   What does that reveal about what each corpus prioritises?
# - The 'sense' column contains German. What challenges does that create?
# - The cu= attribute in SimTex gives cuneiform per line, not per word.
#   How would you assign individual signs to words? What extra steps are needed?

## Egyptian Example 3 (optional)

> **Note:** This example demonstrates how to handle non-Unicode hieroglyphs using
> Gardiner sign numbers embedded with `<g>` tags in a custom CSV export from the
> Thesaurus Linguae Aegyptiae (TLA). **This section is currently being revised by
> the course instructors** to reflect the latest TLA data structure.
> The original cells are preserved below for reference.


In [ ]:
eg2_csv = """,text,line,word,ref,frag,norm,unicode_word,unicode,lemma_id,cf,pos,sense
92,3Z5EM77HJFCOPKZDDZFEMI6KVY,5,7,3Z5EM77HJFCOPKZDDZFEMI6KVY.5.7,gꜣu̯.w,gꜣu̯.w,<g>V96</g>𓅱,"['<', 'g', '>', 'V', '9', '6', '<', '/', 'g', '>', '𓅱']",166210,gꜣu̯,VERB,eng sein; entbehren; (jmdn.) Not leiden lassen
151,4WVXFJZFLNAYHP3Y5O5SLWD7DA,2,2,4WVXFJZFLNAYHP3Y5O5SLWD7DA.2.2,nꜥw,nꜥw,𓈖𓂝𓅱<g>I14C</g>𓏤,"['𓈖', '𓂝', '𓅱', '<', 'g', '>', 'I', '1', '4', 'C', '<', '/', 'g', '>', '𓏤']",80510,Nꜥw,PROPN,Sich windender (Personifikation der Schlange)
153,4WVXFJZFLNAYHP3Y5O5SLWD7DA,2,5,4WVXFJZFLNAYHP3Y5O5SLWD7DA.2.5,nꜥw,nꜥw,𓈖𓂝𓅱<g>I14C</g>𓏤,"['𓈖', '𓂝', '𓅱', '<', 'g', '>', 'I', '1', '4', 'C', '<', '/', 'g', '>', '𓏤']",80510,Nꜥw,PROPN,Sich windender (Personifikation der Schlange)
200,67HZI45S3REA3LWVZOKJ6QJOIE,14,9,67HZI45S3REA3LWVZOKJ6QJOIE.14.9,nbi̯.n,nbi̯.n,𓈖𓎟𓃀<g>D107</g>𓈖,"['𓈖', '𓎟', '𓃀', '<', 'g', '>', 'D', '1', '0', '7', '<', '/', 'g', '>', '𓈖']",82520,nbi̯,VERB,schmelzen; gießen
204,67HZI45S3REA3LWVZOKJ6QJOIE,14,13,67HZI45S3REA3LWVZOKJ6QJOIE.14.13,nḏr.n,nḏr.n,𓈖𓇦𓂋<g>U19A</g>𓆱𓈖,"['𓈖', '𓇦', '𓂋', '<', 'g', '>', 'U', '1', '9', 'A', '<', '/', 'g', '>', '𓆱', '𓈖']",91630,nḏr,VERB,(Holz) bearbeiten; zimmern
206,67HZI45S3REA3LWVZOKJ6QJOIE,14,15,67HZI45S3REA3LWVZOKJ6QJOIE.14.15,b(w)n.wDU,bwn.wDU,𓃀𓈖𓏌𓅱<g>T86</g><g>T86</g>,"['𓃀', '𓈖', '𓏌', '𓅱', '<', 'g', '>', 'T', '8', '6', '<', '/', 'g', '>', '<', 'g', '>', 'T', '8', '6', '<', '/', 'g', '>']",55330,bwn,NOUN,Speerspitzen (des Fischspeeres)
"""

In [ ]:
import pandas as pd
from io import StringIO

# Convert the string into a StringIO object
# This is only necessary because we presented the csv as a string not as a file that is loaded into the notebook
csv_data = StringIO(eg2_csv)

# Read the data into a pandas DataFrame
df = pd.read_csv(csv_data)

# Display the DataFrame
df

In [ ]:
def split_tags(text):
    parts = []  # List to collect output of the function
    while '<g>' in text and '</g>' in text:
        pre, rest = text.split('<g>', 1)  # splits at the first <g> found
        tag_content, post = rest.split('</g>', 1)  # splits the rest at the first </g> found

        # adds elements before the first <g></g> tag to the List
        parts.extend(pre) # splits strings into individual characters and adds them to the List

        #  adds element inside the first <g></g> tag to the List
        parts.append(tag_content)

        # text variable is set to remaining text
        text = post

    # After last tag found, the remainder of the text is split and added to the List
    parts.extend(text)
    return parts

def process_text(text):
    if pd.isna(text): # deals with NaN
        return []
    else:
        return split_tags(text)

# apply functions to every row of the column 'unicode_word'
df['unicode_splitted'] = df['unicode_word'].apply(process_text)
# delete obsolete column
df.drop('unicode', axis=1, inplace=True)

df
#df.to_csv("EG-TLA-example.csv")